In [1]:

from pathlib import Path
import os
import sys
import json

# Locate the project root from the current notebook directory
cwd = Path.cwd()
print("Current directory:", cwd)

candidates = [
    cwd,
    *cwd.parents
]

project_root = None

for path in candidates:
    if (
        (path / "data").exists()
        and (path / "notebooks").exists()
    ):
        project_root = path
        break

if project_root is None:
    # Also support projects where notebooks are stored
    # in a directory named "Notebooks 2".
    for path in candidates:
        if (path / "data").exists():
            project_root = path
            break

print("Project root:", project_root)

if project_root is None:
    print("Could not automatically locate the project root.")
    print("Please share the Current directory printed above.")
else:
    os.chdir(project_root)
    print("Working directory set to:", Path.cwd())
    print("Python version:", sys.version.split()[0])


Current directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/Notebooks 2
Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification
Working directory set to: /Users/vs/Sem 5/Respiratory-Disease-Classification
Python version: 3.14.7


In [2]:

from pathlib import Path

ROOT = Path.cwd()

print("PROJECT ROOT:", ROOT)

# Check important folders and existing model artifacts
checks = [
    "data/processed/deep_learning",
    "data/processed/deep_learning/cnn_baseline",
    "data/processed/deep_learning/disease_class_mapping.csv",
    "models",
    "models/multidataset_pretraining/phenotype_encoder_hf_best.pt",
]

print("\nPATH CHECK")
for item in checks:
    path = ROOT / item
    status = "FOUND" if path.exists() else "NOT FOUND"
    print(f"{status:12} {item}")

# Find relevant notebooks
print("\nRELEVANT NOTEBOOKS")
for folder in [ROOT / "Notebooks", ROOT / "Notebooks 2", ROOT / "notebooks"]:
    if folder.exists():
        for nb in sorted(folder.glob("*.ipynb")):
            if any(key in nb.name.lower() for key in [
                "disease", "multi", "cnn", "transfer"
            ]):
                print(nb.relative_to(ROOT))


PROJECT ROOT: /Users/vs/Sem 5/Respiratory-Disease-Classification

PATH CHECK
FOUND        data/processed/deep_learning
FOUND        data/processed/deep_learning/cnn_baseline
FOUND        data/processed/deep_learning/disease_class_mapping.csv
FOUND        models
FOUND        models/multidataset_pretraining/phenotype_encoder_hf_best.pt

RELEVANT NOTEBOOKS
Notebooks/11_cnn_baseline.ipynb
Notebooks/12_multifeature_cnn.ipynb
Notebooks/13_cnn_lstm_optional.ipynb
Notebooks 2/03_Multidataset_Phenotype_Pretraining_ICBHI_Disease.ipynb
Notebooks 2/04_ICBHI_Disease_Transfer_Learning.ipynb
notebooks/11_cnn_baseline.ipynb
notebooks/12_multifeature_cnn.ipynb
notebooks/13_cnn_lstm_optional.ipynb


In [3]:

from pathlib import Path
import pandas as pd

ROOT = Path.cwd()

# Find the actual manifest files without assuming their names
search_dir = ROOT / "data/processed/deep_learning"

manifest_files = []
for pattern in ["*train*manifest*.csv", "*validation*manifest*.csv",
                "*val*manifest*.csv", "*test*manifest*.csv"]:
    manifest_files.extend(search_dir.rglob(pattern))

manifest_files = sorted(set(manifest_files))

print("MANIFEST FILES FOUND:")
for path in manifest_files:
    print("-", path.relative_to(ROOT))

print("\nDISEASE CLASS MAPPING:")
mapping_path = ROOT / "data/processed/deep_learning/disease_class_mapping.csv"
if mapping_path.exists():
    mapping = pd.read_csv(mapping_path)
    print("Shape:", mapping.shape)
    display(mapping.head(20))
else:
    print("Class mapping file not found.")


MANIFEST FILES FOUND:
- data/processed/deep_learning/cnn_baseline/test_cnn_manifest.csv
- data/processed/deep_learning/cnn_baseline/train_cnn_manifest.csv
- data/processed/deep_learning/cnn_baseline/validation_cnn_manifest.csv
- data/processed/deep_learning/multidataset_pretraining/hf_fraiwan_phenotype_train_manifest.csv
- data/processed/deep_learning/multidataset_pretraining/hf_phenotype_train_manifest.csv
- data/processed/deep_learning/multidataset_pretraining/phenotype_pretraining_manifest.csv
- data/processed/deep_learning/test_manifest.csv
- data/processed/deep_learning/train_manifest.csv

DISEASE CLASS MAPPING:
Shape: (8, 2)


,disease_label,diagnosis
0,0,Asthma
1,1,Bronchiectasis
2,2,Bronchiolitis
3,3,COPD
4,4,Healthy
5,5,LRTI
6,6,Pneumonia
7,7,URTI


In [4]:

from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "data/processed/deep_learning/cnn_baseline"

manifest_paths = {
    "train": DATA_DIR / "train_cnn_manifest.csv",
    "validation": DATA_DIR / "validation_cnn_manifest.csv",
    "test": DATA_DIR / "test_cnn_manifest.csv",
}

manifests = {}

for split, path in manifest_paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {split} manifest: {path}")

    df = pd.read_csv(path)
    manifests[split] = df

    print(f"\n{'=' * 20} {split.upper()} {'=' * 20}")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    label_col = next(
        (c for c in ["disease_label", "label", "diagnosis"]
         if c in df.columns),
        None
    )

    if label_col is None:
        print("No recognized disease-label column found.")
    else:
        print("\nClass counts:")
        print(df[label_col].value_counts(dropna=False).sort_index())

    patient_col = next(
        (c for c in ["patient_id", "subject_id", "patient", "subject"]
         if c in df.columns),
        None
    )

    if patient_col:
        print("Unique patients:", df[patient_col].nunique())
    else:
        print("Patient ID column not identified automatically.")

# Check patient leakage across splits
print("\nPATIENT OVERLAP CHECK")

patient_col = next(
    (c for c in ["patient_id", "subject_id", "patient", "subject"]
     if all(c in df.columns for df in manifests.values())),
    None
)

if patient_col:
    patient_sets = {
        split: set(df[patient_col].dropna().astype(str))
        for split, df in manifests.items()
    }

    for a, b in [("train", "validation"),
                 ("train", "test"),
                 ("validation", "test")]:
        overlap = patient_sets[a] & patient_sets[b]
        print(f"{a} vs {b}: {len(overlap)} shared patients")
        if overlap:
            print("Examples:", sorted(overlap)[:10])
else:
    print("Could not identify a common patient ID column.")
    print("We will inspect the manifest columns before proceeding.")



==================== TRAIN ====================
Rows: 4423
Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split']

Class counts:
disease_label
0       6
1      76
2     114
3    3713
4     194
5      32
6     121
7     167
Name: count, dtype: int64
Unique patients: 82

==================== VALIDATION ====================
Rows: 914
Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split']

Class counts:
disease_label
1      8
2     27
3    696
4     44
6    112
7     27
Name: count, dtype: int64
Unique patients: 19

==================== TEST ====================
Rows: 1561
Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split']

Class counts:
disease_label
1      20
2      19
3    1337
4      84
6      52
7      49
Name: count, dtype: int64
Unique patients: 25

In [5]:

import json
from pathlib import Path

ROOT = Path.cwd()
source_nb = ROOT / "Notebooks/12_multifeature_cnn.ipynb"

if not source_nb.exists():
    raise FileNotFoundError(f"Notebook not found: {source_nb}")

with open(source_nb, "r", encoding="utf-8") as f:
    nb = json.load(f)

keywords = [
    "mfcc", "mel", "chroma", "spectrogram",
    "class ", "def forward", "conv2d", "conv1d",
    "dataset", "dataloader", "normalization",
    "optimizer", "criterion", "checkpoint",
    "train_manifest", "validation_manifest"
]

print("SOURCE NOTEBOOK:", source_nb.name)
print("Total cells:", len(nb["cells"]))

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    matches = [word for word in keywords if word in source.lower()]

    if matches:
        print(f"\n{'=' * 25} CELL {i} {'=' * 25}")
        print("Matched topics:", ", ".join(matches))
        print(source[:7000])
        if len(source) > 7000:
            print("\n... [cell truncated] ...")


SOURCE NOTEBOOK: 12_multifeature_cnn.ipynb
Total cells: 111

========================= CELL 1 =========================
Matched topics: dataset, dataloader
from pathlib import Path
import random
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import soundfile as sf
import librosa

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("PyTorch version:", torch.__version__)
print("Librosa version:", librosa.__version__)

========================= CELL 5 =========================
Matched topics: mel
TARGET_SR = 4000
TARGET_DURATION = 5.0
TARGET_SAMPLES = int(
    TARGET_SR * TARGET_DURATION
)

N_FFT = 512
HOP_LENGTH = 128
N_MELS = 64

FMIN = 0
FMAX =

In [1]:

from pathlib import Path
import pandas as pd
import torch

ROOT = Path.cwd()
DL_DIR = ROOT / "data/processed/deep_learning"
MULTI_DIR = DL_DIR / "multifeature_cnn"
BASELINE_DIR = DL_DIR / "cnn_baseline"
TRANSFER_DIR = ROOT / "models/multidataset_pretraining"

metric_files = [
    BASELINE_DIR / "cnn_baseline_metrics.csv",
    MULTI_DIR / "multifeature_cnn_metrics.csv",
    DL_DIR / "multifeature_cnn_12b" / "metrics.csv",
    DL_DIR / "multifeature_cnn_12B" / "metrics.csv",
    DL_DIR / "multifeature_cnn_12b" / "multifeature_cnn_metrics.csv",
]

print("SAVED METRICS")
found = False

for path in metric_files:
    if path.exists():
        found = True
        print(f"\n--- {path.relative_to(ROOT)} ---")
        display(pd.read_csv(path))

if not found:
    print("No metrics found at the expected locations.")
    print("Searching for existing metric CSV files...")
    for path in DL_DIR.rglob("*metrics*.csv"):
        print("-", path.relative_to(ROOT))

print("\nCHECKPOINT SIZES")
checkpoint_paths = [
    MULTI_DIR / "best_multifeature_cnn.pt",
    TRANSFER_DIR / "phenotype_encoder_hf_best.pt",
]

for path in checkpoint_paths:
    if path.exists():
        print(
            f"{path.relative_to(ROOT)}: "
            f"{path.stat().st_size / 1024:.1f} KB"
        )
    else:
        print("Not found:", path.relative_to(ROOT))

print("\nDEVICE")
print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())


SAVED METRICS
No metrics found at the expected locations.
Searching for existing metric CSV files...

CHECKPOINT SIZES
Not found: data/processed/deep_learning/multifeature_cnn/best_multifeature_cnn.pt
Not found: models/multidataset_pretraining/phenotype_encoder_hf_best.pt

DEVICE
PyTorch: 2.14.1
MPS available: True


In [6]:

from pathlib import Path

ROOT = Path.cwd()

search_roots = [
    ROOT / "models",
    ROOT / "data/processed/deep_learning",
]

patterns = [
    "*.pt",
    "*.pth",
    "*.pkl",
    "*.joblib",
    "*metrics*.csv",
    "*comparison*.csv",
    "*history*.csv",
    "*summary*.json",
]

for root in search_roots:
    print(f"\n{'=' * 20} {root.relative_to(ROOT)} {'=' * 20}")

    if not root.exists():
        print("Directory not found.")
        continue

    found = set()
    for pattern in patterns:
        found.update(root.rglob(pattern))

    if not found:
        print("No matching artifacts found.")
    else:
        for path in sorted(found):
            size_kb = path.stat().st_size / 1024
            print(f"{path.relative_to(ROOT)} ({size_kb:.1f} KB)")



==================== models ====================
models/deep_learning/cnn_lstm/best_cnn_lstm.pt (364.7 KB)
models/deep_learning/cnn_lstm/cnn_lstm_training_history.csv (1.3 KB)
models/deep_learning/experiment_b/experiment_b_cnn.pt (168.1 KB)
models/deep_learning/lightweight_mel_cnn/best_lightweight_mel_cnn.pt (303.6 KB)
models/deep_learning/lightweight_mel_cnn/cnn_training_history.csv (3.3 KB)
models/deep_learning/multifeature_cnn/best_multifeature_cnn.pt (453.7 KB)
models/deep_learning/multifeature_cnn/multifeature_cnn_training_history.csv (0.4 KB)
models/deep_learning/regularized_multifeature_cnn/best_regularized_multifeature_cnn.pt (454.6 KB)
models/deep_learning/regularized_multifeature_cnn/training_history.csv (0.8 KB)
models/multidataset_pretraining/phenotype_encoder_hf_best.pt (115.8 KB)
models/phenotype_cnn/cnn_A_balanced_seed21.pt (168.1 KB)
models/phenotype_cnn/cnn_A_balanced_seed42.pt (168.1 KB)
models/phenotype_cnn/cnn_A_balanced_seed7.pt (168.1 KB)
models/phenotype_cnn/cnn

In [7]:

from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
DL_DIR = ROOT / "data/processed/deep_learning"

result_files = [
    DL_DIR / "cnn_baseline/cnn_baseline_metrics.csv",
    DL_DIR / "multifeature_cnn/multifeature_cnn_metrics.csv",
    DL_DIR / "multifeature_cnn_12b/metrics.csv",
    DL_DIR / "cnn_lstm/cnn_lstm_metrics.csv",
    DL_DIR / "final_deep_learning_comparison.csv",
    DL_DIR / "multifeature_cnn/final_cnn_experiment_comparison.csv",
    DL_DIR / "multifeature_cnn/disease_level_cnn_comparison.csv",
    DL_DIR / "patient_level_prediction/final_patient_level_comparison.csv",
]

print("=" * 65)
print("EXISTING DISEASE MODEL RESULTS")
print("=" * 65)

for path in result_files:
    if path.exists():
        print(f"\n--- {path.relative_to(ROOT)} ---")
        display(pd.read_csv(path))
    else:
        print(f"\nNOT FOUND: {path.relative_to(ROOT)}")

summary_files = [
    DL_DIR / "final_model_comparison_summary.json",
    DL_DIR / "multifeature_cnn/notebook12_experiment_summary.json",
    DL_DIR / "cnn_lstm/cnn_lstm_experiment_summary.json",
    DL_DIR / "patient_level_prediction/patient_level_experiment_summary.json",
]

print("\n" + "=" * 65)
print("EXPERIMENT SUMMARIES")
print("=" * 65)

for path in summary_files:
    if path.exists():
        print(f"\n--- {path.relative_to(ROOT)} ---")
        with open(path, "r", encoding="utf-8") as f:
            print(json.dumps(json.load(f), indent=2))


EXISTING DISEASE MODEL RESULTS

--- data/processed/deep_learning/cnn_baseline/cnn_baseline_metrics.csv ---


,model,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,best_epoch,best_validation_loss,test_samples,test_patients
0,Lightweight Mel CNN,0.716848,0.310168,0.213023,0.265858,0.21288,0.768721,32,1.207017,1561,25



--- data/processed/deep_learning/multifeature_cnn/multifeature_cnn_metrics.csv ---


,model,best_epoch,best_validation_loss,test_samples,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1
0,Multi-Feature CNN,1,2.452791,1561,0.855221,0.166417,0.14272,0.166417,0.15366,0.789662



--- data/processed/deep_learning/multifeature_cnn_12b/metrics.csv ---


...


--- data/processed/deep_learning/multifeature_cnn/disease_level_cnn_comparison.csv ---


,Disease,Baseline_F1,MultiFeature_F1,F1_Difference,Baseline_Support,MultiFeature_Support
0,Asthma,0.000000,0.000000,0.000000,0,0
1,Bronchiectasis,0.085714,0.000000,-0.085714,20,20
2,Bronchiolitis,0.126761,0.000000,-0.126761,19,19
3,COPD,0.876823,0.921961,0.045138,1337,1337
4,Healthy,0.089888,0.000000,-0.089888,84,84
5,LRTI,0.000000,0.000000,0.000000,0,0
6,Pneumonia,0.250000,0.000000,-0.250000,52,52
7,URTI,0.060976,0.000000,-0.060976,49,49



--- data/processed/deep_learning/patient_level_prediction/final_patient_level_comparison.csv ---


,rank,aggregation_method,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1
0,1,Majority Vote,0.56,0.333333,0.132812,0.25,0.162069,0.482207
1,2,Mean Probability,0.56,0.333333,0.132812,0.25,0.162069,0.482207



EXPERIMENT SUMMARIES

--- data/processed/deep_learning/final_model_comparison_summary.json ---
{
  "best_classical_by_macro_f1": "Random Forest",
  "best_classical_by_balanced_accuracy": "SVM",
  "best_deep_learning_by_macro_f1": "Lightweight Mel CNN",
  "best_deep_learning_by_balanced_accuracy": "Lightweight Mel CNN",
  "best_patient_level_aggregation": "Majority Vote",
  "primary_final_deep_learning_model": "Lightweight Mel CNN"
}

--- data/processed/deep_learning/multifeature_cnn/notebook12_experiment_summary.json ---
{
  "notebook": "12_multifeature_cnn",
  "experiments": [
    "12A - Multi-Feature CNN",
    "12B - Regularized Multi-Feature CNN"
  ],
  "baseline_comparison": "11 - Lightweight Mel CNN",
  "input_features": [
    "Log-Mel",
    "MFCC",
    "Chroma"
  ],
  "patient_disjoint": true,
  "selection_priority": [
    "Macro F1",
    "Balanced Accuracy"
  ],
  "best_cnn": "Lightweight Mel CNN",
  "12A_best_epoch": 1,
  "12B_best_epoch": 2,
  "results_file": "/Users/vs/Sem 5

In [8]:

from pathlib import Path
import torch

ROOT = Path.cwd()

checkpoints = {
    "Lightweight Mel CNN": (
        ROOT / "models/deep_learning/lightweight_mel_cnn/"
        / "best_lightweight_mel_cnn.pt"
    ),
    "Multi-Feature CNN": (
        ROOT / "models/deep_learning/multifeature_cnn/"
        / "best_multifeature_cnn.pt"
    ),
    "Regularized Multi-Feature CNN": (
        ROOT / "models/deep_learning/regularized_multifeature_cnn/"
        / "best_regularized_multifeature_cnn.pt"
    ),
    "HF Phenotype Encoder": (
        ROOT / "models/multidataset_pretraining/"
        / "phenotype_encoder_hf_best.pt"
    ),
}

print("SAVED CHECKPOINT SIZES")
print("-" * 65)

for name, path in checkpoints.items():
    if path.exists():
        size_kb = path.stat().st_size / 1024
        try:
            state = torch.load(
                path,
                map_location="cpu",
                weights_only=True
            )

            if isinstance(state, dict) and "state_dict" in state:
                state = state["state_dict"]

            if isinstance(state, dict):
                tensors = [
                    value for value in state.values()
                    if isinstance(value, torch.Tensor)
                ]
                params = sum(t.numel() for t in tensors)
                print(
                    f"{name}: {size_kb:.1f} KB | "
                    f"{params:,} tensor parameters"
                )
            else:
                print(f"{name}: {size_kb:.1f} KB | format needs inspection")

        except Exception as exc:
            print(f"{name}: {size_kb:.1f} KB | load inspection: {exc}")
    else:
        print(f"{name}: NOT FOUND")


SAVED CHECKPOINT SIZES
-----------------------------------------------------------------
Lightweight Mel CNN: 303.6 KB | 0 tensor parameters
Multi-Feature CNN: 453.7 KB | 109,457 tensor parameters
Regularized Multi-Feature CNN: 454.6 KB | 109,457 tensor parameters
HF Phenotype Encoder: 115.8 KB | 0 tensor parameters


In [9]:

from pathlib import Path
import torch

ROOT = Path.cwd()

checkpoints = {
    "Lightweight Mel CNN":
        ROOT / "models/deep_learning/lightweight_mel_cnn/best_lightweight_mel_cnn.pt",
    "Multi-Feature CNN":
        ROOT / "models/deep_learning/multifeature_cnn/best_multifeature_cnn.pt",
    "Regularized Multi-Feature CNN":
        ROOT / "models/deep_learning/regularized_multifeature_cnn/best_regularized_multifeature_cnn.pt",
    "HF Phenotype Encoder":
        ROOT / "models/multidataset_pretraining/phenotype_encoder_hf_best.pt",
}

for name, path in checkpoints.items():
    print(f"\n{'=' * 60}\n{name}")

    if not path.exists():
        print("Checkpoint not found:", path)
        continue

    state = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    print("Top-level type:", type(state).__name__)

    if isinstance(state, dict):
        print("Top-level keys:", list(state.keys())[:15])

        # Support both raw state dictionaries and wrapped checkpoints
        candidate = state
        for key in ["state_dict", "model_state_dict", "model"]:
            if key in state and isinstance(state[key], dict):
                candidate = state[key]
                print("Nested state dictionary:", key)
                break

        tensors = {
            key: value for key, value in candidate.items()
            if isinstance(value, torch.Tensor)
        }

        print("Tensor entries:", len(tensors))
        print("Total tensor elements:",
              f"{sum(t.numel() for t in tensors.values()):,}")

        for key, value in list(tensors.items())[:6]:
            print(f"  {key}: {tuple(value.shape)}")



Lightweight Mel CNN
Top-level type: dict
Top-level keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_loss', 'val_accuracy', 'class_mapping', 'config']
Nested state dictionary: model_state_dict
Tensor entries: 23
Total tensor elements: 24,267
  features.0.weight: (16, 1, 3, 3)
  features.0.bias: (16,)
  features.1.weight: (16,)
  features.1.bias: (16,)
  features.1.running_mean: (16,)
  features.1.running_var: (16,)

Multi-Feature CNN
Top-level type: OrderedDict
Top-level keys: ['logmel_branch.features.0.weight', 'logmel_branch.features.0.bias', 'logmel_branch.features.1.weight', 'logmel_branch.features.1.bias', 'logmel_branch.features.1.running_mean', 'logmel_branch.features.1.running_var', 'logmel_branch.features.1.num_batches_tracked', 'logmel_branch.features.4.weight', 'logmel_branch.features.4.bias', 'logmel_branch.features.5.weight', 'logmel_branch.features.5.bias', 'logmel_branch.features.5.running_mean', 'logmel_branch.features.5.running_var', 'logmel_branch.feat

In [10]:

from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "data/processed/deep_learning/cnn_baseline"

splits = {
    "train": pd.read_csv(DATA_DIR / "train_cnn_manifest.csv"),
    "validation": pd.read_csv(DATA_DIR / "validation_cnn_manifest.csv"),
    "test": pd.read_csv(DATA_DIR / "test_cnn_manifest.csv"),
}

class_mapping = pd.read_csv(
    ROOT / "data/processed/deep_learning/disease_class_mapping.csv"
)
class_names = dict(zip(
    class_mapping["disease_label"].astype(int),
    class_mapping["diagnosis"]
))

for split_name, df in splits.items():
    print(f"\n{'=' * 75}")
    print(f"{split_name.upper()} — SUPPORT BY DISEASE")
    print("=" * 75)

    report = (
        df.groupby("disease_label")
        .agg(
            samples=("disease_label", "size"),
            patients=("patient_id", "nunique"),
            recordings=("recording_id", "nunique"),
        )
        .reindex(range(8), fill_value=0)
        .reset_index()
    )

    report["disease"] = report["disease_label"].map(class_names)
    report = report[
        ["disease_label", "disease", "samples", "recordings", "patients"]
    ]

    display(report)

print("\nPATIENT-DISJOINT CHECK")
patient_sets = {
    name: set(df["patient_id"].dropna())
    for name, df in splits.items()
}

for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    overlap = patient_sets[a] & patient_sets[b]
    print(f"{a} vs {b}: {len(overlap)} overlapping patients")



TRAIN — SUPPORT BY DISEASE


,disease_label,disease,samples,recordings,patients
0,0,Asthma,6,1,1
1,1,Bronchiectasis,76,3,4
2,2,Bronchiolitis,114,1,4
3,3,COPD,3713,32,41
4,4,Healthy,194,1,17
5,5,LRTI,32,1,2
6,6,Pneumonia,121,4,4
7,7,URTI,167,1,9



VALIDATION — SUPPORT BY DISEASE


,disease_label,disease,samples,recordings,patients
0,0,Asthma,0,0,0
1,1,Bronchiectasis,8,1,1
2,2,Bronchiolitis,27,1,1
3,3,COPD,696,17,10
4,4,Healthy,44,2,4
5,5,LRTI,0,0,0
6,6,Pneumonia,112,3,1
7,7,URTI,27,1,2



TEST — SUPPORT BY DISEASE


,disease_label,disease,samples,recordings,patients
0,0,Asthma,0,0,0
1,1,Bronchiectasis,20,2,2
2,2,Bronchiolitis,19,1,1
3,3,COPD,1337,19,13
4,4,Healthy,84,2,5
5,5,LRTI,0,0,0
6,6,Pneumonia,52,3,1
7,7,URTI,49,2,3



PATIENT-DISJOINT CHECK
train vs validation: 0 overlapping patients
train vs test: 0 overlapping patients
validation vs test: 0 overlapping patients


In [11]:

import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
DATA_DIR = ROOT / "data/processed/deep_learning/cnn_baseline"

paths = {
    "train": DATA_DIR / "train_cnn_manifest.csv",
    "validation": DATA_DIR / "validation_cnn_manifest.csv",
    "test": DATA_DIR / "test_cnn_manifest.csv",
}

frames = []
for split, path in paths.items():
    df = pd.read_csv(path).copy()
    df["current_split"] = split
    frames.append(df)

all_data = pd.concat(frames, ignore_index=True)
all_data["disease_label"] = all_data["disease_label"].astype(int)

print("TOTAL UNIQUE PATIENTS:", all_data["patient_id"].nunique())
print("TOTAL UNIQUE RECORDINGS:", all_data["recording_id"].nunique())
print("TOTAL ROWS:", len(all_data))

# Count independent patients and recordings by disease and original split.
print("\nSUPPORT BY DISEASE AND OFFICIAL SPLIT")

summary = (
    all_data.groupby(["diagnosis", "official_split"], dropna=False)
    .agg(
        cycles=("disease_label", "size"),
        recordings=("recording_id", "nunique"),
        patients=("patient_id", "nunique"),
    )
    .reset_index()
    .sort_values(["diagnosis", "official_split"])
)

display(summary)

# Check whether one patient has more than one disease label.
patient_label_counts = (
    all_data.groupby("patient_id")["disease_label"].nunique()
)

multi_label_patients = patient_label_counts[patient_label_counts > 1]

print("\nPATIENTS ASSOCIATED WITH MULTIPLE DISEASE LABELS:",
      len(multi_label_patients))

if len(multi_label_patients):
    print("Examples:")
    display(
        all_data[
            all_data["patient_id"].isin(multi_label_patients.index)
        ][["patient_id", "recording_id", "diagnosis", "official_split"]]
        .drop_duplicates()
        .head(20)
    )


TOTAL UNIQUE PATIENTS: 126
TOTAL UNIQUE RECORDINGS: 34
TOTAL ROWS: 6898

SUPPORT BY DISEASE AND OFFICIAL SPLIT


,diagnosis,official_split,cycles,recordings,patients
0,Asthma,train,6,1,1
1,Bronchiectasis,test,14,2,1
2,Bronchiectasis,train,90,3,6
3,Bronchiolitis,test,86,1,3
4,Bronchiolitis,train,74,1,3
5,COPD,test,2409,27,27
6,COPD,train,3337,29,39
7,Healthy,test,175,2,14
8,Healthy,train,147,2,12
9,LRTI,train,32,1,2



PATIENTS ASSOCIATED WITH MULTIPLE DISEASE LABELS: 0


In [12]:

import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
DATA_DIR = ROOT / "data/processed/deep_learning/cnn_baseline"

frames = []
for filename in [
    "train_cnn_manifest.csv",
    "validation_cnn_manifest.csv",
    "test_cnn_manifest.csv",
]:
    df = pd.read_csv(DATA_DIR / filename)
    frames.append(df)

all_data = pd.concat(frames, ignore_index=True)

print("CURRENT CNN SPLIT vs OFFICIAL SPLIT")
display(pd.crosstab(
    all_data["official_split"],
    all_data["cnn_split"],
    margins=True
))

print("\nDISEASE COUNTS BY BOTH SPLIT DEFINITIONS")
display(pd.crosstab(
    [all_data["diagnosis"], all_data["official_split"]],
    all_data["cnn_split"],
    margins=False
))

print("\nRECORDING OVERLAP BETWEEN CURRENT CNN SPLITS")
recording_sets = {
    split: set(df["recording_id"].astype(str))
    for split, df in zip(
        ["train", "validation", "test"], frames
    )
}

for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    overlap = recording_sets[a] & recording_sets[b]
    print(f"{a} vs {b}: {len(overlap)} shared recordings")


CURRENT CNN SPLIT vs OFFICIAL SPLIT


cnn_split,test,train,validation,All
official_split,,,,
test,480,1943,333,2756
train,1081,2480,581,4142
All,1561,4423,914,6898



DISEASE COUNTS BY BOTH SPLIT DEFINITIONS


cnn_split                      test  train  validation
diagnosis      official_split                         
Asthma         train              0      6           0
Bronchiectasis test              14      0           0
               train              6     76           8
Bronchiolitis  test               0     86           0
               train             19     28          27
COPD           test             417   1718         274
               train            920   1995         422
Healthy        test               0    139          36
               train             84     55           8
LRTI           train              0     32           0
Pneumonia      train             52    121         112
URTI           test              49      0          23
               train              0    167           4


RECORDING OVERLAP BETWEEN CURRENT CNN SPLITS
train vs validation: 16 shared recordings
train vs test: 19 shared recordings
validation vs test: 14 shared recordings


In [13]:

import pandas as pd
from pathlib import Path

root = Path.cwd()
data_dir = root / "data/processed/deep_learning/cnn_baseline"

splits = {}
for name in ["train", "validation", "test"]:
    path = data_dir / f"{name}_cnn_manifest.csv"
    splits[name] = pd.read_csv(path)

print("PATIENT OVERLAP BETWEEN CNN SPLITS")
patient_sets = {
    name: set(df["patient_id"].astype(str))
    for name, df in splits.items()
}

for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    overlap = patient_sets[a] & patient_sets[b]
    print(f"{a} vs {b}: {len(overlap)} shared patients")
    if overlap:
        print("  Examples:", sorted(overlap)[:10])

print("\nOFFICIAL SPLIT LABELS WITHIN EACH CNN SPLIT")
for name, df in splits.items():
    print(f"\n{name.upper()}")
    print(df.groupby("official_split").agg(
        rows=("patient_id", "size"),
        patients=("patient_id", "nunique"),
        recordings=("recording_id", "nunique"),
    ))

print("\nPATIENTS WITH MULTIPLE CNN SPLIT ASSIGNMENTS")
combined = pd.concat(
    [df.assign(current_split=name) for name, df in splits.items()],
    ignore_index=True
)
patient_assignments = combined.groupby("patient_id")[
    "current_split"
].nunique()
print("Patients assigned to multiple splits:",
      int((patient_assignments > 1).sum()))


PATIENT OVERLAP BETWEEN CNN SPLITS
train vs validation: 0 shared patients
train vs test: 0 shared patients
validation vs test: 0 shared patients

OFFICIAL SPLIT LABELS WITHIN EACH CNN SPLIT

TRAIN
                rows  patients  recordings
official_split                            
test            1943        31          23
train           2480        52          27

VALIDATION
                rows  patients  recordings
official_split                            
test             333         7          14
train            581        12           9

TEST
                rows  patients  recordings
official_split                            
test             480        11          10
train           1081        15          16

PATIENTS WITH MULTIPLE CNN SPLIT ASSIGNMENTS
Patients assigned to multiple splits: 0


In [14]:

import pandas as pd
from pathlib import Path

root = Path.cwd()
data_dir = root / "data/processed/deep_learning/cnn_baseline"

frames = []
for split in ["train", "validation", "test"]:
    df = pd.read_csv(data_dir / f"{split}_cnn_manifest.csv")
    df["current_split"] = split
    frames.append(df)

all_data = pd.concat(frames, ignore_index=True)

print("RECORDING ID CONSISTENCY")
recording_check = all_data.groupby("recording_id").agg(
    patients=("patient_id", "nunique"),
    diagnoses=("diagnosis", "nunique"),
    current_splits=("current_split", "nunique"),
    audio_paths=("audio_path", "nunique"),
)
print("Recording IDs mapped to multiple patients:",
      int((recording_check["patients"] > 1).sum()))
print("Recording IDs mapped to multiple diagnoses:",
      int((recording_check["diagnoses"] > 1).sum()))
print("Recording IDs found in multiple CNN splits:",
      int((recording_check["current_splits"] > 1).sum()))
print("Recording IDs mapped to multiple audio paths:",
      int((recording_check["audio_paths"] > 1).sum()))

print("\nCLASS SUPPORT BY CURRENT SPLIT")
support = all_data.groupby(
    ["current_split", "diagnosis"]
).agg(
    cycles=("diagnosis", "size"),
    recordings=("recording_id", "nunique"),
    patients=("patient_id", "nunique"),
).reset_index()

display(support)

print("\nUNIQUE PATIENTS AND RECORDINGS PER SPLIT")
display(all_data.groupby("current_split").agg(
    cycles=("diagnosis", "size"),
    patients=("patient_id", "nunique"),
    recordings=("recording_id", "nunique"),
))


RECORDING ID CONSISTENCY
Recording IDs mapped to multiple patients: 25
Recording IDs mapped to multiple diagnoses: 7
Recording IDs found in multiple CNN splits: 21
Recording IDs mapped to multiple audio paths: 34

CLASS SUPPORT BY CURRENT SPLIT


,current_split,diagnosis,cycles,recordings,patients
0,test,Bronchiectasis,20,2,2
1,test,Bronchiolitis,19,1,1
2,test,COPD,1337,19,13
3,test,Healthy,84,2,5
4,test,Pneumonia,52,3,1
5,test,URTI,49,2,3
6,train,Asthma,6,1,1
7,train,Bronchiectasis,76,3,4
8,train,Bronchiolitis,114,1,4
9,train,COPD,3713,32,41



UNIQUE PATIENTS AND RECORDINGS PER SPLIT


,cycles,patients,recordings
current_split,,,
test,1561,25,19
train,4423,82,32
validation,914,19,18


In [15]:

import pandas as pd
from pathlib import Path
import os

root = Path.cwd()
data_dir = root / "data/processed/deep_learning/cnn_baseline"

frames = {}
for split in ["train", "validation", "test"]:
    df = pd.read_csv(data_dir / f"{split}_cnn_manifest.csv")
    df["current_split"] = split
    df["resolved_audio_path"] = df["audio_path"].apply(
        lambda p: str((root / str(p)).resolve())
        if not Path(str(p)).is_absolute()
        else str(Path(str(p)).resolve())
    )
    frames[split] = df

path_sets = {
    split: set(df["resolved_audio_path"])
    for split, df in frames.items()
}

print("UNIQUE AUDIO PATHS BY SPLIT")
for split, paths in path_sets.items():
    print(f"{split}: {len(paths)}")

print("\nEXACT AUDIO-PATH OVERLAP")
for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    overlap = path_sets[a] & path_sets[b]
    print(f"{a} vs {b}: {len(overlap)} shared paths")
    for path in sorted(overlap)[:5]:
        print(" ", path)

all_df = pd.concat(frames.values(), ignore_index=True)

print("\nPATHS ASSOCIATED WITH MULTIPLE PATIENTS")
path_patient_counts = all_df.groupby(
    "resolved_audio_path"
)["patient_id"].nunique()
print(int((path_patient_counts > 1).sum()))

print("\nMISSING AUDIO PATHS")
missing = sorted({
    p for p in all_df["resolved_audio_path"].unique()
    if not os.path.isfile(p)
})
print("Missing:", len(missing))
for path in missing[:10]:
    print(" ", path)


UNIQUE AUDIO PATHS BY SPLIT
train: 4423
validation: 914
test: 1561

EXACT AUDIO-PATH OVERLAP
train vs validation: 0 shared paths
train vs test: 0 shared paths
validation vs test: 0 shared paths

PATHS ASSOCIATED WITH MULTIPLE PATIENTS
0

MISSING AUDIO PATHS
Missing: 0


In [16]:

import pandas as pd
from pathlib import Path

root = Path.cwd()
data_dir = root / "data/processed/deep_learning/cnn_baseline"

train = pd.read_csv(data_dir / "train_cnn_manifest.csv")
validation = pd.read_csv(data_dir / "validation_cnn_manifest.csv")
test = pd.read_csv(data_dir / "test_cnn_manifest.csv")

print("TRAINING CLASS IMBALANCE")
counts = train["diagnosis"].value_counts().sort_values(ascending=False)
summary = pd.DataFrame({
    "cycles": counts,
    "fraction_percent": (counts / len(train) * 100).round(2),
    "imbalance_vs_largest": (counts.max() / counts).round(1),
})
display(summary)

print("\nINDEPENDENT SUPPORT BY SPLIT")
for split_name, df in [
    ("train", train),
    ("validation", validation),
    ("test", test),
]:
    support = df.groupby("diagnosis").agg(
        cycles=("diagnosis", "size"),
        patients=("patient_id", "nunique"),
        audio_files=("audio_path", "nunique"),
    )
    print(f"\n{split_name.upper()}")
    display(support)

print("\nCLASS-WEIGHT FEASIBILITY")
present = train["diagnosis"].value_counts()
print("Classes represented in training:", len(present))
print("Classes absent from training:", sorted(
    set([
        "Asthma", "Bronchiectasis", "Bronchiolitis", "COPD",
        "Healthy", "LRTI", "Pneumonia", "URTI"
    ]) - set(present.index)
))


TRAINING CLASS IMBALANCE


,cycles,fraction_percent,imbalance_vs_largest
diagnosis,,,
COPD,3713,83.95,1.0
Healthy,194,4.39,19.1
URTI,167,3.78,22.2
Pneumonia,121,2.74,30.7
Bronchiolitis,114,2.58,32.6
Bronchiectasis,76,1.72,48.9
LRTI,32,0.72,116.0
Asthma,6,0.14,618.8



INDEPENDENT SUPPORT BY SPLIT

TRAIN


,cycles,patients,audio_files
diagnosis,,,
Asthma,6,1,6
Bronchiectasis,76,4,76
Bronchiolitis,114,4,114
COPD,3713,41,3713
Healthy,194,17,194
LRTI,32,2,32
Pneumonia,121,4,121
URTI,167,9,167



VALIDATION


,cycles,patients,audio_files
diagnosis,,,
Bronchiectasis,8,1,8
Bronchiolitis,27,1,27
COPD,696,10,696
Healthy,44,4,44
Pneumonia,112,1,112
URTI,27,2,27



TEST


,cycles,patients,audio_files
diagnosis,,,
Bronchiectasis,20,2,20
Bronchiolitis,19,1,19
COPD,1337,13,1337
Healthy,84,5,84
Pneumonia,52,1,52
URTI,49,3,49



CLASS-WEIGHT FEASIBILITY
Classes represented in training: 8
Classes absent from training: []


In [17]:

import json
from pathlib import Path

notebook_path = (
    Path.cwd()
    / "Notebooks 2"
    / "04_ICBHI_Disease_Transfer_Learning.ipynb"
)

if not notebook_path.exists():
    # Also support running from inside the Notebooks 2 directory.
    notebook_path = (
        Path.cwd() / "04_ICBHI_Disease_Transfer_Learning.ipynb"
    )

print("Notebook:", notebook_path.resolve())
print("Exists:", notebook_path.exists())

if notebook_path.exists():
    nb = json.loads(notebook_path.read_text(encoding="utf-8"))

    keywords = [
        "class_weight",
        "CrossEntropyLoss",
        "WeightedRandomSampler",
        "train_loader",
        "class_mapping",
        "optimizer",
        "macro_f1",
        "state_dict",
    ]

    for i, cell in enumerate(nb["cells"]):
        if cell["cell_type"] != "code":
            continue

        source = "".join(cell.get("source", []))
        matches = [kw for kw in keywords if kw.lower() in source.lower()]

        if matches:
            print(f"\n{'=' * 18} CELL {i} {'=' * 18}")
            print("Matched:", ", ".join(matches))
            print(source[:7000])


Notebook: /Users/vs/Sem 5/Respiratory-Disease-Classification/Notebooks 2/04_ICBHI_Disease_Transfer_Learning.ipynb
Exists: True

================== CELL 1 ==================
Matched: class_mapping

from pathlib import Path

ROOT = Path.cwd()

print("PROJECT ROOT:", ROOT)

# Check important folders and existing model artifacts
checks = [
    "data/processed/deep_learning",
    "data/processed/deep_learning/cnn_baseline",
    "data/processed/deep_learning/disease_class_mapping.csv",
    "models",
    "models/multidataset_pretraining/phenotype_encoder_hf_best.pt",
]

print("\nPATH CHECK")
for item in checks:
    path = ROOT / item
    status = "FOUND" if path.exists() else "NOT FOUND"
    print(f"{status:12} {item}")

# Find relevant notebooks
print("\nRELEVANT NOTEBOOKS")
for folder in [ROOT / "Notebooks", ROOT / "Notebooks 2", ROOT / "notebooks"]:
    if folder.exists():
        for nb in sorted(folder.glob("*.ipynb")):
            if any(key in nb.name.lower() for key in [
             

In [18]:

import json
from pathlib import Path

ROOT = Path.cwd()
source_path = ROOT / "Notebooks/12_multifeature_cnn.ipynb"

with open(source_path, "r", encoding="utf-8") as f:
    source_nb = json.load(f)

keywords = [
    "class MultiFeatureCNN",
    "class FeatureBranch",
    "class MultiFeatureRespiratoryDataset",
    "compute_class_weight",
    "CrossEntropyLoss",
    "num_classes",
    "disease_label",
    "class_names",
    "best_multifeature_cnn.pt",
]

for i, cell in enumerate(source_nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    code = "".join(cell.get("source", []))
    matches = [
        key for key in keywords
        if key.lower() in code.lower()
    ]

    if matches:
        print(f"\n{'=' * 20} SOURCE CELL {i} {'=' * 20}")
        print("Matches:", ", ".join(matches))
        print(code[:5000])
        if len(code) > 5000:
            print("... [truncated] ...")



==================== SOURCE CELL 1 ====================
Matches: compute_class_weight
from pathlib import Path
import random
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import soundfile as sf
import librosa

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("PyTorch version:", torch.__version__)
print("Librosa version:", librosa.__version__)

==================== SOURCE CELL 6 ====================
Matches: num_classes, disease_label, class_names
class_mapping_path = (
    DL_DIR / "disease_class_mapping.csv"
)

class_mapping = pd.read_csv(
    class_mapping_path
)

class_names = (
    class_mapping
    .sort_values("disease_label")["diagn

In [19]:

import json
from pathlib import Path

ROOT = Path.cwd()
source_path = ROOT / "Notebooks/12_multifeature_cnn.ipynb"

with open(source_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

terms = [
    "BATCH_SIZE",
    "train_loader =",
    "DataLoader(",
    "WeightedRandomSampler",
    "optimizer =",
    "LEARNING_RATE",
    "WEIGHT_DECAY",
    "for epoch in range",
    "criterion(",
    "val_loss",
    "EARLY_STOPPING_PATIENCE",
]

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    code = "".join(cell.get("source", []))
    matches = [term for term in terms if term.lower() in code.lower()]

    if matches:
        print(f"\n{'=' * 18} CELL {i} {'=' * 18}")
        print("Matched:", ", ".join(matches))
        print(code[:4500])
        if len(code) > 4500:
            print("... [truncated] ...")



================== CELL 38 ==================
Matched: BATCH_SIZE, train_loader =, DataLoader(
BATCH_SIZE = 32
NUM_WORKERS = 0  # safer for macOS/Jupyter

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False
)

print("DataLoaders created.")
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

================== CELL 43 ==================
Matched: BATCH_SIZE
print("=== Multi-Feature DataLoader Summary ===")
print(f"Batch size: {BATCH_SIZE}")
print(f"Number of workers: {NUM_WORKERS}")
print(f"Train samples: {len(train_dataset)}")
print(f"Validat

In [20]:

import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
DATA_DIR = ROOT / "data/processed/deep_learning/cnn_baseline"

mapping = pd.read_csv(
    ROOT / "data/processed/deep_learning/disease_class_mapping.csv"
).sort_values("disease_label")

train = pd.read_csv(DATA_DIR / "train_cnn_manifest.csv")
validation = pd.read_csv(DATA_DIR / "validation_cnn_manifest.csv")

print("SAVED CLASS MAPPING")
display(mapping[["disease_label", "diagnosis"]])

print("\nTRAIN LABEL COUNTS")
train_counts = train["disease_label"].astype(int).value_counts().sort_index()
display(train_counts.rename("cycles").to_frame())

print("\nVALIDATION LABEL COUNTS")
val_counts = (
    validation["disease_label"].astype(int)
    .value_counts()
    .sort_index()
)
display(val_counts.rename("cycles").to_frame())

expected_labels = set(mapping["disease_label"].astype(int))
observed_labels = set(train["disease_label"].astype(int))

print("\nChecks:")
print("Mapping labels:", sorted(expected_labels))
print("Training labels:", sorted(observed_labels))
print("All expected labels represented:", expected_labels == observed_labels)
print("Missing labels:", sorted(expected_labels - observed_labels))
print("Unexpected labels:", sorted(observed_labels - expected_labels))


SAVED CLASS MAPPING


,disease_label,diagnosis
0,0,Asthma
1,1,Bronchiectasis
2,2,Bronchiolitis
3,3,COPD
4,4,Healthy
5,5,LRTI
6,6,Pneumonia
7,7,URTI



TRAIN LABEL COUNTS


,cycles
disease_label,
0,6
1,76
2,114
3,3713
4,194
5,32
6,121
7,167



VALIDATION LABEL COUNTS


,cycles
disease_label,
1,8
2,27
3,696
4,44
6,112
7,27



Checks:
Mapping labels: [0, 1, 2, 3, 4, 5, 6, 7]
Training labels: [0, 1, 2, 3, 4, 5, 6, 7]
All expected labels represented: True
Missing labels: []
Unexpected labels: []


In [21]:

import json
from pathlib import Path

ROOT = Path.cwd()

EXPERIMENT_DIR = (
    ROOT
    / "models"
    / "deep_learning"
    / "imbalance_comparison"
)

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "experiment_group": "disease_imbalance_comparison",
    "classes": [
        "Asthma",
        "Bronchiectasis",
        "Bronchiolitis",
        "COPD",
        "Healthy",
        "LRTI",
        "Pneumonia",
        "URTI",
    ],
    "split_policy": "existing_patient_disjoint_cnn_manifests",
    "experiment_A": {
        "name": "weighted_cross_entropy",
        "sampling": "shuffle",
        "loss": "class_weighted_cross_entropy",
    },
    "experiment_B": {
        "name": "balanced_sampling",
        "sampling": "WeightedRandomSampler",
        "loss": "unweighted_cross_entropy",
    },
    "checkpoint_selection": "validation_macro_f1",
    "additional_validation_metric": "validation_loss",
    "test_policy": "do_not_use_for_model_selection",
    "preserve_existing_artifacts": True,
}

config_path = EXPERIMENT_DIR / "experiment_config.json"

if config_path.exists():
    print("Existing configuration found; not overwriting it.")
    print(config_path)
else:
    config_path.write_text(
        json.dumps(CONFIG, indent=2),
        encoding="utf-8",
    )
    print("Created experiment directory:")
    print(EXPERIMENT_DIR)
    print("\nSaved configuration:")
    print(config_path)

print("\nExisting model checkpoint directories remain untouched.")


Created experiment directory:
/Users/vs/Sem 5/Respiratory-Disease-Classification/models/deep_learning/imbalance_comparison

Saved configuration:
/Users/vs/Sem 5/Respiratory-Disease-Classification/models/deep_learning/imbalance_comparison/experiment_config.json

Existing model checkpoint directories remain untouched.


In [22]:

import json
from pathlib import Path

ROOT = Path.cwd()

EXPERIMENT_DIR = (
    ROOT / "models" / "deep_learning" / "imbalance_comparison"
)

RESULTS_DIR = (
    ROOT / "data" / "processed" / "deep_learning"
    / "imbalance_comparison"
)

original_checkpoints = {
    "Lightweight Mel CNN":
        ROOT / "models/deep_learning/lightweight_mel_cnn/best_lightweight_mel_cnn.pt",
    "Multi-Feature CNN":
        ROOT / "models/deep_learning/multifeature_cnn/best_multifeature_cnn.pt",
    "Regularized Multi-Feature CNN":
        ROOT / "models/deep_learning/regularized_multifeature_cnn/best_regularized_multifeature_cnn.pt",
}

config_path = EXPERIMENT_DIR / "experiment_config.json"

print("EXPERIMENT DIRECTORY:", EXPERIMENT_DIR)
print("Exists:", EXPERIMENT_DIR.exists())

if config_path.exists():
    config = json.loads(config_path.read_text(encoding="utf-8"))
    print("\nEXPERIMENT CONFIGURATION")
    print(json.dumps(config, indent=2))
else:
    print("\nConfiguration not found.")
    print("Run the Step 22 setup cell first.")

print("\nORIGINAL CHECKPOINTS")
for name, path in original_checkpoints.items():
    print(f"{name}: {'FOUND' if path.exists() else 'NOT FOUND'}")

print("\nRESULTS DIRECTORY:", RESULTS_DIR)
print("Exists:", RESULTS_DIR.exists())


EXPERIMENT DIRECTORY: /Users/vs/Sem 5/Respiratory-Disease-Classification/models/deep_learning/imbalance_comparison
Exists: True

EXPERIMENT CONFIGURATION
{
  "experiment_group": "disease_imbalance_comparison",
  "classes": [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "LRTI",
    "Pneumonia",
    "URTI"
  ],
  "split_policy": "existing_patient_disjoint_cnn_manifests",
  "experiment_A": {
    "name": "weighted_cross_entropy",
    "sampling": "shuffle",
    "loss": "class_weighted_cross_entropy"
  },
  "experiment_B": {
    "name": "balanced_sampling",
    "sampling": "WeightedRandomSampler",
    "loss": "unweighted_cross_entropy"
  },
  "checkpoint_selection": "validation_macro_f1",
  "additional_validation_metric": "validation_loss",
  "test_policy": "do_not_use_for_model_selection",
  "preserve_existing_artifacts": true
}

ORIGINAL CHECKPOINTS
Lightweight Mel CNN: FOUND
Multi-Feature CNN: FOUND
Regularized Multi-Feature CNN: FOUND

RESULTS D

In [24]:

from pathlib import Path
import pandas as pd

ROOT = Path.cwd()

EXPERIMENT_DIR = (
    ROOT / "models" / "deep_learning" / "imbalance_comparison"
)
RESULTS_DIR = (
    ROOT / "data" / "processed" / "deep_learning"
    / "imbalance_comparison"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

required_files = {
    "Training manifest":
        ROOT / "data/processed/deep_learning/cnn_baseline/train_cnn_manifest.csv",
    "Validation manifest":
        ROOT / "data/processed/deep_learning/cnn_baseline/validation_cnn_manifest.csv",
    "Test manifest":
        ROOT / "data/processed/deep_learning/cnn_baseline/test_cnn_manifest.csv",
    "Class mapping":
        ROOT / "data/processed/deep_learning/disease_class_mapping.csv",
    "Feature extraction notebook":
        ROOT / "Notebooks/12_multifeature_cnn.ipynb",
}

print("RESULTS DIRECTORY READY:", RESULTS_DIR.exists())
print("\nREQUIRED FILES")

for name, path in required_files.items():
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}")

if not all(path.exists() for path in required_files.values()):
    raise FileNotFoundError(
        "One or more required files are missing. "
        "Resolve the paths before continuing."
    )

print("\nMANIFEST SUMMARY")
for name, path in list(required_files.items())[:3]:
    df = pd.read_csv(path)
    print(
        f"{name}: {len(df)} cycles, "
        f"{df['patient_id'].nunique()} patients, "
        f"{df['audio_path'].nunique()} audio paths"
    )

print("\nNo existing checkpoint or metric file was changed.")


RESULTS DIRECTORY READY: True

REQUIRED FILES
Training manifest: FOUND
Validation manifest: FOUND
Test manifest: FOUND
Class mapping: FOUND
Feature extraction notebook: FOUND

MANIFEST SUMMARY
Training manifest: 4423 cycles, 82 patients, 4423 audio paths
Validation manifest: 914 cycles, 19 patients, 914 audio paths
Test manifest: 1561 cycles, 25 patients, 1561 audio paths

No existing checkpoint or metric file was changed.


In [25]:

import json
from pathlib import Path

ROOT = Path.cwd()
source_path = ROOT / "Notebooks/12_multifeature_cnn.ipynb"

with open(source_path, "r", encoding="utf-8") as f:
    source_nb = json.load(f)

terms = [
    "def load_processed_audio",
    "def extract_logmel",
    "def extract_mfcc",
    "def extract_chroma",
    "def augment_training_audio",
    "class MultiFeatureRespiratoryDataset",
    "logmel_mean",
    "mfcc_mean",
    "chroma_mean",
]

for i, cell in enumerate(source_nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    code = "".join(cell.get("source", []))
    matches = [
        term for term in terms
        if term.lower() in code.lower()
    ]

    if matches:
        print(f"\n{'=' * 16} SOURCE CELL {i} {'=' * 16}")
        print("Matched:", ", ".join(matches))
        print(code[:5500])
        if len(code) > 5500:
            print("... [truncated] ...")



================ SOURCE CELL 11 ================
Matched: def load_processed_audio
def load_processed_audio(audio_path):
    signal, sr = sf.read(audio_path)

    if signal.ndim > 1:
        signal = np.mean(signal, axis=1)

    if sr != TARGET_SR:
        raise ValueError(
            f"Expected {TARGET_SR} Hz, got {sr} Hz"
        )

    signal = signal.astype(np.float32)

    signal = fix_audio_length(
        signal,
        TARGET_SAMPLES
    )

    return signal

================ SOURCE CELL 12 ================
Matched: def extract_logmel
def extract_logmel(signal):
    mel_power = librosa.feature.melspectrogram(
        y=signal,
        sr=TARGET_SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=FMAX,
        power=2.0
    )

    mel_db = librosa.power_to_db(
        mel_power,
        ref=np.max
    )

    return mel_db.astype(np.float32)

================ SOURCE CELL 13 ================
Matched: def extract_mfcc
N_

In [26]:

import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()

# Inspect the original notebook for the feature settings and
# the path used to save normalization statistics.
source_path = ROOT / "Notebooks/12_multifeature_cnn.ipynb"
nb = json.loads(source_path.read_text(encoding="utf-8"))

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    code = "".join(cell.get("source", []))
    if (
        "normalization_df.to_csv" in code
        or "NORMALIZATION_PATH" in code
        or "TARGET_SR =" in code
        or "TARGET_SAMPLES =" in code
    ):
        print(f"\n--- SOURCE CELL {i} ---")
        print(code[:3000])

print("\nEXISTING MULTI-FEATURE ARTIFACTS")
artifact_dir = ROOT / "data/processed/deep_learning/multifeature_cnn"

if artifact_dir.exists():
    for path in sorted(artifact_dir.iterdir()):
        if path.is_file():
            print(path.name, f"({path.stat().st_size:,} bytes)")
else:
    print("Artifact directory not found:", artifact_dir)



--- SOURCE CELL 5 ---
TARGET_SR = 4000
TARGET_DURATION = 5.0
TARGET_SAMPLES = int(
    TARGET_SR * TARGET_DURATION
)

N_FFT = 512
HOP_LENGTH = 128
N_MELS = 64

FMIN = 0
FMAX = 2000

print("Target SR:", TARGET_SR)
print("Target duration:", TARGET_DURATION)
print("Target samples:", TARGET_SAMPLES)
print("N_FFT:", N_FFT)
print("Hop length:", HOP_LENGTH)
print("Mel bands:", N_MELS)
print("Frequency range:", FMIN, "-", FMAX, "Hz")

--- SOURCE CELL 26 ---
normalization_path = (
    MULTIFEATURE_DIR /
    "multifeature_train_normalization.csv"
)

normalization_df.to_csv(
    normalization_path,
    index=False
)

print("Saved normalization statistics to:")
print(normalization_path)

EXISTING MULTI-FEATURE ARTIFACTS
best_multifeature_cnn.pt (464,538 bytes)
cnn_baseline_vs_multifeature_comparison.csv (485 bytes)
disease_level_cnn_comparison.csv (478 bytes)
final_cnn_experiment_comparison.csv (576 bytes)
multifeature_cnn_confusion_matrix.csv (275 bytes)
multifeature_cnn_metrics.csv (288 bytes)


In [27]:

from pathlib import Path
import pandas as pd
import soundfile as sf
import numpy as np

ROOT = Path.cwd()

normalization_path = (
    ROOT / "data/processed/deep_learning/multifeature_cnn"
    / "multifeature_train_normalization.csv"
)

train_manifest_path = (
    ROOT / "data/processed/deep_learning/cnn_baseline"
    / "train_cnn_manifest.csv"
)

normalization_df = pd.read_csv(normalization_path)
train_manifest = pd.read_csv(train_manifest_path)

print("NORMALIZATION STATISTICS")
display(normalization_df)

required_columns = {"feature", "mean", "std"}
if not required_columns.issubset(normalization_df.columns):
    raise ValueError(
        f"Expected columns {required_columns}; "
        f"found {set(normalization_df.columns)}"
    )

expected_features = {"logmel", "mfcc", "chroma"}
if set(normalization_df["feature"]) != expected_features:
    raise ValueError("Unexpected or missing feature names.")

if (
    not np.isfinite(normalization_df[["mean", "std"]].to_numpy()).all()
    or (normalization_df["std"] <= 0).any()
):
    raise ValueError("Invalid normalization mean/std values.")

print("\nTRAINING MANIFEST SAMPLE")
row = train_manifest.iloc[0]
audio_path = Path(str(row["audio_path"]))

if not audio_path.is_absolute():
    audio_path = (ROOT / audio_path).resolve()

print("Diagnosis:", row["diagnosis"])
print("Patient:", row["patient_id"])
print("Audio:", audio_path)

if not audio_path.is_file():
    raise FileNotFoundError(audio_path)

signal, sr = sf.read(audio_path)

if signal.ndim > 1:
    signal = signal.mean(axis=1)

print("\nAUDIO CHECK")
print("Sample rate:", sr)
print("Samples:", len(signal))
print("Duration (seconds):", round(len(signal) / sr, 3))
print("Finite waveform:", bool(np.isfinite(signal).all()))

if sr != 4000:
    raise ValueError(f"Expected 4000 Hz audio, found {sr} Hz.")

print("\nPASS: saved normalization and sample audio are valid.")


NORMALIZATION STATISTICS


,feature,count,mean,std
0,logmel,44442304,-66.230481,17.661886
1,mfcc,9027343,-22.509034,137.274416
2,chroma,8332932,0.385656,0.389059



TRAINING MANIFEST SAMPLE
Diagnosis: Healthy
Patient: 102
Audio: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/audio/cycles/102/1b1_Ar_001.wav

AUDIO CHECK
Sample rate: 4000
Samples: 5889
Duration (seconds): 1.472
Finite waveform: True

PASS: saved normalization and sample audio are valid.


In [28]:

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from pathlib import Path

ROOT = Path.cwd()

TARGET_SR = 4000
TARGET_DURATION = 5.0
TARGET_SAMPLES = int(TARGET_SR * TARGET_DURATION)

N_FFT = 512
HOP_LENGTH = 128
N_MELS = 64
N_MFCC = 13
N_CHROMA = 12
FMIN = 0
FMAX = 2000

# Reuse the exact length-fixing logic from the source notebook.
source_nb = __import__("json").loads(
    (ROOT / "Notebooks/12_multifeature_cnn.ipynb")
    .read_text(encoding="utf-8")
)

fix_code = None
for cell in source_nb["cells"]:
    if cell.get("cell_type") == "code":
        code = "".join(cell.get("source", []))
        if "def fix_audio_length" in code:
            fix_code = code
            break

if fix_code is None:
    raise RuntimeError("Could not locate fix_audio_length in source notebook.")

print("Found original fix_audio_length function:")
print(fix_code[:1500])


Found original fix_audio_length function:
def fix_audio_length(signal, target_samples):
    if len(signal) < target_samples:
        signal = np.pad(
            signal,
            (0, target_samples - len(signal)),
            mode="constant"
        )
    else:
        signal = signal[:target_samples]

    return signal.astype(np.float32)


In [29]:

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from pathlib import Path

ROOT = Path.cwd()

TARGET_SR = 4000
TARGET_SAMPLES = 20_000
N_FFT = 512
HOP_LENGTH = 128
N_MELS = 64
N_MFCC = 13
N_CHROMA = 12
FMIN = 0
FMAX = 2000

def fix_audio_length(signal, target_samples):
    if len(signal) < target_samples:
        signal = np.pad(
            signal,
            (0, target_samples - len(signal)),
            mode="constant",
        )
    else:
        signal = signal[:target_samples]
    return signal.astype(np.float32)

train_manifest = pd.read_csv(
    ROOT / "data/processed/deep_learning/cnn_baseline/train_cnn_manifest.csv"
)

audio_path = Path(str(train_manifest.iloc[0]["audio_path"]))
if not audio_path.is_absolute():
    audio_path = (ROOT / audio_path).resolve()

signal, sr = sf.read(audio_path)

if signal.ndim > 1:
    signal = np.mean(signal, axis=1)

if sr != TARGET_SR:
    raise ValueError(f"Expected {TARGET_SR} Hz, got {sr} Hz")

signal = fix_audio_length(signal, TARGET_SAMPLES)

logmel = librosa.power_to_db(
    librosa.feature.melspectrogram(
        y=signal,
        sr=TARGET_SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=FMAX,
        power=2.0,
    ),
    ref=np.max,
).astype(np.float32)

mfcc = librosa.feature.mfcc(
    y=signal,
    sr=TARGET_SR,
    n_mfcc=N_MFCC,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    fmin=FMIN,
    fmax=FMAX,
).astype(np.float32)

chroma = librosa.feature.chroma_stft(
    y=signal,
    sr=TARGET_SR,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_chroma=N_CHROMA,
    tuning=0.0,
).astype(np.float32)

print("Audio shape:", signal.shape)
print("Log-Mel shape:", logmel.shape)
print("MFCC shape:", mfcc.shape)
print("Chroma shape:", chroma.shape)

for name, feature in [
    ("Log-Mel", logmel),
    ("MFCC", mfcc),
    ("Chroma", chroma),
]:
    print(
        f"{name}: finite={np.isfinite(feature).all()}, "
        f"mean={feature.mean():.4f}, std={feature.std():.4f}"
    )

assert signal.shape == (TARGET_SAMPLES,)
assert logmel.shape[0] == N_MELS
assert mfcc.shape[0] == N_MFCC
assert chroma.shape[0] == N_CHROMA
assert all(
    np.isfinite(x).all()
    for x in [logmel, mfcc, chroma]
)

print("\nPASS: all three features extracted successfully.")


Audio shape: (20000,)
Log-Mel shape: (64, 157)
MFCC shape: (13, 157)
Chroma shape: (12, 157)
Log-Mel: finite=True, mean=-73.6370, std=12.2206
MFCC: finite=True, mean=-33.1251, std=146.6715
Chroma: finite=True, mean=0.2449, std=0.3793

PASS: all three features extracted successfully.


In [30]:

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import soundfile as sf
import librosa

ROOT = Path.cwd()

DATA_DIR = ROOT / "data/processed/deep_learning/cnn_baseline"
NORMALIZATION_PATH = (
    ROOT / "data/processed/deep_learning/multifeature_cnn"
    / "multifeature_train_normalization.csv"
)

train_manifest = pd.read_csv(DATA_DIR / "train_cnn_manifest.csv")
val_manifest = pd.read_csv(DATA_DIR / "validation_cnn_manifest.csv")
test_manifest = pd.read_csv(DATA_DIR / "test_cnn_manifest.csv")
normalization_df = pd.read_csv(NORMALIZATION_PATH)

norm = normalization_df.set_index("feature")
logmel_mean, logmel_std = float(norm.loc["logmel", "mean"]), float(norm.loc["logmel", "std"])
mfcc_mean, mfcc_std = float(norm.loc["mfcc", "mean"]), float(norm.loc["mfcc", "std"])
chroma_mean, chroma_std = float(norm.loc["chroma", "mean"]), float(norm.loc["chroma", "std"])

TARGET_SR = 4000
TARGET_SAMPLES = 20_000
N_FFT = 512
HOP_LENGTH = 128
N_MELS = 64
N_MFCC = 13
N_CHROMA = 12

def _fix_length(signal, target_samples=TARGET_SAMPLES):
    if len(signal) < target_samples:
        signal = np.pad(signal, (0, target_samples - len(signal)))
    else:
        signal = signal[:target_samples]
    return signal.astype(np.float32)

def _load_audio(audio_path):
    path = Path(str(audio_path))
    if not path.is_absolute():
        path = (ROOT / path).resolve()

    signal, sr = sf.read(path)
    if signal.ndim > 1:
        signal = signal.mean(axis=1)
    if sr != TARGET_SR:
        raise ValueError(f"Expected {TARGET_SR} Hz, got {sr}: {path}")

    return _fix_length(signal)

def _augment_audio(signal):
    max_shift = int(len(signal) * 0.10)
    shift = np.random.randint(-max_shift, max_shift + 1)
    if shift:
        shifted = np.zeros_like(signal)
        if shift > 0:
            shifted[shift:] = signal[:-shift]
        else:
            shifted[:shift] = signal[-shift:]
        signal = shifted

    noise = np.random.normal(0, 1, signal.shape).astype(np.float32)
    return (signal + 0.005 * noise).astype(np.float32)

def _extract_features(signal):
    logmel = librosa.power_to_db(
        librosa.feature.melspectrogram(
            y=signal, sr=TARGET_SR, n_fft=N_FFT,
            hop_length=HOP_LENGTH, n_mels=N_MELS,
            fmin=0, fmax=2000, power=2.0
        ),
        ref=np.max
    ).astype(np.float32)

    mfcc = librosa.feature.mfcc(
        y=signal, sr=TARGET_SR, n_mfcc=N_MFCC,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
        fmin=0, fmax=2000
    ).astype(np.float32)

    chroma = librosa.feature.chroma_stft(
        y=signal, sr=TARGET_SR, n_fft=N_FFT,
        hop_length=HOP_LENGTH, n_chroma=N_CHROMA,
        tuning=0.0
    ).astype(np.float32)

    logmel = (logmel - logmel_mean) / logmel_std
    mfcc = (mfcc - mfcc_mean) / mfcc_std
    chroma = (chroma - chroma_mean) / chroma_std

    return tuple(
        torch.from_numpy(x[np.newaxis].astype(np.float32))
        for x in (logmel, mfcc, chroma)
    )

class DiseaseMultiFeatureDataset(Dataset):
    def __init__(self, manifest, augment=False):
        self.manifest = manifest.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, index):
        row = self.manifest.iloc[index]
        signal = _load_audio(row["audio_path"])

        if self.augment:
            signal = _augment_audio(signal)

        features = _extract_features(signal)
        label = torch.tensor(int(row["disease_label"]), dtype=torch.long)
        return (*features, label)

train_dataset = DiseaseMultiFeatureDataset(train_manifest, augment=True)
val_dataset = DiseaseMultiFeatureDataset(val_manifest, augment=False)
test_dataset = DiseaseMultiFeatureDataset(test_manifest, augment=False)

BATCH_SIZE = 32
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=False
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=False
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=False
)

print("Datasets and loaders created.")
print("Train:", len(train_dataset), "| batches:", len(train_loader))
print("Validation:", len(val_dataset), "| batches:", len(val_loader))
print("Test:", len(test_dataset), "| batches:", len(test_loader))


Datasets and loaders created.
Train: 4423 | batches: 139
Validation: 914 | batches: 29
Test: 1561 | batches: 49


In [31]:

import torch

batch = next(iter(train_loader))
logmel_batch, mfcc_batch, chroma_batch, labels_batch = batch

print("Log-Mel batch:", tuple(logmel_batch.shape))
print("MFCC batch:", tuple(mfcc_batch.shape))
print("Chroma batch:", tuple(chroma_batch.shape))
print("Labels batch:", tuple(labels_batch.shape))
print("Label dtype:", labels_batch.dtype)
print("Labels present:", sorted(labels_batch.unique().tolist()))

for name, tensor in [
    ("Log-Mel", logmel_batch),
    ("MFCC", mfcc_batch),
    ("Chroma", chroma_batch),
]:
    print(
        f"{name}: finite={torch.isfinite(tensor).all().item()}, "
        f"mean={tensor.mean().item():.4f}, "
        f"std={tensor.std().item():.4f}"
    )

assert logmel_batch.shape == (32, 1, 64, 157)
assert mfcc_batch.shape == (32, 1, 13, 157)
assert chroma_batch.shape == (32, 1, 12, 157)
assert labels_batch.shape == (32,)
assert labels_batch.dtype == torch.long

print("\nPASS: batch shapes, labels, and features are valid.")


Log-Mel batch: (32, 1, 64, 157)
MFCC batch: (32, 1, 13, 157)
Chroma batch: (32, 1, 12, 157)
Labels batch: (32,)
Label dtype: torch.int64
Labels present: [1, 2, 3, 4, 5, 6]
Log-Mel: finite=True, mean=0.8070, std=0.5722
MFCC: finite=True, mean=0.0668, std=0.6793
Chroma: finite=True, mean=0.8116, std=0.5244

PASS: batch shapes, labels, and features are valid.


In [32]:

import torch
import torch.nn as nn

class FeatureBranch(nn.Module):
    def __init__(self, output_dim=64):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, output_dim),
            nn.ReLU(),
            nn.Dropout(0.30)
        )

    def forward(self, x):
        return self.projection(self.features(x))


class MultiFeatureCNN(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()

        self.logmel_branch = FeatureBranch(output_dim=64)
        self.mfcc_branch = FeatureBranch(output_dim=64)
        self.chroma_branch = FeatureBranch(output_dim=64)

        self.classifier = nn.Sequential(
            nn.Linear(192, 128),
            nn.ReLU(),
            nn.Dropout(0.40),
            nn.Linear(128, num_classes)
        )

    def forward(self, logmel, mfcc, chroma):
        logmel_features = self.logmel_branch(logmel)
        mfcc_features = self.mfcc_branch(mfcc)
        chroma_features = self.chroma_branch(chroma)

        combined = torch.cat(
            [logmel_features, mfcc_features, chroma_features],
            dim=1
        )

        return self.classifier(combined)


# Use the available hardware where possible.
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = MultiFeatureCNN(num_classes=8).to(device)

# Test the model with the batch already validated in Step 31.
model.eval()
with torch.no_grad():
    test_logits = model(
        logmel_batch.to(device),
        mfcc_batch.to(device),
        chroma_batch.to(device)
    )

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Device:", device)
print("Output shape:", tuple(test_logits.shape))
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print("Output finite:", bool(torch.isfinite(test_logits).all()))

assert test_logits.shape == (32, 8)
assert torch.isfinite(test_logits).all()

print("\nPASS: model forward pass is valid.")


Device: mps
Output shape: (32, 8)
Total parameters: 108,776
Trainable parameters: 108,776
Output finite: True

PASS: model forward pass is valid.


In [33]:

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.utils.class_weight import compute_class_weight
from pathlib import Path

# Reproducibility for sampler generation.
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

NUM_CLASSES = 8
BATCH_SIZE = 32

# Calculate counts using the training manifest only.
class_names = [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "LRTI",
    "Pneumonia",
    "URTI"
]

train_labels = train_manifest["disease_label"].to_numpy(dtype=np.int64)
class_ids = np.arange(NUM_CLASSES)

class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=class_ids,
    y=train_labels
)

class_weights_tensor = torch.tensor(
    class_weights, dtype=torch.float32, device=device
)

# Per-example sampling weights: inverse of that class's training count.
sample_weights = 1.0 / class_counts[train_labels]

sampler_generator = torch.Generator()
sampler_generator.manual_seed(SEED)

balanced_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_labels),
    replacement=True,
    generator=sampler_generator
)

# Experiment A: ordinary shuffled sampling + weighted loss.
train_loader_A = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

# Experiment B: class-balanced sampling + unweighted loss.
train_loader_B = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=balanced_sampler,
    num_workers=0,
    pin_memory=False
)

# Keep validation and test ordering deterministic.
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=0, pin_memory=False
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=0, pin_memory=False
)

class_summary = pd.DataFrame({
    "class_id": class_ids,
    "class_name": class_names,
    "train_cycles": class_counts,
    "weighted_CE_weight": class_weights
})

print("Training class distribution:")
display(class_summary)

print("\nExperiment A batches:", len(train_loader_A))
print("Experiment B batches:", len(train_loader_B))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))
print("Sampler draws per epoch:", len(balanced_sampler))
print("Class weights finite:", bool(np.isfinite(class_weights).all()))
print("All classes represented in training:", bool((class_counts > 0).all()))

assert len(class_counts) == NUM_CLASSES
assert (class_counts > 0).all()
assert np.isfinite(class_weights).all()
assert len(train_loader_A) == len(train_loader_B)

print("\nPASS: both imbalance strategies are prepared.")


Training class distribution:


,class_id,class_name,train_cycles,weighted_CE_weight
0,0,Asthma,6,92.145833
1,1,Bronchiectasis,76,7.274671
2,2,Bronchiolitis,114,4.849781
3,3,COPD,3713,0.148903
4,4,Healthy,194,2.849871
5,5,LRTI,32,17.277344
6,6,Pneumonia,121,4.569215
7,7,URTI,167,3.310629



Experiment A batches: 139
Experiment B batches: 139
Validation batches: 29
Test batches: 49
Sampler draws per epoch: 4423
Class weights finite: True
All classes represented in training: True

PASS: both imbalance strategies are prepared.


In [34]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support
)

criterion_unweighted = nn.CrossEntropyLoss()

criterion_weighted = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


def run_training_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_examples = 0
    all_targets = []
    all_predictions = []

    for logmel, mfcc, chroma, labels in loader:
        logmel = logmel.to(device)
        mfcc = mfcc.to(device)
        chroma = chroma.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(logmel, mfcc, chroma)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

        predictions = logits.argmax(dim=1)
        all_targets.extend(labels.detach().cpu().numpy())
        all_predictions.extend(predictions.detach().cpu().numpy())

    y_true = np.asarray(all_targets)
    y_pred = np.asarray(all_predictions)

    return {
        "loss": total_loss / total_examples,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true, y_pred, labels=np.arange(NUM_CLASSES),
            average="macro", zero_division=0
        )
    }


@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()

    total_loss = 0.0
    total_examples = 0
    all_targets = []
    all_predictions = []

    for logmel, mfcc, chroma, labels in loader:
        logmel = logmel.to(device)
        mfcc = mfcc.to(device)
        chroma = chroma.to(device)
        labels = labels.to(device)

        logits = model(logmel, mfcc, chroma)
        loss = criterion_unweighted(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

        predictions = logits.argmax(dim=1)
        all_targets.extend(labels.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())

    y_true = np.asarray(all_targets)
    y_pred = np.asarray(all_predictions)

    # Only classes with actual validation support contribute to the
    # checkpoint-selection macro-F1.
    supported_labels = np.unique(y_true)

    supported_macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=supported_labels,
        average="macro",
        zero_division=0
    )

    precision, recall, class_f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=np.arange(NUM_CLASSES),
            zero_division=0
        )
    )

    per_class_rows = []
    for i, name in enumerate(class_names):
        has_support = support[i] > 0

        per_class_rows.append({
            "class_id": i,
            "class_name": name,
            "support": int(support[i]),
            "precision": float(precision[i]) if has_support else np.nan,
            "recall": float(recall[i]) if has_support else np.nan,
            "f1": float(class_f1[i]) if has_support else np.nan
        })

    metrics = {
        "loss": total_loss / total_examples,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy_supported": balanced_accuracy_score(
            y_true, y_pred
        ),
        "macro_f1_supported_classes": supported_macro_f1,
        "supported_validation_classes": int(len(supported_labels))
    }

    return metrics, pd.DataFrame(per_class_rows), y_true, y_pred


print("Training and evaluation functions defined.")
print("Checkpoint metric: validation macro-F1 over supported classes.")
print("Validation loss: diagnostic only.")
print("Test set: not used for checkpoint selection.")


Training and evaluation functions defined.
Checkpoint metric: validation macro-F1 over supported classes.
Validation loss: diagnostic only.
Test set: not used for checkpoint selection.


In [35]:

import json
import random
from pathlib import Path
import numpy as np
import torch

ROOT = Path.cwd()

EXPERIMENT_DIR = ROOT / "models/deep_learning/imbalance_comparison"
RESULTS_DIR = ROOT / "data/processed/deep_learning/imbalance_comparison"

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Separate output files for each experiment.
EXPERIMENTS = {
    "A": {
        "name": "weighted_ce",
        "checkpoint": EXPERIMENT_DIR / "experiment_A_weighted_ce_best.pt",
        "history": RESULTS_DIR / "experiment_A_weighted_ce_history.csv",
        "validation_metrics": RESULTS_DIR / "experiment_A_weighted_ce_validation_metrics.json",
        "validation_report": RESULTS_DIR / "experiment_A_weighted_ce_validation_report.csv",
        "sampling": "shuffle",
        "loss": "weighted_cross_entropy"
    },
    "B": {
        "name": "balanced_sampler",
        "checkpoint": EXPERIMENT_DIR / "experiment_B_balanced_sampler_best.pt",
        "history": RESULTS_DIR / "experiment_B_balanced_sampler_history.csv",
        "validation_metrics": RESULTS_DIR / "experiment_B_balanced_sampler_validation_metrics.json",
        "validation_report": RESULTS_DIR / "experiment_B_balanced_sampler_validation_report.csv",
        "sampling": "weighted_random_sampler",
        "loss": "unweighted_cross_entropy"
    }
}

# Training configuration.
MAX_EPOCHS = 40
PATIENCE = 7
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

# Record the configuration used for reproducibility.
config = {
    "experiment_group": "disease_imbalance_comparison",
    "seed": SEED,
    "num_classes": NUM_CLASSES,
    "class_names": class_names,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": PATIENCE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "checkpoint_selection": "validation_macro_f1_supported_classes",
    "validation_loss_role": "diagnostic_only",
    "test_used_for_model_selection": False,
    "device": str(device),
    "experiments": {
        key: {
            "name": value["name"],
            "sampling": value["sampling"],
            "loss": value["loss"],
            "checkpoint": str(value["checkpoint"].relative_to(ROOT)),
            "history": str(value["history"].relative_to(ROOT))
        }
        for key, value in EXPERIMENTS.items()
    }
}

config_path = EXPERIMENT_DIR / "imbalance_comparison_config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("Experiment configuration saved:", config_path)
print("Device:", device)
print("Epoch limit:", MAX_EPOCHS)
print("Early stopping patience:", PATIENCE)

for key, exp in EXPERIMENTS.items():
    print(f"\nExperiment {key}: {exp['name']}")
    print("  Checkpoint:", exp["checkpoint"])
    print("  Checkpoint already exists:", exp["checkpoint"].exists())
    print("  History:", exp["history"])

print("\nConfiguration ready. No training has started.")


Experiment configuration saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/models/deep_learning/imbalance_comparison/imbalance_comparison_config.json
Device: mps
Epoch limit: 40
Early stopping patience: 7

Experiment A: weighted_ce
  Checkpoint: /Users/vs/Sem 5/Respiratory-Disease-Classification/models/deep_learning/imbalance_comparison/experiment_A_weighted_ce_best.pt
  Checkpoint already exists: False
  History: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/imbalance_comparison/experiment_A_weighted_ce_history.csv

Experiment B: balanced_sampler
  Checkpoint: /Users/vs/Sem 5/Respiratory-Disease-Classification/models/deep_learning/imbalance_comparison/experiment_B_balanced_sampler_best.pt
  Checkpoint already exists: False
  History: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/imbalance_comparison/experiment_B_balanced_sampler_history.csv

Configuration ready. No training has started.


In [36]:

import time
import torch

# Reproducible benchmark initialization.
torch.manual_seed(SEED)
np.random.seed(SEED)

benchmark_model = MultiFeatureCNN(num_classes=NUM_CLASSES).to(device)
benchmark_optimizer = torch.optim.Adam(
    benchmark_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Warm-up: fetch a batch, including its audio feature extraction.
start = time.perf_counter()
benchmark_batch = next(iter(train_loader_A))
data_seconds = time.perf_counter() - start

logmel_b, mfcc_b, chroma_b, labels_b = benchmark_batch

# Warm-up forward/backward, then time one optimizer update.
benchmark_model.train()
logmel_b = logmel_b.to(device)
mfcc_b = mfcc_b.to(device)
chroma_b = chroma_b.to(device)
labels_b = labels_b.to(device)

benchmark_optimizer.zero_grad(set_to_none=True)
warmup_logits = benchmark_model(logmel_b, mfcc_b, chroma_b)
warmup_loss = criterion_weighted(warmup_logits, labels_b)
warmup_loss.backward()
benchmark_optimizer.step()

if device.type == "mps":
    torch.mps.synchronize()

start = time.perf_counter()

benchmark_optimizer.zero_grad(set_to_none=True)
logits = benchmark_model(logmel_b, mfcc_b, chroma_b)
loss = criterion_weighted(logits, labels_b)
loss.backward()
benchmark_optimizer.step()

if device.type == "mps":
    torch.mps.synchronize()

update_seconds = time.perf_counter() - start

print(f"Batch data loading + feature extraction: {data_seconds:.2f} seconds")
print(f"One optimizer update: {update_seconds:.2f} seconds")
print(f"Estimated training batches per epoch: {len(train_loader_A)}")
print(
    "Rough compute-only time per epoch:",
    f"{len(train_loader_A) * update_seconds / 60:.1f} minutes"
)
print("Benchmark complete; no checkpoint was saved.")


Batch data loading + feature extraction: 0.15 seconds
One optimizer update: 0.04 seconds
Estimated training batches per epoch: 139
Rough compute-only time per epoch: 0.1 minutes
Benchmark complete; no checkpoint was saved.


In [37]:

import copy
import time
import random
import pandas as pd
import numpy as np
import torch

def set_experiment_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_experiment_A():
    exp = EXPERIMENTS["A"]

    # Avoid silently overwriting a previous run.
    if exp["checkpoint"].exists():
        raise FileExistsError(
            f"Checkpoint already exists: {exp['checkpoint']}. "
            "Choose a new output path before rerunning."
        )

    set_experiment_seed(SEED)

    model_A = MultiFeatureCNN(num_classes=NUM_CLASSES).to(device)
    optimizer_A = torch.optim.Adam(
        model_A.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )
    scheduler_A = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_A, mode="max", factor=0.5, patience=2
    )

    best_macro_f1 = -1.0
    best_epoch = 0
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.perf_counter()

        train_metrics = run_training_epoch(
            model_A, train_loader_A, optimizer_A,
            criterion_weighted, device
        )
        val_metrics, val_report, y_val, pred_val = evaluate_model(
            model_A, val_loader, device
        )

        selection_score = val_metrics["macro_f1_supported_classes"]
        scheduler_A.step(selection_score)

        row = {
            "epoch": epoch,
            "learning_rate": optimizer_A.param_groups[0]["lr"],
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_balanced_accuracy": train_metrics["balanced_accuracy"],
            "train_macro_f1_all_classes": train_metrics["macro_f1"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy_supported": val_metrics["balanced_accuracy_supported"],
            "val_macro_f1_supported_classes": selection_score,
            "epoch_seconds": time.perf_counter() - epoch_start
        }
        history.append(row)

        # Persist history after each epoch.
        pd.DataFrame(history).to_csv(exp["history"], index=False)

        improved = selection_score > best_macro_f1
        if improved:
            best_macro_f1 = selection_score
            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save({
                "model_state_dict": model_A.state_dict(),
                "num_classes": NUM_CLASSES,
                "class_names": class_names,
                "epoch": epoch,
                "validation_macro_f1_supported_classes": selection_score,
                "validation_metrics": val_metrics,
                "seed": SEED,
                "experiment": exp["name"],
                "architecture": "MultiFeatureCNN",
                "total_parameters": sum(
                    p.numel() for p in model_A.parameters()
                )
            }, exp["checkpoint"])

            val_report.to_csv(exp["validation_report"], index=False)
            with open(exp["validation_metrics"], "w") as f:
                import json
                json.dump({
                    **val_metrics,
                    "best_epoch": best_epoch,
                    "selection_metric": "macro_f1_supported_classes",
                    "validation_supported_classes": [
                        class_names[i] for i in sorted(np.unique(y_val))
                    ]
                }, f, indent=2)
        else:
            epochs_without_improvement += 1

        print(
            f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
            f"train_loss={train_metrics['loss']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_macro_f1_supported={selection_score:.4f} | "
            f"val_bal_acc={val_metrics['balanced_accuracy_supported']:.4f} | "
            f"{row['epoch_seconds']:.1f}s"
            + (" | BEST" if improved else "")
        )

        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping after epoch {epoch}.")
            break

    print("\nExperiment A finished.")
    print("Best epoch:", best_epoch)
    print("Best supported-class validation macro-F1:", round(best_macro_f1, 4))
    print("Checkpoint:", exp["checkpoint"])
    print("History:", exp["history"])

    return model_A, pd.DataFrame(history)


model_A, history_A = train_experiment_A()


Epoch 01/40 | train_loss=1.8156 | val_loss=1.0548 | val_macro_f1_supported=0.1776 | val_bal_acc=0.2241 | 28.8s | BEST
Epoch 02/40 | train_loss=1.5442 | val_loss=1.5438 | val_macro_f1_supported=0.1644 | val_bal_acc=0.2203 | 23.1s
Epoch 03/40 | train_loss=1.4473 | val_loss=0.9695 | val_macro_f1_supported=0.3173 | val_bal_acc=0.3172 | 23.1s | BEST
Epoch 04/40 | train_loss=1.3650 | val_loss=1.1196 | val_macro_f1_supported=0.1953 | val_bal_acc=0.2317 | 23.2s
Epoch 05/40 | train_loss=1.2965 | val_loss=1.8490 | val_macro_f1_supported=0.2458 | val_bal_acc=0.2349 | 24.2s
Epoch 06/40 | train_loss=1.2973 | val_loss=1.5432 | val_macro_f1_supported=0.2032 | val_bal_acc=0.3148 | 23.2s
Epoch 07/40 | train_loss=1.2089 | val_loss=1.3583 | val_macro_f1_supported=0.2904 | val_bal_acc=0.2851 | 24.2s
Epoch 08/40 | train_loss=1.2047 | val_loss=1.3000 | val_macro_f1_supported=0.3001 | val_bal_acc=0.3483 | 23.0s
Epoch 09/40 | train_loss=1.1664 | val_loss=2.0656 | val_macro_f1_supported=0.2628 | val_bal_acc=0.

In [38]:

def train_experiment_B():
    exp = EXPERIMENTS["B"]

    if exp["checkpoint"].exists():
        raise FileExistsError(
            f"Checkpoint already exists: {exp['checkpoint']}. "
            "Choose a new output path before rerunning."
        )

    set_experiment_seed(SEED)

    model_B = MultiFeatureCNN(num_classes=NUM_CLASSES).to(device)
    optimizer_B = torch.optim.Adam(
        model_B.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )
    scheduler_B = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_B, mode="max", factor=0.5, patience=2
    )

    best_macro_f1 = -1.0
    best_epoch = 0
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.perf_counter()

        train_metrics = run_training_epoch(
            model_B, train_loader_B, optimizer_B,
            criterion_unweighted, device
        )
        val_metrics, val_report, y_val, pred_val = evaluate_model(
            model_B, val_loader, device
        )

        selection_score = val_metrics["macro_f1_supported_classes"]
        scheduler_B.step(selection_score)

        row = {
            "epoch": epoch,
            "learning_rate": optimizer_B.param_groups[0]["lr"],
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_balanced_accuracy": train_metrics["balanced_accuracy"],
            "train_macro_f1_all_classes": train_metrics["macro_f1"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy_supported": val_metrics["balanced_accuracy_supported"],
            "val_macro_f1_supported_classes": selection_score,
            "epoch_seconds": time.perf_counter() - epoch_start
        }
        history.append(row)

        pd.DataFrame(history).to_csv(exp["history"], index=False)

        improved = selection_score > best_macro_f1
        if improved:
            best_macro_f1 = selection_score
            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save({
                "model_state_dict": model_B.state_dict(),
                "num_classes": NUM_CLASSES,
                "class_names": class_names,
                "epoch": epoch,
                "validation_macro_f1_supported_classes": selection_score,
                "validation_metrics": val_metrics,
                "seed": SEED,
                "experiment": exp["name"],
                "architecture": "MultiFeatureCNN",
                "total_parameters": sum(
                    p.numel() for p in model_B.parameters()
                )
            }, exp["checkpoint"])

            val_report.to_csv(exp["validation_report"], index=False)
            with open(exp["validation_metrics"], "w") as f:
                import json
                json.dump({
                    **val_metrics,
                    "best_epoch": best_epoch,
                    "selection_metric": "macro_f1_supported_classes",
                    "validation_supported_classes": [
                        class_names[i] for i in sorted(np.unique(y_val))
                    ]
                }, f, indent=2)
        else:
            epochs_without_improvement += 1

        print(
            f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
            f"train_loss={train_metrics['loss']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_macro_f1_supported={selection_score:.4f} | "
            f"val_bal_acc={val_metrics['balanced_accuracy_supported']:.4f} | "
            f"{row['epoch_seconds']:.1f}s"
            + (" | BEST" if improved else "")
        )

        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping after epoch {epoch}.")
            break

    print("\nExperiment B finished.")
    print("Best epoch:", best_epoch)
    print("Best supported-class validation macro-F1:", round(best_macro_f1, 4))
    print("Checkpoint:", exp["checkpoint"])
    print("History:", exp["history"])

    return model_B, pd.DataFrame(history)


model_B, history_B = train_experiment_B()


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Epoch 01/40 | train_loss=1.5033 | val_loss=1.7316 | val_macro_f1_supported=0.2352 | val_bal_acc=0.2289 | 24.5s | BEST


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Epoch 02/40 | train_loss=1.1175 | val_loss=3.4286 | val_macro_f1_supported=0.2351 | val_bal_acc=0.2287 | 22.7s
Epoch 03/40 | train_loss=1.0030 | val_loss=1.8654 | val_macro_f1_supported=0.3248 | val_bal_acc=0.3422 | 23.6s | BEST
Epoch 04/40 | train_loss=0.9652 | val_loss=2.1992 | val_macro_f1_supported=0.2488 | val_bal_acc=0.3286 | 22.1s
Epoch 05/40 | train_loss=0.9011 | val_loss=2.7186 | val_macro_f1_supported=0.2436 | val_bal_acc=0.2339 | 22.5s
Epoch 06/40 | train_loss=0.8511 | val_loss=3.3433 | val_macro_f1_supported=0.2066 | val_bal_acc=0.2277 | 22.7s
Epoch 07/40 | train_loss=0.7614 | val_loss=2.2873 | val_macro_f1_supported=0.2276 | val_bal_acc=0.2268 | 22.6s


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Epoch 08/40 | train_loss=0.7293 | val_loss=3.2582 | val_macro_f1_supported=0.2147 | val_bal_acc=0.2261 | 21.8s


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Epoch 09/40 | train_loss=0.7204 | val_loss=3.6244 | val_macro_f1_supported=0.2103 | val_bal_acc=0.2234 | 23.6s
Epoch 10/40 | train_loss=0.6585 | val_loss=4.0491 | val_macro_f1_supported=0.1742 | val_bal_acc=0.1863 | 23.0s
Early stopping after epoch 10.

Experiment B finished.
Best epoch: 3
Best supported-class validation macro-F1: 0.3248
Checkpoint: /Users/vs/Sem 5/Respiratory-Disease-Classification/models/deep_learning/imbalance_comparison/experiment_B_balanced_sampler_best.pt
History: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/imbalance_comparison/experiment_B_balanced_sampler_history.csv


In [39]:

import json
import pandas as pd
import torch

comparison_rows = []
validation_reports = {}

for key in ["A", "B"]:
    exp = EXPERIMENTS[key]

    with open(exp["validation_metrics"], "r") as f:
        metrics = json.load(f)

    report = pd.read_csv(exp["validation_report"])
    validation_reports[key] = report

    checkpoint = torch.load(
        exp["checkpoint"],
        map_location="cpu",
        weights_only=False
    )

    comparison_rows.append({
        "experiment": key,
        "strategy": exp["name"],
        "best_epoch": checkpoint["epoch"],
        "validation_macro_f1": metrics["macro_f1_supported_classes"],
        "validation_balanced_accuracy": metrics["balanced_accuracy_supported"],
        "validation_accuracy": metrics["accuracy"],
        "validation_loss": metrics["loss"],
        "checkpoint_exists": exp["checkpoint"].exists(),
        "parameters": checkpoint["total_parameters"]
    })

print("EXPERIMENT COMPARISON")
display(pd.DataFrame(comparison_rows))

for key in ["A", "B"]:
    print(f"\nEXPERIMENT {key} — PER-CLASS VALIDATION RESULTS")
    display(validation_reports[key])

print("\nCheckpoint inspection complete.")
print("No test-set evaluation performed.")


EXPERIMENT COMPARISON


,experiment,strategy,best_epoch,validation_macro_f1,validation_balanced_accuracy,validation_accuracy,validation_loss,checkpoint_exists,parameters
0,A,weighted_ce,3,0.317256,0.317224,0.722101,0.969529,True,108776
1,B,balanced_sampler,3,0.324783,0.342201,0.728665,1.865421,True,108776



EXPERIMENT A — PER-CLASS VALIDATION RESULTS


,class_id,class_name,support,precision,recall,f1
0,0,Asthma,0,NaN,NaN,NaN
1,1,Bronchiectasis,8,0.428571,0.375000,0.400000
2,2,Bronchiolitis,27,0.363636,0.148148,0.210526
3,3,COPD,696,0.802296,0.903736,0.850000
4,4,Healthy,44,0.345455,0.431818,0.383838
5,5,LRTI,0,NaN,NaN,NaN
6,6,Pneumonia,112,0.087719,0.044643,0.059172
7,7,URTI,27,0.000000,0.000000,0.000000



EXPERIMENT B — PER-CLASS VALIDATION RESULTS


,class_id,class_name,support,precision,recall,f1
0,0,Asthma,0,NaN,NaN,NaN
1,1,Bronchiectasis,8,1.000000,0.375000,0.545455
2,2,Bronchiolitis,27,0.258621,0.555556,0.352941
3,3,COPD,696,0.791822,0.918103,0.850299
4,4,Healthy,44,0.195652,0.204545,0.200000
5,5,LRTI,0,NaN,NaN,NaN
6,6,Pneumonia,112,0.000000,0.000000,0.000000
7,7,URTI,27,0.000000,0.000000,0.000000



Checkpoint inspection complete.
No test-set evaluation performed.


In [40]:

from collections import Counter
import numpy as np
import pandas as pd
import torch

def inspect_validation_predictions(experiment_key):
    exp = EXPERIMENTS[experiment_key]
    checkpoint = torch.load(
        exp["checkpoint"],
        map_location=device,
        weights_only=False
    )

    inspection_model = MultiFeatureCNN(num_classes=NUM_CLASSES).to(device)
    inspection_model.load_state_dict(checkpoint["model_state_dict"])
    inspection_model.eval()

    y_true_all = []
    y_pred_all = []

    with torch.no_grad():
        for logmel, mfcc, chroma, labels in val_loader:
            logits = inspection_model(
                logmel.to(device),
                mfcc.to(device),
                chroma.to(device)
            )
            predictions = logits.argmax(dim=1).cpu().numpy()
            y_pred_all.extend(predictions)
            y_true_all.extend(labels.numpy())

    y_true_all = np.asarray(y_true_all)
    y_pred_all = np.asarray(y_pred_all)

    rows = []
    for class_id, class_name in enumerate(class_names):
        rows.append({
            "class_name": class_name,
            "true_validation_count": int((y_true_all == class_id).sum()),
            "predicted_count": int((y_pred_all == class_id).sum()),
            "correct_count": int(
                ((y_true_all == class_id) & (y_pred_all == class_id)).sum()
            )
        })

    return pd.DataFrame(rows)


for key in ["A", "B"]:
    print(f"\nEXPERIMENT {key}: VALIDATION PREDICTION DISTRIBUTION")
    display(inspect_validation_predictions(key))

print("\nPrediction-distribution inspection complete.")
print("Test set was not used.")



EXPERIMENT A: VALIDATION PREDICTION DISTRIBUTION


,class_name,true_validation_count,predicted_count,correct_count
0,Asthma,0,0,0
1,Bronchiectasis,8,7,3
2,Bronchiolitis,27,11,4
3,COPD,696,784,629
4,Healthy,44,55,19
5,LRTI,0,0,0
6,Pneumonia,112,57,5
7,URTI,27,0,0



EXPERIMENT B: VALIDATION PREDICTION DISTRIBUTION


,class_name,true_validation_count,predicted_count,correct_count
0,Asthma,0,0,0
1,Bronchiectasis,8,3,3
2,Bronchiolitis,27,58,15
3,COPD,696,807,639
4,Healthy,44,46,9
5,LRTI,0,0,0
6,Pneumonia,112,0,0
7,URTI,27,0,0



Prediction-distribution inspection complete.
Test set was not used.


In [41]:

import pandas as pd
from pathlib import Path

manifest_paths = {
    "train": DATA_DIR / "train_cnn_manifest.csv",
    "validation": DATA_DIR / "validation_cnn_manifest.csv",
    "test": DATA_DIR / "test_cnn_manifest.csv"
}

split_frames = {}
for split_name, path in manifest_paths.items():
    df = pd.read_csv(path)
    df["audit_split"] = split_name
    split_frames[split_name] = df

all_splits = pd.concat(split_frames.values(), ignore_index=True)

# Disease support by split.
support_table = (
    all_splits
    .groupby(["audit_split", "diagnosis"])
    .agg(
        cycles=("audio_path", "size"),
        patients=("patient_id", "nunique"),
        unique_audio_paths=("audio_path", "nunique")
    )
    .reset_index()
)

print("DISEASE SUPPORT BY SPLIT")
display(support_table.pivot(
    index="diagnosis",
    columns="audit_split",
    values="cycles"
).fillna(0).astype(int))

print("\nUNIQUE PATIENTS BY DISEASE AND SPLIT")
display(support_table.pivot(
    index="diagnosis",
    columns="audit_split",
    values="patients"
).fillna(0).astype(int))

# Check whether any patient appears in more than one split.
patient_sets = {
    name: set(df["patient_id"].dropna().astype(str))
    for name, df in split_frames.items()
}

print("\nPATIENT OVERLAP")
for first, second in [("train", "validation"), ("train", "test"),
                      ("validation", "test")]:
    overlap = patient_sets[first] & patient_sets[second]
    print(f"{first} vs {second}: {len(overlap)} overlapping patients")

# Compare official split membership, if the column exists.
if "official_split" in all_splits.columns:
    print("\nOFFICIAL SPLIT MEMBERSHIP WITHIN OUR CURRENT SPLITS")
    display(pd.crosstab(
        all_splits["audit_split"],
        all_splits["official_split"],
        margins=True
    ))

print("\nManifest audit complete. No files were modified.")


DISEASE SUPPORT BY SPLIT


audit_split,test,train,validation
diagnosis,,,
Asthma,0,6,0
Bronchiectasis,20,76,8
Bronchiolitis,19,114,27
COPD,1337,3713,696
Healthy,84,194,44
LRTI,0,32,0
Pneumonia,52,121,112
URTI,49,167,27



UNIQUE PATIENTS BY DISEASE AND SPLIT


audit_split,test,train,validation
diagnosis,,,
Asthma,0,1,0
Bronchiectasis,2,4,1
Bronchiolitis,1,4,1
COPD,13,41,10
Healthy,5,17,4
LRTI,0,2,0
Pneumonia,1,4,1
URTI,3,9,2



PATIENT OVERLAP
train vs validation: 0 overlapping patients
train vs test: 0 overlapping patients
validation vs test: 0 overlapping patients

OFFICIAL SPLIT MEMBERSHIP WITHIN OUR CURRENT SPLITS


official_split,test,train,All
audit_split,,,
test,480,1081,1561
train,1943,2480,4423
validation,333,581,914
All,2756,4142,6898



Manifest audit complete. No files were modified.


In [42]:

from pathlib import Path

ROOT = Path.cwd()

search_roots = [
    ROOT / "Notebooks",
    ROOT / "Notebooks 2",
    ROOT / "src",
    ROOT / "scripts"
]

terms = [
    "train_cnn_manifest.csv",
    "validation_cnn_manifest.csv",
    "official_split",
    "GroupShuffleSplit",
    "GroupKFold",
    "StratifiedGroupKFold"
]

matches = []

for search_root in search_roots:
    if not search_root.exists():
        continue

    for path in search_root.rglob("*"):
        if path.suffix not in {".py", ".ipynb"}:
            continue

        try:
            content = path.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue

        found = [term for term in terms if term in content]
        if found:
            matches.append({
                "file": str(path.relative_to(ROOT)),
                "terms_found": ", ".join(found),
                "lines": len(content.splitlines())
            })

print("FILES CONTAINING SPLIT-GENERATION REFERENCES")
if matches:
    display(pd.DataFrame(matches))
else:
    print("No matches found in the searched folders.")

print("\nSearch complete. No files were modified.")


FILES CONTAINING SPLIT-GENERATION REFERENCES


,file,terms_found,lines
0,Notebooks/11_cnn_baseline.ipynb,"train_cnn_manifest.csv, validation_cnn_manifes...",6137
1,Notebooks/09_classical_ml.ipynb,official_split,1661
2,Notebooks/13_cnn_lstm_optional.ipynb,"train_cnn_manifest.csv, validation_cnn_manifes...",2419
3,Notebooks/10_deep_learning_data_preparation.ipynb,official_split,2975
4,Notebooks/12_multifeature_cnn.ipynb,"train_cnn_manifest.csv, validation_cnn_manifes...",5928
5,Notebooks/08_pca_and_clustering.ipynb,official_split,2099
6,Notebooks/02_data_cleaning.ipynb,official_split,3276
7,Notebooks/07_feature_selection.ipynb,official_split,5056
8,Notebooks/16_final_model_comparison.ipynb,"train_cnn_manifest.csv, validation_cnn_manifes...",2819
9,Notebooks/01_dataset_understanding.ipynb,official_split,7633



Search complete. No files were modified.


In [43]:

import json
from pathlib import Path

candidate_notebooks = [
    ROOT / "Notebooks 2/02_Leakage_Safe_Preprocessing.ipynb",
    ROOT / "Notebooks/11_cnn_baseline.ipynb",
    ROOT / "Notebooks 2/04_ICBHI_Disease_Transfer_Learning.ipynb"
]

keywords = [
    "train_cnn_manifest",
    "validation_cnn_manifest",
    "test_cnn_manifest",
    "official_split",
    "GroupShuffleSplit",
    "GroupKFold",
    "StratifiedGroupKFold",
    "to_csv("
]

for notebook_path in candidate_notebooks:
    print("\n" + "=" * 90)
    print("NOTEBOOK:", notebook_path.relative_to(ROOT))

    if not notebook_path.exists():
        print("Notebook not found.")
        continue

    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    found_count = 0

    for cell_index, cell in enumerate(notebook.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        matched = [word for word in keywords if word in source]

        if matched:
            found_count += 1
            print(f"\n--- CELL {cell_index} | matches: {', '.join(matched)} ---")
            print(source[:7000])
            if len(source) > 7000:
                print("\n[Cell truncated for display]")

    print(f"\nRelevant code cells found: {found_count}")

print("\nInspection complete. No files were modified.")



NOTEBOOK: Notebooks 2/02_Leakage_Safe_Preprocessing.ipynb

--- CELL 3 | matches: GroupShuffleSplit, GroupKFold, StratifiedGroupKFold ---

import nbformat

notebooks_to_audit = [
    "07_feature_selection.ipynb",
    "09_classical_ml.ipynb",
    "10_deep_learning_data_preparation.ipynb",
    "11_cnn_baseline.ipynb",
    "12_multifeature_cnn.ipynb",
    "14_patient_level_prediction.ipynb",
]

keywords = [
    "fit_transform",
    ".fit(",
    "StandardScaler",
    "MinMaxScaler",
    "SelectKBest",
    "feature_selection",
    "train_test_split",
    "GroupShuffleSplit",
    "GroupKFold",
    "StratifiedGroupKFold",
    "class_weight",
    "WeightedRandomSampler",
    "normalization",
    "mean(",
    "std(",
    "SMOTE",
    "resample",
]

for notebook_name in notebooks_to_audit:
    path = PROJECT_ROOT / "notebooks" / notebook_name

    print("\n" + "=" * 80)
    print("NOTEBOOK:", notebook_name)

    if not path.exists():
        print("Notebook not found at:", path)
        continue

In [44]:

import json
from pathlib import Path

notebooks_to_inspect = [
    ROOT / "Notebooks/11_cnn_baseline.ipynb",
    ROOT / "Notebooks/10_deep_learning_data_preparation.ipynb",
    ROOT / "Notebooks 2/02_Leakage_Safe_Preprocessing.ipynb",
]

search_terms = [
    "cnn_split",
    "dl_manifest =",
    "dl_manifest[",
    "GroupShuffleSplit",
    "StratifiedGroupKFold",
    "GroupKFold",
    "train_test_split",
    "official_split",
    "patient_id"
]

for notebook_path in notebooks_to_inspect:
    print("\n" + "=" * 90)
    print("NOTEBOOK:", notebook_path.relative_to(ROOT))

    if not notebook_path.exists():
        print("Not found.")
        continue

    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    for i, cell in enumerate(notebook.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        matched = [
            term for term in search_terms
            if term.lower() in source.lower()
        ]

        # Focus on code that may create or assign the split.
        if any(term in matched for term in [
            "cnn_split", "dl_manifest =", "GroupShuffleSplit",
            "StratifiedGroupKFold", "GroupKFold", "train_test_split"
        ]):
            print(f"\n--- CELL {i} | matched: {matched} ---")
            print(source[:6000])
            if len(source) > 6000:
                print("[Cell truncated]")

print("\nRead-only inspection complete. No files were modified.")



NOTEBOOK: Notebooks/11_cnn_baseline.ipynb

--- CELL 1 | matched: ['train_test_split'] ---
from pathlib import Path
import random
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

print("PyTorch version:", torch.__version__)

--- CELL 3 | matched: ['dl_manifest =', 'dl_manifest[', 'patient_id'] ---
manifest_path = DL_DIR / "deep_learning_manifest.csv"

dl_manifest = pd.read_csv(manifest_path)

print("Manifest shape:", dl_manifest.shape)
print("Unique patients:", dl_manifest["patient_id"].nunique())
print("Unique cycles:", len(dl_manifest))

display(dl_manifest.head())

--- CELL 9 | matched: ['train_test_split'] ---
train_val_regular, test_regular = train_test_split(
    regular_pati

In [45]:

import json
from pathlib import Path

notebook_path = ROOT / "Notebooks/11_cnn_baseline.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

for cell_index in [8, 9, 10, 11, 12, 13]:
    cell = notebook["cells"][cell_index]
    if cell.get("cell_type") == "code":
        print("\n" + "=" * 80)
        print(f"CELL {cell_index}")
        print("".join(cell.get("source", [])))

print("\nRead-only inspection complete.")



CELL 8
rare_patients = patient_df[
    patient_df["diagnosis"].isin(rare_classes)
].copy()

regular_patients = patient_df[
    ~patient_df["diagnosis"].isin(rare_classes)
].copy()

print("Rare-class patients:", len(rare_patients))
print("Regular patients:", len(regular_patients))

display(rare_patients)

CELL 9
train_val_regular, test_regular = train_test_split(
    regular_patients,
    test_size=0.20,
    random_state=SEED,
    stratify=regular_patients["diagnosis"]
)

train_regular, val_regular = train_test_split(
    train_val_regular,
    test_size=0.1875,
    random_state=SEED,
    stratify=train_val_regular["diagnosis"]
)

print("Regular train:", len(train_regular))
print("Regular validation:", len(val_regular))
print("Regular test:", len(test_regular))

CELL 10
train_patients = pd.concat(
    [
        train_regular,
        rare_patients
    ],
    ignore_index=True
)

val_patients = val_regular.copy()
test_patients = test_regular.copy()

print("Final patient counts:")
print(

In [48]:

from pathlib import Path
import pandas as pd

# Find the project root from the current working directory.
cwd = Path.cwd()
candidates = [cwd, *cwd.parents]

PROJECT_ROOT = next(
    (
        p for p in candidates
        if (p / "data/processed/deep_learning/cnn_baseline").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Please run print(Path.cwd()) and share the output."
    )

manifest_path = (
    PROJECT_ROOT
    / "data/processed/deep_learning/cnn_baseline"
    / "train_cnn_manifest.csv"
)

if not manifest_path.exists():
    raise FileNotFoundError(f"Manifest not found: {manifest_path}")

# The saved split manifests can be audited without dl_manifest.
split_files = {
    "train": "train_cnn_manifest.csv",
    "validation": "validation_cnn_manifest.csv",
    "test": "test_cnn_manifest.csv",
}

frames = {}
for split, filename in split_files.items():
    path = manifest_path.parent / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing manifest: {path}")
    frames[split] = pd.read_csv(path)

all_data = pd.concat(
    [df.assign(cnn_split=split) for split, df in frames.items()],
    ignore_index=True
)

required = {"patient_id", "diagnosis", "official_split"}
missing = required - set(all_data.columns)

if missing:
    print("Missing columns:", sorted(missing))
    print("Available columns:", all_data.columns.tolist())
else:
    print("PATIENT COUNTS BY DIAGNOSIS AND OFFICIAL SPLIT")
    patient_level = all_data[
        ["patient_id", "diagnosis", "official_split"]
    ].drop_duplicates()

    display(
        patient_level.groupby(
            ["diagnosis", "official_split"]
        )["patient_id"]
        .nunique()
        .unstack(fill_value=0)
        .sort_index()
    )

    print("\nPATIENT OVERLAP BETWEEN CNN SPLITS")
    for a, b in [
        ("train", "validation"),
        ("train", "test"),
        ("validation", "test"),
    ]:
        pa = set(frames[a]["patient_id"].dropna())
        pb = set(frames[b]["patient_id"].dropna())
        print(f"{a} vs {b}: {len(pa & pb)} overlapping patients")

    print("\nCYCLE COUNTS BY DIAGNOSIS AND OFFICIAL SPLIT")
    display(pd.crosstab(
        all_data["diagnosis"],
        all_data["official_split"],
        margins=True
    ))

    print("\nRead-only audit complete.")


PATIENT COUNTS BY DIAGNOSIS AND OFFICIAL SPLIT


official_split,test,train
diagnosis,,
Asthma,0,1
Bronchiectasis,1,6
Bronchiolitis,3,3
COPD,27,39
Healthy,14,12
LRTI,0,2
Pneumonia,0,6
URTI,4,10



PATIENT OVERLAP BETWEEN CNN SPLITS
train vs validation: 0 overlapping patients
train vs test: 0 overlapping patients
validation vs test: 0 overlapping patients

CYCLE COUNTS BY DIAGNOSIS AND OFFICIAL SPLIT


official_split,test,train,All
diagnosis,,,
Asthma,0,6,6
Bronchiectasis,14,90,104
Bronchiolitis,86,74,160
COPD,2409,3337,5746
Healthy,175,147,322
LRTI,0,32,32
Pneumonia,0,285,285
URTI,72,171,243
All,2756,4142,6898



Read-only audit complete.


In [49]:

import json
from pathlib import Path

notebook_path = PROJECT_ROOT / "Notebooks/10_deep_learning_data_preparation.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    if any(term in source.lower() for term in [
        "official_split",
        "features_with_split",
        "deep_learning_manifest.csv",
    ]):
        print("\n" + "=" * 80)
        print(f"CELL {i}")
        print(source)

print("\nRead-only inspection complete.")



CELL 4
features_with_split = processed_cycles_df.merge(
    recordings_df[
        [
            "patient_id",
            "recording_id",
            "chest_location",
            "official_split"
        ]
    ],
    on=[
        "patient_id",
        "recording_id",
        "chest_location"
    ],
    how="left",
    validate="many_to_one"
)

print("Merged dataset shape:", features_with_split.shape)

print("\nMissing official split values:")
print(
    features_with_split["official_split"].isna().sum()
)

CELL 5
print("Official split distribution:")
print(
    features_with_split["official_split"]
    .value_counts()
)

print("\nSplit percentages:")
print(
    features_with_split["official_split"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

CELL 6
train_dl_df = features_with_split[
    features_with_split["official_split"] == "train"
].copy()

test_dl_df = features_with_split[
    features_with_split["official_split"] == "test"
].copy()

print("Training cycles:

In [50]:

import json
from pathlib import Path

notebook_path = PROJECT_ROOT / "Notebooks/10_deep_learning_data_preparation.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

for i in range(min(4, len(nb["cells"]))):
    cell = nb["cells"][i]
    print("\n" + "=" * 80)
    print(f"CELL {i} | {cell.get('cell_type')}")

    source = "".join(cell.get("source", []))
    print(source if source.strip() else "[Empty cell]")

print("\nRead-only inspection complete.")



CELL 0 | markdown
# 10 — Deep-Learning Data Preparation

Generates Mel, MFCC, and chroma channels from the saved Notebook 04 4 kHz cycle WAVs, without repeating cycle extraction/resampling. It consumes the patient split created in Notebook 06. Features are cached in a compressed archive; augmentation is intentionally deferred to training loaders and will never touch validation/test cycles.

CELL 1 | code
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder

print("Imports completed successfully.")

CELL 2 | code
PROJECT_ROOT = Path(
    "/Users/vs/Sem 5/Respiratory-Disease-Classification"
)

PROCESSED_CYCLES_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "processed_cycles.csv"
)

RECORDINGS_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "recordings.csv"
)

DL_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "deep_learning"
)

DL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok

In [51]:

import json
from pathlib import Path

notebook_path = PROJECT_ROOT / "Notebooks/06_patient_split.ipynb"

if not notebook_path.exists():
    print("Exact path not found:", notebook_path)
    print("\nCandidate notebooks:")
    for p in sorted(PROJECT_ROOT.rglob("*.ipynb")):
        if "06" in p.name.lower() and "patient" in p.name.lower():
            print(p.relative_to(PROJECT_ROOT))
else:
    with open(notebook_path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    for i, cell in enumerate(nb["cells"]):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        if any(term in source.lower() for term in [
            "official_split",
            "recordings_df",
            "train_test_split",
            "patient_id",
            "split",
        ]):
            print("\n" + "=" * 80)
            print(f"CELL {i}")
            print(source)

print("\nRead-only inspection complete.")


Exact path not found: /Users/vs/Sem 5/Respiratory-Disease-Classification/Notebooks/06_patient_split.ipynb

Candidate notebooks:

Read-only inspection complete.


In [52]:

from pathlib import Path

print("Project root:", PROJECT_ROOT)

print("\nAll notebooks with 'split' in their filename:")
for p in sorted(PROJECT_ROOT.rglob("*.ipynb")):
    if "split" in p.name.lower():
        print(p.relative_to(PROJECT_ROOT))

print("\nAll notebooks with '06' in their filename:")
for p in sorted(PROJECT_ROOT.rglob("*.ipynb")):
    if "06" in p.name.lower():
        print(p.relative_to(PROJECT_ROOT))

print("\nCandidate source files:")
for folder_name in ["Notebooks", "Notebooks 2", "scripts"]:
    folder = PROJECT_ROOT / folder_name
    if folder.exists():
        for p in sorted(folder.iterdir()):
            if p.is_file() and any(
                word in p.name.lower()
                for word in ["patient", "split", "manifest"]
            ):
                print(p.relative_to(PROJECT_ROOT))


Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification

All notebooks with 'split' in their filename:

All notebooks with '06' in their filename:
notebooks/.ipynb_checkpoints/06_feature_analysis-checkpoint.ipynb
notebooks/06_feature_analysis.ipynb

Candidate source files:
Notebooks/14_patient_level_prediction.ipynb
scripts/evaluate_common_patient_split.py


In [53]:

from pathlib import Path

script_path = PROJECT_ROOT / "scripts" / "evaluate_common_patient_split.py"

if not script_path.exists():
    raise FileNotFoundError(f"Script not found: {script_path}")

print("SCRIPT:", script_path.relative_to(PROJECT_ROOT))
print("=" * 80)

with open(script_path, "r", encoding="utf-8") as f:
    script_text = f.read()

print(script_text)
print("\nRead-only inspection complete.")


SCRIPT: scripts/evaluate_common_patient_split.py
"""Retrain classical baselines and compare all models on the CNN patient holdout.

The script does not overwrite app artifacts or legacy experiment reports. It uses
the CNN patient-disjoint manifests, fits feature selection/scaling/models on the
training patients only, and checks row alignment for saved neural predictions.
"""

from __future__ import annotations

import json
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC

ROOT = Path(__fi

In [54]:

from pathlib import Path
import json
import pandas as pd

root = PROJECT_ROOT
audit_dir = root / "reports/research_audit/common_patient_split"

print("EXISTING AUDIT OUTPUTS")
if audit_dir.exists():
    for p in sorted(audit_dir.iterdir()):
        if p.is_file():
            print(f"{p.name}: {p.stat().st_size:,} bytes")
else:
    print("Audit output directory does not exist yet.")

print("\nEXPERIMENT A/B CHECKPOINTS")
for name in [
    "experiment_A_weighted_ce_best.pt",
    "experiment_B_balanced_sampler_best.pt",
]:
    p = root / "models/deep_learning/imbalance_comparison" / name
    print(f"{name}: {'EXISTS' if p.exists() else 'MISSING'}")

print("\nCLASSICAL FEATURE TABLE")
features_path = root / "data/processed/features/respiratory_cycle_features.csv"
print("Exists:", features_path.exists())
if features_path.exists():
    print("Size:", f"{features_path.stat().st_size:,} bytes")

protocol_path = audit_dir / "protocol.json"
if protocol_path.exists():
    print("\nEXISTING PROTOCOL")
    with open(protocol_path, "r", encoding="utf-8") as f:
        protocol = json.load(f)
    print(json.dumps(protocol, indent=2))
else:
    print("\nNo existing protocol.json found.")


EXISTING AUDIT OUTPUTS
classical_selected_features.json: 607 bytes
confusion_matrix_cnn_lstm.csv: 291 bytes
confusion_matrix_lightweight_mel_cnn.csv: 295 bytes
confusion_matrix_logistic_regression.csv: 294 bytes
confusion_matrix_multifeature_cnn.csv: 287 bytes
confusion_matrix_random_forest.csv: 294 bytes
confusion_matrix_regularized_multifeature_cnn.csv: 287 bytes
confusion_matrix_svm.csv: 295 bytes
model_artifact_sizes.csv: 641 bytes
overall_metrics.csv: 1,508 bytes
patient_level_confusion_matrix_cnn_lstm.csv: 280 bytes
patient_level_confusion_matrix_lightweight_mel_cnn.csv: 280 bytes
patient_level_confusion_matrix_logistic_regression.csv: 280 bytes
patient_level_confusion_matrix_multifeature_cnn.csv: 280 bytes
patient_level_confusion_matrix_random_forest.csv: 280 bytes
patient_level_confusion_matrix_regularized_multifeature_cnn.csv: 280 bytes
patient_level_confusion_matrix_svm.csv: 280 bytes
patient_level_overall_metrics.csv: 957 bytes
patient_level_per_class_metrics.csv: 3,200 byte

In [55]:

from pathlib import Path

root = PROJECT_ROOT

print("EXPERIMENT A/B DIRECTORY")
experiment_dir = root / "data/processed/deep_learning/imbalance_comparison"

if experiment_dir.exists():
    for p in sorted(experiment_dir.iterdir()):
        if p.is_file():
            print(f"{p.name}: {p.stat().st_size:,} bytes")
else:
    print("Directory not found:", experiment_dir)

print("\nEXISTING COMMON-SPLIT NEURAL PREDICTIONS")
prediction_paths = [
    root / "data/processed/deep_learning/cnn_baseline/cnn_test_predictions.csv",
    root / "data/processed/deep_learning/multifeature_cnn/multifeature_cnn_test_predictions.csv",
    root / "data/processed/deep_learning/multifeature_cnn_12b/test_predictions.csv",
    root / "data/processed/deep_learning/cnn_lstm/cnn_lstm_test_predictions.csv",
]

for p in prediction_paths:
    print(f"{p.relative_to(root)}: {'EXISTS' if p.exists() else 'MISSING'}")

print("\nRead-only inspection complete.")


EXPERIMENT A/B DIRECTORY
experiment_A_weighted_ce_history.csv: 2,011 bytes
experiment_A_weighted_ce_validation_metrics.json: 420 bytes
experiment_A_weighted_ce_validation_report.csv: 426 bytes
experiment_B_balanced_sampler_history.csv: 1,989 bytes
experiment_B_balanced_sampler_validation_metrics.json: 417 bytes
experiment_B_balanced_sampler_validation_report.csv: 372 bytes

EXISTING COMMON-SPLIT NEURAL PREDICTIONS
data/processed/deep_learning/cnn_baseline/cnn_test_predictions.csv: EXISTS
data/processed/deep_learning/multifeature_cnn/multifeature_cnn_test_predictions.csv: EXISTS
data/processed/deep_learning/multifeature_cnn_12b/test_predictions.csv: EXISTS
data/processed/deep_learning/cnn_lstm/cnn_lstm_test_predictions.csv: EXISTS

Read-only inspection complete.


In [56]:

from pathlib import Path
import json
import pandas as pd

root = PROJECT_ROOT

print("EXPERIMENT CONFIG FILES")
config_dir = root / "models/deep_learning/imbalance_comparison"

for name in [
    "experiment_config.json",
    "imbalance_comparison_config.json",
]:
    path = config_dir / name
    print(f"\n{name}: {'EXISTS' if path.exists() else 'MISSING'}")
    if path.exists():
        print(path.read_text(encoding="utf-8"))

print("\nTRAINING NORMALIZATION STATISTICS")
norm_path = (
    root / "data/processed/deep_learning/multifeature_cnn"
    / "multifeature_train_normalization.csv"
)

if norm_path.exists():
    display(pd.read_csv(norm_path))
else:
    print("Normalization file not found:", norm_path)

print("\nEXISTING TEST MANIFEST")
test_path = (
    root / "data/processed/deep_learning/cnn_baseline"
    / "test_cnn_manifest.csv"
)

if test_path.exists():
    test_manifest = pd.read_csv(test_path)
    print("Rows:", len(test_manifest))
    print("Columns:", test_manifest.columns.tolist())
    display(test_manifest.head(3))
else:
    print("Test manifest not found:", test_path)

print("\nRead-only inspection complete.")


EXPERIMENT CONFIG FILES

experiment_config.json: EXISTS
{
  "experiment_group": "disease_imbalance_comparison",
  "classes": [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "LRTI",
    "Pneumonia",
    "URTI"
  ],
  "split_policy": "existing_patient_disjoint_cnn_manifests",
  "experiment_A": {
    "name": "weighted_cross_entropy",
    "sampling": "shuffle",
    "loss": "class_weighted_cross_entropy"
  },
  "experiment_B": {
    "name": "balanced_sampling",
    "sampling": "WeightedRandomSampler",
    "loss": "unweighted_cross_entropy"
  },
  "checkpoint_selection": "validation_macro_f1",
  "additional_validation_metric": "validation_loss",
  "test_policy": "do_not_use_for_model_selection",
  "preserve_existing_artifacts": true
}

imbalance_comparison_config.json: EXISTS
{
  "experiment_group": "disease_imbalance_comparison",
  "seed": 42,
  "num_classes": 8,
  "class_names": [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",

,feature,count,mean,std
0,logmel,44442304,-66.230481,17.661886
1,mfcc,9027343,-22.509034,137.274416
2,chroma,8332932,0.385656,0.389059



EXISTING TEST MANIFEST
Rows: 1561
Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split']


,patient_id,recording_id,chest_location,diagnosis,disease_label,official_split,audio_path,cnn_split
0,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
1,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
2,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test



Read-only inspection complete.


In [57]:

import json
from pathlib import Path

notebook_path = (
    PROJECT_ROOT
    / "Notebooks 2"
    / "04_ICBHI_Disease_Transfer_Learning.ipynb"
)

if not notebook_path.exists():
    raise FileNotFoundError(f"Notebook not found: {notebook_path}")

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

keywords = [
    "class FeatureBranch",
    "class MultiFeatureCNN",
    "def extract",
    "def audio_to",
    "def evaluate_model",
    "TARGET_SR",
    "N_FFT",
    "normalization",
    "torch.load",
    "torch.save",
]

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    matched = [k for k in keywords if k.lower() in source.lower()]

    if matched:
        print("\n" + "=" * 90)
        print(f"CELL {i} | matches: {matched}")
        print(source)

print("\nRead-only inspection complete.")



CELL 4 | matches: ['normalization']

import json
from pathlib import Path

ROOT = Path.cwd()
source_nb = ROOT / "Notebooks/12_multifeature_cnn.ipynb"

if not source_nb.exists():
    raise FileNotFoundError(f"Notebook not found: {source_nb}")

with open(source_nb, "r", encoding="utf-8") as f:
    nb = json.load(f)

keywords = [
    "mfcc", "mel", "chroma", "spectrogram",
    "class ", "def forward", "conv2d", "conv1d",
    "dataset", "dataloader", "normalization",
    "optimizer", "criterion", "checkpoint",
    "train_manifest", "validation_manifest"
]

print("SOURCE NOTEBOOK:", source_nb.name)
print("Total cells:", len(nb["cells"]))

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    matches = [word for word in keywords if word in source.lower()]

    if matches:
        print(f"\n{'=' * 25} CELL {i} {'=' * 25}")
        print("Matched topics:", ", ".join(matches))
        print(source[:7000

In [58]:

from pathlib import Path
import torch

root = PROJECT_ROOT

checkpoint_paths = {
    "A": root / "models/deep_learning/imbalance_comparison/experiment_A_weighted_ce_best.pt",
    "B": root / "models/deep_learning/imbalance_comparison/experiment_B_balanced_sampler_best.pt",
}

expected_classes = [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "LRTI",
    "Pneumonia",
    "URTI",
]

for key, path in checkpoint_paths.items():
    print("\n" + "=" * 70)
    print(f"EXPERIMENT {key}")
    print("Path:", path.relative_to(root))
    print("Exists:", path.exists())

    if not path.exists():
        continue

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print("Top-level type:", type(checkpoint).__name__)

    if not isinstance(checkpoint, dict):
        print("Unexpected checkpoint format.")
        continue

    print("Keys:", list(checkpoint.keys()))
    print("Architecture:", checkpoint.get("architecture"))
    print("Saved classes:", checkpoint.get("class_names"))
    print("Saved epoch:", checkpoint.get("epoch"))
    print("Saved parameter count:", checkpoint.get("total_parameters"))

    state = checkpoint.get("model_state_dict")
    if not isinstance(state, dict):
        print("ERROR: model_state_dict is missing.")
        continue

    print("State-dict tensors:", len(state))
    print("State-dict tensor elements:",
          f"{sum(v.numel() for v in state.values() if isinstance(v, torch.Tensor)):,}")

    assert checkpoint.get("architecture") == "MultiFeatureCNN"
    assert checkpoint.get("class_names") == expected_classes
    assert len(state) > 0

print("\nCheckpoint compatibility metadata inspected.")



EXPERIMENT A
Path: models/deep_learning/imbalance_comparison/experiment_A_weighted_ce_best.pt
Exists: True
Top-level type: dict
Keys: ['model_state_dict', 'num_classes', 'class_names', 'epoch', 'validation_macro_f1_supported_classes', 'validation_metrics', 'seed', 'experiment', 'architecture', 'total_parameters']
Architecture: MultiFeatureCNN
Saved classes: ['Asthma', 'Bronchiectasis', 'Bronchiolitis', 'COPD', 'Healthy', 'LRTI', 'Pneumonia', 'URTI']
Saved epoch: 3
Saved parameter count: 108776
State-dict tensors: 73
State-dict tensor elements: 109,457

EXPERIMENT B
Path: models/deep_learning/imbalance_comparison/experiment_B_balanced_sampler_best.pt
Exists: True
Top-level type: dict
Keys: ['model_state_dict', 'num_classes', 'class_names', 'epoch', 'validation_macro_f1_supported_classes', 'validation_metrics', 'seed', 'experiment', 'architecture', 'total_parameters']
Architecture: MultiFeatureCNN
Saved classes: ['Asthma', 'Bronchiectasis', 'Bronchiolitis', 'COPD', 'Healthy', 'LRTI', 'P

In [59]:

import torch
import torch.nn as nn
from pathlib import Path

root = PROJECT_ROOT

class FeatureBranch(nn.Module):
    def __init__(self, output_dim=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, output_dim),
            nn.ReLU(),
            nn.Dropout(0.30),
        )

    def forward(self, x):
        return self.projection(self.features(x))


class MultiFeatureCNN(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        self.logmel_branch = FeatureBranch(64)
        self.mfcc_branch = FeatureBranch(64)
        self.chroma_branch = FeatureBranch(64)
        self.classifier = nn.Sequential(
            nn.Linear(192, 128),
            nn.ReLU(),
            nn.Dropout(0.40),
            nn.Linear(128, num_classes),
        )

    def forward(self, logmel, mfcc, chroma):
        logmel_features = self.logmel_branch(logmel)
        mfcc_features = self.mfcc_branch(mfcc)
        chroma_features = self.chroma_branch(chroma)
        combined = torch.cat(
            [logmel_features, mfcc_features, chroma_features],
            dim=1,
        )
        return self.classifier(combined)


device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

checkpoint_paths = {
    "A": root / "models/deep_learning/imbalance_comparison/experiment_A_weighted_ce_best.pt",
    "B": root / "models/deep_learning/imbalance_comparison/experiment_B_balanced_sampler_best.pt",
}

loaded_models = {}

for key, path in checkpoint_paths.items():
    checkpoint = torch.load(
        path, map_location="cpu", weights_only=False
    )
    model = MultiFeatureCNN(num_classes=8)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model = model.to(device).eval()

    actual_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )

    print(
        f"Experiment {key}: loaded successfully | "
        f"parameters={actual_params:,} | "
        f"trainable={trainable_params:,} | "
        f"device={device}"
    )
    print(
        "Checkpoint metadata parameter count:",
        checkpoint["total_parameters"]
    )

    loaded_models[key] = model

print("\nBoth models loaded. No inference or training performed.")


Experiment A: loaded successfully | parameters=108,776 | trainable=108,776 | device=mps
Checkpoint metadata parameter count: 108776
Experiment B: loaded successfully | parameters=108,776 | trainable=108,776 | device=mps
Checkpoint metadata parameter count: 108776

Both models loaded. No inference or training performed.


In [60]:

import json
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)

root = PROJECT_ROOT
data_dir = root / "data/processed/deep_learning/cnn_baseline"
norm_path = (
    root / "data/processed/deep_learning/multifeature_cnn"
    / "multifeature_train_normalization.csv"
)
test_path = data_dir / "test_cnn_manifest.csv"

class_names = [
    "Asthma", "Bronchiectasis", "Bronchiolitis", "COPD",
    "Healthy", "LRTI", "Pneumonia", "URTI",
]

# Load the original test manifest and saved training-only normalization.
test_manifest = pd.read_csv(test_path).reset_index(drop=True)
norm_df = pd.read_csv(norm_path).set_index("feature")

logmel_mean = float(norm_df.loc["logmel", "mean"])
logmel_std = float(norm_df.loc["logmel", "std"])
mfcc_mean = float(norm_df.loc["mfcc", "mean"])
mfcc_std = float(norm_df.loc["mfcc", "std"])
chroma_mean = float(norm_df.loc["chroma", "mean"])
chroma_std = float(norm_df.loc["chroma", "std"])

TARGET_SR = 4000
TARGET_SAMPLES = 20_000
N_FFT = 512
HOP_LENGTH = 128
N_MELS = 64
N_MFCC = 13
N_CHROMA = 12

def extract_test_features(audio_path):
    path = Path(str(audio_path))
    if not path.is_absolute():
        path = (root / path).resolve()

    signal, sr = sf.read(path)
    if signal.ndim > 1:
        signal = signal.mean(axis=1)

    if sr != TARGET_SR:
        raise ValueError(f"Expected {TARGET_SR} Hz, got {sr}: {path}")

    if len(signal) < TARGET_SAMPLES:
        signal = np.pad(signal, (0, TARGET_SAMPLES - len(signal)))
    else:
        signal = signal[:TARGET_SAMPLES]

    signal = signal.astype(np.float32)

    logmel = librosa.power_to_db(
        librosa.feature.melspectrogram(
            y=signal, sr=TARGET_SR, n_fft=N_FFT,
            hop_length=HOP_LENGTH, n_mels=N_MELS,
            fmin=0, fmax=2000, power=2.0,
        ),
        ref=np.max,
    ).astype(np.float32)

    mfcc = librosa.feature.mfcc(
        y=signal, sr=TARGET_SR, n_mfcc=N_MFCC,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
        fmin=0, fmax=2000,
    ).astype(np.float32)

    chroma = librosa.feature.chroma_stft(
        y=signal, sr=TARGET_SR, n_fft=N_FFT,
        hop_length=HOP_LENGTH, n_chroma=N_CHROMA,
        tuning=0.0,
    ).astype(np.float32)

    features = [
        (logmel - logmel_mean) / logmel_std,
        (mfcc - mfcc_mean) / mfcc_std,
        (chroma - chroma_mean) / chroma_std,
    ]

    expected_shapes = [(64, 157), (13, 157), (12, 157)]
    for name, feature, expected in zip(
        ["Log-Mel", "MFCC", "Chroma"], features, expected_shapes
    ):
        if feature.shape != expected:
            raise ValueError(
                f"{name} shape {feature.shape}, expected {expected}"
            )
        if not np.isfinite(feature).all():
            raise ValueError(f"Non-finite values in {name}: {path}")

    return [
        torch.from_numpy(feature[np.newaxis].astype(np.float32))
        for feature in features
    ]


# Validate the manifest before inference.
required_cols = {
    "patient_id", "recording_id", "chest_location",
    "diagnosis", "disease_label", "audio_path",
}
missing = required_cols - set(test_manifest.columns)
if missing:
    raise ValueError(f"Test manifest is missing columns: {sorted(missing)}")

if len(test_manifest) != 1561:
    raise ValueError(
        f"Expected 1561 test rows from the current manifest; "
        f"found {len(test_manifest)}"
    )

if not test_manifest["disease_label"].between(0, 7).all():
    raise ValueError("Unexpected disease_label values.")

# Generate deterministic, unaugmented predictions in manifest row order.
all_logits = {"A": [], "B": []}
all_true = []

for row_index, row in test_manifest.iterrows():
    features = extract_test_features(row["audio_path"])
    inputs = [
        tensor.unsqueeze(0).to(device)
        for tensor in features
    ]

    with torch.no_grad():
        for key, model in loaded_models.items():
            logits = model(*inputs)
            all_logits[key].append(logits.cpu().numpy()[0])

    all_true.append(int(row["disease_label"]))

    if (row_index + 1) % 100 == 0 or row_index + 1 == len(test_manifest):
        print(f"Processed {row_index + 1}/{len(test_manifest)} test rows")

y_true = np.asarray(all_true, dtype=int)
predictions_by_experiment = {
    key: np.asarray(logits).argmax(axis=1)
    for key, logits in all_logits.items()
}

# Verify that inference preserved all rows and labels in manifest order.
assert len(y_true) == len(test_manifest)
assert np.array_equal(
    y_true, test_manifest["disease_label"].to_numpy(dtype=int)
)
for key, predictions in predictions_by_experiment.items():
    assert predictions.shape == y_true.shape
    assert np.isin(predictions, np.arange(8)).all()

# Save to a new directory. Do not overwrite historical artifacts.
output_dir = (
    root / "data/processed/deep_learning/imbalance_comparison"
    / "common_patient_holdout_predictions"
)
output_dir.mkdir(parents=True, exist_ok=True)

metrics_rows = []
per_class_rows = []
patient_rows = []

for key, y_pred in predictions_by_experiment.items():
    experiment_name = (
        "weighted_ce" if key == "A" else "balanced_sampler"
    )

    prediction_df = test_manifest[
        [
            "patient_id", "recording_id", "chest_location",
            "diagnosis", "disease_label", "official_split",
            "audio_path", "cnn_split",
        ]
    ].copy()
    prediction_df["true_label"] = y_true
    prediction_df["true_disease"] = [
        class_names[i] for i in y_true
    ]
    prediction_df["predicted_label"] = y_pred
    prediction_df["predicted_disease"] = [
        class_names[i] for i in y_pred
    ]
    prediction_df["experiment"] = key

    prediction_df.to_csv(
        output_dir / f"experiment_{key}_{experiment_name}_test_predictions.csv",
        index=False,
    )

    supported = np.unique(y_true)
    metrics_rows.append({
        "experiment": key,
        "strategy": experiment_name,
        "test_cycles": len(y_true),
        "test_patients": int(test_manifest["patient_id"].nunique()),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1_supported_classes": f1_score(
            y_true, y_pred, labels=supported,
            average="macro", zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_true, y_pred, labels=np.arange(8),
            average="weighted", zero_division=0,
        ),
    })

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=np.arange(8), zero_division=0
    )
    for i, name in enumerate(class_names):
        class_test_rows = test_manifest[
            test_manifest["disease_label"] == i
        ]
        per_class_rows.append({
            "experiment": key,
            "strategy": experiment_name,
            "disease": name,
            "support_cycles": int(support[i]),
            "support_patients": int(class_test_rows["patient_id"].nunique()),
            "precision": float(precision[i]) if support[i] else np.nan,
            "recall": float(recall[i]) if support[i] else np.nan,
            "f1": float(f1[i]) if support[i] else np.nan,
            "metric_status": "measured" if support[i] else "no held-out support",
        })

    # Patient-level majority vote, keeping each patient's true diagnosis.
    for patient_id, patient_frame in prediction_df.groupby("patient_id", sort=False):
        true_labels = patient_frame["true_label"].unique()
        if len(true_labels) != 1:
            raise ValueError(
                f"Patient {patient_id} has conflicting true labels."
            )

        votes = np.bincount(
            patient_frame["predicted_label"].to_numpy(dtype=int),
            minlength=8,
        )
        patient_rows.append({
            "experiment": key,
            "patient_id": patient_id,
            "true_label": int(true_labels[0]),
            "true_disease": class_names[int(true_labels[0])],
            "predicted_label": int(votes.argmax()),
            "predicted_disease": class_names[int(votes.argmax())],
            "cycles": len(patient_frame),
        })

metrics_df = pd.DataFrame(metrics_rows)
per_class_df = pd.DataFrame(per_class_rows)
patient_df_results = pd.DataFrame(patient_rows)

metrics_df.to_csv(output_dir / "overall_metrics.csv", index=False)
per_class_df.to_csv(output_dir / "per_class_metrics.csv", index=False)
patient_df_results.to_csv(output_dir / "patient_level_predictions.csv", index=False)

protocol = {
    "protocol": "existing common patient-disjoint CNN holdout",
    "test_rows": len(test_manifest),
    "test_patients": int(test_manifest["patient_id"].nunique()),
    "official_test_protocol": False,
    "models_retrained": False,
    "normalization": "saved training-only statistics from multifeature_train_normalization.csv",
    "test_used_for_model_selection": False,
    "prediction_order": "original test manifest row order",
    "limitations": [
        "This is not the official ICBHI test cohort.",
        "Asthma and LRTI have no held-out support in this test manifest.",
        "Cycle-level metrics include correlated cycles from the same patient.",
    ],
}
(output_dir / "protocol.json").write_text(
    json.dumps(protocol, indent=2) + "\n",
    encoding="utf-8",
)

print("\nOVERALL RESULTS")
display(metrics_df)

print("\nPER-CLASS RESULTS")
display(per_class_df)

print("\nPATIENT-LEVEL PREDICTIONS")
display(patient_df_results.head())

print("\nSaved new predictions and reports to:")
print(output_dir)


Processed 100/1561 test rows
Processed 200/1561 test rows
Processed 300/1561 test rows
Processed 400/1561 test rows
Processed 500/1561 test rows
Processed 600/1561 test rows
Processed 700/1561 test rows
Processed 800/1561 test rows
Processed 900/1561 test rows
Processed 1000/1561 test rows
Processed 1100/1561 test rows
Processed 1200/1561 test rows
Processed 1300/1561 test rows
Processed 1400/1561 test rows
Processed 1500/1561 test rows
Processed 1561/1561 test rows

OVERALL RESULTS


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


,experiment,strategy,test_cycles,test_patients,accuracy,balanced_accuracy,macro_f1_supported_classes,weighted_f1
0,A,weighted_ce,1561,25,0.806534,0.251776,0.228644,0.804531
1,B,balanced_sampler,1561,25,0.830878,0.304497,0.234531,0.810587



PER-CLASS RESULTS


,experiment,strategy,disease,support_cycles,support_patients,precision,recall,f1,metric_status
0,A,weighted_ce,Asthma,0,0,NaN,NaN,NaN,no held-out support
1,A,weighted_ce,Bronchiectasis,20,2,0.000000,0.000000,0.000000,measured
2,A,weighted_ce,Bronchiolitis,19,1,0.000000,0.000000,0.000000,measured
3,A,weighted_ce,COPD,1337,13,0.917236,0.903515,0.910324,measured
4,A,weighted_ce,Healthy,84,5,0.372263,0.607143,0.461538,measured
5,A,weighted_ce,LRTI,0,0,NaN,NaN,NaN,no held-out support
6,A,weighted_ce,Pneumonia,52,1,0.000000,0.000000,0.000000,measured
7,A,weighted_ce,URTI,49,3,0.000000,0.000000,0.000000,measured
8,B,balanced_sampler,Asthma,0,0,NaN,NaN,NaN,no held-out support
9,B,balanced_sampler,Bronchiectasis,20,2,0.000000,0.000000,0.000000,measured



PATIENT-LEVEL PREDICTIONS


,experiment,patient_id,true_label,true_disease,predicted_label,predicted_disease,cycles
0,A,101,7,URTI,4,Healthy,23
1,A,106,3,COPD,3,COPD,18
2,A,107,3,COPD,3,COPD,231
3,A,119,7,URTI,4,Healthy,9
4,A,121,4,Healthy,4,Healthy,14



Saved new predictions and reports to:
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/imbalance_comparison/common_patient_holdout_predictions


In [61]:
from pathlib import Path
import pandas as pd

audit_dir = (
    PROJECT_ROOT
    / "reports/research_audit/common_patient_split"
)

for filename in [
    "overall_metrics.csv",
    "per_class_metrics.csv",
    "model_artifact_sizes.csv",
]:
    path = audit_dir / filename
    print(f"\n{'=' * 70}\n{filename}\n{'=' * 70}")

    if path.exists():
        df = pd.read_csv(path)
        display(df)
    else:
        print("Not found:", path)


overall_metrics.csv


,model_id,training_patients,validation_patients_not_used,test_patients,test_recordings,test_cycle_support,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,test_cycles,estimator_predict_ms_per_cycle,serialized_estimator_bytes,deployed_artifact_bytes
0,logistic_regression,82,19,25,53,1561,0.782191,0.402684,0.400001,0.402684,0.389169,0.810493,1561,0.000188,2896.0,2782
1,svm,82,19,25,53,1561,0.846893,0.372946,0.404335,0.372946,0.372833,0.849899,1561,0.149212,681083.0,550636
2,random_forest,82,19,25,53,1561,0.855221,0.284939,0.289027,0.284939,0.283384,0.839171,1561,0.025272,21550699.0,4329187
3,lightweight_mel_cnn,82,19,25,53,1561,0.716848,0.310168,0.248526,0.310168,0.248360,0.768721,1561,NaN,NaN,310935
4,multifeature_cnn,82,19,25,53,1561,0.855221,0.166417,0.142720,0.166417,0.153660,0.789662,1561,NaN,NaN,464538
5,regularized_multifeature_cnn,82,19,25,53,1561,0.856502,0.166667,0.142750,0.166667,0.153784,0.790299,1561,NaN,NaN,465550
6,cnn_lstm,82,19,25,53,1561,0.813581,0.226085,0.179297,0.226085,0.195295,0.786735,1561,NaN,NaN,373412



per_class_metrics.csv


,model_id,disease,precision,recall,f1,support_cycles,metric_status,support_patients,support_recordings
0,logistic_regression,Asthma,NaN,NaN,NaN,0,no held-out support,0,0
1,logistic_regression,Bronchiectasis,0.769231,0.500000,0.606061,20,measured,2,3
2,logistic_regression,Bronchiolitis,0.194444,0.368421,0.254545,19,measured,1,1
3,logistic_regression,COPD,0.949346,0.869110,0.907458,1337,measured,13,37
4,logistic_regression,Healthy,0.173554,0.250000,0.204878,84,measured,5,6
5,logistic_regression,LRTI,NaN,NaN,NaN,0,no held-out support,0,0
6,logistic_regression,Pneumonia,0.000000,0.000000,0.000000,52,measured,1,3
7,logistic_regression,URTI,0.313433,0.428571,0.362069,49,measured,3,3
8,svm,Asthma,NaN,NaN,NaN,0,no held-out support,0,0
9,svm,Bronchiectasis,0.666667,0.300000,0.413793,20,measured,2,3



model_artifact_sizes.csv


,model_id,artifact_path,artifact_bytes
0,logistic_regression,models/traditional_ml_patient_disjoint/logisti...,2782
1,svm,models/traditional_ml_patient_disjoint/svm.joblib,550636
2,random_forest,models/traditional_ml_patient_disjoint/random_...,4329187
3,lightweight_mel_cnn,models/deep_learning/lightweight_mel_cnn/best_...,310935
4,multifeature_cnn,models/deep_learning/multifeature_cnn/best_mul...,464538
5,regularized_multifeature_cnn,models/deep_learning/regularized_multifeature_...,465550
6,cnn_lstm,models/deep_learning/cnn_lstm/best_cnn_lstm.pt,373412


In [62]:

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

root = PROJECT_ROOT
audit_dir = root / "reports/research_audit/common_patient_split"
new_dir = (
    root / "data/processed/deep_learning/imbalance_comparison"
    / "common_patient_holdout_predictions"
)

print("Existing patient-level audit files:")
for path in sorted(audit_dir.glob("*patient*")):
    print(" -", path.name)

new_patient_path = new_dir / "patient_level_predictions.csv"
new_patients = pd.read_csv(new_patient_path)

class_names = [
    "Asthma", "Bronchiectasis", "Bronchiolitis", "COPD",
    "Healthy", "LRTI", "Pneumonia", "URTI",
]

rows = []

for experiment, frame in new_patients.groupby("experiment"):
    y_true = frame["true_label"].to_numpy(dtype=int)
    y_pred = frame["predicted_label"].to_numpy(dtype=int)
    supported = np.unique(y_true)

    rows.append({
        "model": f"Experiment {experiment}",
        "patients": frame["patient_id"].nunique(),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1_supported_classes": f1_score(
            y_true, y_pred, labels=supported,
            average="macro", zero_division=0
        ),
    })

    print(f"\nExperiment {experiment}: patient-level report")
    print(classification_report(
        y_true, y_pred,
        labels=np.arange(8),
        target_names=class_names,
        zero_division=0,
    ))

print("\nPatient-level summary for new experiments:")
display(pd.DataFrame(rows))

# Show candidate existing patient-level files for the next comparison.
for path in sorted(audit_dir.glob("*patient*metrics*.csv")):
    print(f"\nExisting metrics: {path.name}")
    display(pd.read_csv(path).head(20))


Existing patient-level audit files:
 - patient_level_confusion_matrix_cnn_lstm.csv
 - patient_level_confusion_matrix_lightweight_mel_cnn.csv
 - patient_level_confusion_matrix_logistic_regression.csv
 - patient_level_confusion_matrix_multifeature_cnn.csv
 - patient_level_confusion_matrix_random_forest.csv
 - patient_level_confusion_matrix_regularized_multifeature_cnn.csv
 - patient_level_confusion_matrix_svm.csv
 - patient_level_overall_metrics.csv
 - patient_level_per_class_metrics.csv
 - patient_level_predictions.csv

Experiment A: patient-level report
                precision    recall  f1-score   support

        Asthma       0.00      0.00      0.00         0
Bronchiectasis       0.00      0.00      0.00         2
 Bronchiolitis       0.00      0.00      0.00         1
          COPD       0.71      0.92      0.80        13
       Healthy       0.43      0.60      0.50         5
          LRTI       0.00      0.00      0.00         0
     Pneumonia       0.00      0.00      0.00  

,model,patients,accuracy,balanced_accuracy,macro_f1_supported_classes
0,Experiment A,25,0.6,0.253846,0.216667
1,Experiment B,25,0.6,0.366667,0.242960



Existing metrics: patient_level_overall_metrics.csv


,model_id,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,test_patients
0,logistic_regression,0.64,0.338889,0.440972,0.338889,0.363711,0.616683,25
1,svm,0.68,0.322222,0.277083,0.322222,0.297044,0.620493,25
2,random_forest,0.68,0.322222,0.322368,0.322222,0.313194,0.603833,25
3,lightweight_mel_cnn,0.56,0.333333,0.177083,0.333333,0.216092,0.482207,25
4,multifeature_cnn,0.52,0.166667,0.086667,0.166667,0.114035,0.355789,25
5,regularized_multifeature_cnn,0.52,0.166667,0.086667,0.166667,0.114035,0.355789,25
6,cnn_lstm,0.48,0.153846,0.086957,0.153846,0.111111,0.346667,25



Existing metrics: patient_level_per_class_metrics.csv


,model_id,disease,precision,recall,f1,metric_status,support_patients,support_recordings
0,logistic_regression,Asthma,NaN,NaN,NaN,no held-out support,0,0
1,logistic_regression,Bronchiectasis,1.000000,0.500000,0.666667,measured,2,3
2,logistic_regression,Bronchiolitis,0.000000,0.000000,0.000000,measured,1,1
3,logistic_regression,COPD,0.812500,1.000000,0.896552,measured,13,37
4,logistic_regression,Healthy,0.500000,0.200000,0.285714,measured,5,6
5,logistic_regression,LRTI,NaN,NaN,NaN,no held-out support,0,0
6,logistic_regression,Pneumonia,0.000000,0.000000,0.000000,measured,1,3
7,logistic_regression,URTI,0.333333,0.333333,0.333333,measured,3,3
8,svm,Asthma,NaN,NaN,NaN,no held-out support,0,0
9,svm,Bronchiectasis,0.000000,0.000000,0.000000,measured,2,3


In [63]:
from pathlib import Path
import json

root = PROJECT_ROOT

files = [
    root / "models/deep_learning/imbalance_comparison/experiment_config.json",
    root / "models/deep_learning/imbalance_comparison/imbalance_comparison_config.json",
    root / "reports/research_audit/common_patient_split/protocol.json",
]

for path in files:
    print("\n" + "=" * 80)
    print(path.relative_to(root))
    print("=" * 80)

    if path.exists():
        try:
            print(json.dumps(
                json.loads(path.read_text(encoding="utf-8")),
                indent=2
            ))
        except json.JSONDecodeError:
            print(path.read_text(encoding="utf-8"))
    else:
        print("FILE NOT FOUND")


models/deep_learning/imbalance_comparison/experiment_config.json
{
  "experiment_group": "disease_imbalance_comparison",
  "classes": [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "LRTI",
    "Pneumonia",
    "URTI"
  ],
  "split_policy": "existing_patient_disjoint_cnn_manifests",
  "experiment_A": {
    "name": "weighted_cross_entropy",
    "sampling": "shuffle",
    "loss": "class_weighted_cross_entropy"
  },
  "experiment_B": {
    "name": "balanced_sampling",
    "sampling": "WeightedRandomSampler",
    "loss": "unweighted_cross_entropy"
  },
  "checkpoint_selection": "validation_macro_f1",
  "additional_validation_metric": "validation_loss",
  "test_policy": "do_not_use_for_model_selection",
  "preserve_existing_artifacts": true
}

models/deep_learning/imbalance_comparison/imbalance_comparison_config.json
{
  "experiment_group": "disease_imbalance_comparison",
  "seed": 42,
  "num_classes": 8,
  "class_names": [
    "Asthma",
    "Bronc

In [64]:

from pathlib import Path
import pandas as pd

root = PROJECT_ROOT
manifest_dir = root / "data/processed/deep_learning/cnn_baseline"

split_files = {
    "train": manifest_dir / "train_cnn_manifest.csv",
    "validation": manifest_dir / "validation_cnn_manifest.csv",
    "test": manifest_dir / "test_cnn_manifest.csv",
}

class_names = [
    "Asthma", "Bronchiectasis", "Bronchiolitis", "COPD",
    "Healthy", "LRTI", "Pneumonia", "URTI",
]

frames = {}

for split, path in split_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Manifest not found: {path}")

    df = pd.read_csv(path)
    required = {"patient_id", "disease_label"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path.name} missing columns: {sorted(missing)}")

    df = df.copy()
    df["split"] = split
    frames[split] = df

    print(
        f"{split}: {len(df)} cycles, "
        f"{df['patient_id'].nunique()} patients"
    )

# Check that no patient appears in more than one split.
patient_sets = {
    split: set(df["patient_id"].dropna().unique())
    for split, df in frames.items()
}

for left, right in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    overlap = patient_sets[left] & patient_sets[right]
    print(f"{left} ∩ {right}: {len(overlap)} overlapping patients")
    if overlap:
        print("WARNING: overlapping patient IDs:", sorted(overlap))

# Summarize cycle and patient support by disease and split.
combined = pd.concat(frames.values(), ignore_index=True)
combined["disease"] = combined["disease_label"].map(
    dict(enumerate(class_names))
)

summary = (
    combined.groupby(["disease", "split"], dropna=False)
    .agg(
        cycles=("disease_label", "size"),
        patients=("patient_id", "nunique"),
    )
    .reset_index()
)

table = summary.pivot(
    index="disease",
    columns="split",
    values=["cycles", "patients"],
).fillna(0).astype(int)

display(table.reindex(class_names))


train: 4423 cycles, 82 patients
validation: 914 cycles, 19 patients
test: 1561 cycles, 25 patients
train ∩ validation: 0 overlapping patients
train ∩ test: 0 overlapping patients
validation ∩ test: 0 overlapping patients


cycles                  patients                 
split            test train validation     test train validation
disease                                                         
Asthma              0     6          0        0     1          0
Bronchiectasis     20    76          8        2     4          1
Bronchiolitis      19   114         27        1     4          1
COPD             1337  3713        696       13    41         10
Healthy            84   194         44        5    17          4
LRTI                0    32          0        0     2          0
Pneumonia          52   121        112        1     4          1
URTI               49   167         27        3     9          2

In [65]:

import pandas as pd
from pathlib import Path

root = PROJECT_ROOT
manifest_dir = root / "data/processed/deep_learning/cnn_baseline"

paths = {
    "train": manifest_dir / "train_cnn_manifest.csv",
    "validation": manifest_dir / "validation_cnn_manifest.csv",
    "test": manifest_dir / "test_cnn_manifest.csv",
}

class_names = [
    "Asthma", "Bronchiectasis", "Bronchiolitis", "COPD",
    "Healthy", "LRTI", "Pneumonia", "URTI",
]

frames = []
for split, path in paths.items():
    df = pd.read_csv(path)
    df["split"] = split
    frames.append(df)

all_data = pd.concat(frames, ignore_index=True)
all_data["disease"] = all_data["disease_label"].map(
    dict(enumerate(class_names))
)

full_support = (
    all_data.groupby("disease")
    .agg(
        total_cycles=("disease_label", "size"),
        total_patients=("patient_id", "nunique"),
    )
    .reindex(class_names)
    .reset_index()
)

print("Full current dataset: disease support")
display(full_support)

print("\nDistinct patients per disease and split")
patient_counts = all_data.pivot_table(
    index="disease",
    columns="split",
    values="patient_id",
    aggfunc=pd.Series.nunique,
    fill_value=0,
).reindex(class_names)

display(patient_counts)


Full current dataset: disease support


,disease,total_cycles,total_patients
0,Asthma,6,1
1,Bronchiectasis,104,7
2,Bronchiolitis,160,6
3,COPD,5746,64
4,Healthy,322,26
5,LRTI,32,2
6,Pneumonia,285,6
7,URTI,243,14



Distinct patients per disease and split


split,test,train,validation
disease,,,
Asthma,0,1,0
Bronchiectasis,2,4,1
Bronchiolitis,1,4,1
COPD,13,41,10
Healthy,5,17,4
LRTI,0,2,0
Pneumonia,1,4,1
URTI,3,9,2


In [66]:
from pathlib import Path
import pandas as pd

root = PROJECT_ROOT
manifest_dir = root / "data/processed/deep_learning/cnn_baseline"

files = [
    manifest_dir / "train_cnn_manifest.csv",
    manifest_dir / "validation_cnn_manifest.csv",
    manifest_dir / "test_cnn_manifest.csv",
]

df = pd.concat(
    [pd.read_csv(path) for path in files],
    ignore_index=True,
)

class_names = [
    "Asthma", "Bronchiectasis", "Bronchiolitis", "COPD",
    "Healthy", "LRTI", "Pneumonia", "URTI",
]
df["disease"] = df["disease_label"].map(dict(enumerate(class_names)))

patient_label_counts = (
    df.groupby("patient_id")
    .agg(
        distinct_diseases=("disease", "nunique"),
        diseases=("disease", lambda s: ", ".join(sorted(s.unique()))),
        cycles=("disease_label", "size"),
        splits=("cnn_split", lambda s: ", ".join(sorted(s.unique()))),
    )
    .reset_index()
)

print("Unique patients:", df["patient_id"].nunique())
print("Patients with multiple disease labels:",
      int((patient_label_counts["distinct_diseases"] > 1).sum()))

display(
    patient_label_counts[
        patient_label_counts["distinct_diseases"] > 1
    ].sort_values("distinct_diseases", ascending=False)
)

print("\nPatient counts by disease (a patient may appear in multiple rows):")
display(
    df.groupby("disease")["patient_id"]
    .nunique()
    .reindex(class_names)
    .rename("distinct_patients")
    .to_frame()
)

Unique patients: 126
Patients with multiple disease labels: 0


,patient_id,distinct_diseases,diseases,cycles,splits



Patient counts by disease (a patient may appear in multiple rows):


,distinct_patients
disease,
Asthma,1
Bronchiectasis,7
Bronchiolitis,6
COPD,64
Healthy,26
LRTI,2
Pneumonia,6
URTI,14


In [67]:
from pathlib import Path
import pandas as pd

root = PROJECT_ROOT
manifest_dir = root / "data/processed/deep_learning/cnn_baseline"

files = {
    "train": manifest_dir / "train_cnn_manifest.csv",
    "validation": manifest_dir / "validation_cnn_manifest.csv",
    "test": manifest_dir / "test_cnn_manifest.csv",
}

frames = []
for split, path in files.items():
    df = pd.read_csv(path)
    df["current_split"] = split
    frames.append(df)

data = pd.concat(frames, ignore_index=True)

if "official_split" not in data.columns:
    print("The manifests do not contain an official_split column.")
else:
    print("Official split values:")
    print(data["official_split"].value_counts(dropna=False))

    print("\nOfficial split by current split:")
    display(pd.crosstab(
        data["official_split"],
        data["current_split"],
        margins=True,
        dropna=False,
    ))

    print("\nDistinct patients by official split:")
    patient_split = (
        data.groupby("patient_id")
        .agg(
            official_memberships=("official_split", "nunique"),
            official_split=("official_split", "first"),
            current_split=("current_split", "first"),
        )
        .reset_index()
    )

    print(
        "Patients with multiple official memberships:",
        int((patient_split["official_memberships"] > 1).sum())
    )
    display(pd.crosstab(
        patient_split["official_split"],
        patient_split["current_split"],
        margins=True,
    ))

Official split values:
official_split
train    4142
test     2756
Name: count, dtype: int64

Official split by current split:


current_split,test,train,validation,All
official_split,,,,
test,480,1943,333,2756
train,1081,2480,581,4142
All,1561,4423,914,6898



Distinct patients by official split:
Patients with multiple official memberships: 2


current_split,test,train,validation,All
official_split,,,,
test,11,31,7,49
train,14,51,12,77
All,25,82,19,126


In [68]:

import pandas as pd
from pathlib import Path

root = PROJECT_ROOT
manifest_dir = root / "data/processed/deep_learning/cnn_baseline"

paths = [
    manifest_dir / "train_cnn_manifest.csv",
    manifest_dir / "validation_cnn_manifest.csv",
    manifest_dir / "test_cnn_manifest.csv",
]

df = pd.concat(
    [pd.read_csv(path) for path in paths],
    ignore_index=True,
).drop_duplicates()

required = {
    "patient_id", "official_split", "recording_id",
    "disease_label", "current_split",
}
# Some manifests use cnn_split rather than current_split.
if "current_split" not in df.columns and "cnn_split" in df.columns:
    df = df.rename(columns={"cnn_split": "current_split"})

missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

membership_counts = (
    df.groupby("patient_id")["official_split"]
    .nunique(dropna=False)
)
mixed_ids = membership_counts[membership_counts > 1].index.tolist()

print("Patients with mixed official membership:", mixed_ids)

details = df[df["patient_id"].isin(mixed_ids)].sort_values(
    ["patient_id", "recording_id", "official_split"]
)

display(details[
    [
        "patient_id", "recording_id", "official_split",
        "current_split", "disease_label"
    ]
].drop_duplicates())

print("\nSummary by patient, recording and official membership:")
display(
    details.groupby(
        ["patient_id", "current_split", "official_split"],
        dropna=False
    )
    .agg(
        recordings=("recording_id", "nunique"),
        cycles=("disease_label", "size"),
    )
    .reset_index()
)


Patients with mixed official membership: [156, 218]


,patient_id,recording_id,official_split,current_split,disease_label
2090,156,2b3,test,train,3
2096,156,2b3,train,train,3
2135,156,5b3,test,train,3
2126,156,5b3,train,train,3
2180,156,8b3,test,train,3
2188,156,8b3,train,train,3
6685,218,1b1,test,test,3
6700,218,1b1,train,test,3
6777,218,1p1,test,test,3
6771,218,1p1,train,test,3



Summary by patient, recording and official membership:


,patient_id,current_split,official_split,recordings,cycles
0,156,train,test,3,61
1,156,train,train,3,69
2,218,test,test,2,59
3,218,test,train,2,48


In [69]:
import pandas as pd
from pathlib import Path

root = PROJECT_ROOT
manifest_dir = root / "data/processed/deep_learning/cnn_baseline"

paths = [
    manifest_dir / "train_cnn_manifest.csv",
    manifest_dir / "validation_cnn_manifest.csv",
    manifest_dir / "test_cnn_manifest.csv",
]

df = pd.concat(
    [pd.read_csv(path) for path in paths],
    ignore_index=True,
).drop_duplicates()

if "cnn_split" in df.columns and "current_split" not in df.columns:
    df = df.rename(columns={"cnn_split": "current_split"})

recording_membership = (
    df.groupby(["patient_id", "recording_id"])["official_split"]
    .agg(
        official_memberships="nunique",
        official_values=lambda s: ", ".join(
            sorted(s.dropna().astype(str).unique())
        ),
    )
    .reset_index()
)

mixed_recordings = recording_membership[
    recording_membership["official_memberships"] > 1
]

print("Recordings with mixed official membership:", len(mixed_recordings))
display(mixed_recordings)

print("\nOfficial membership by current split, at recording level:")
recording_rows = df.drop_duplicates(
    ["patient_id", "recording_id", "official_split", "current_split"]
)
display(pd.crosstab(
    recording_rows["official_split"],
    recording_rows["current_split"],
    margins=True,
))

Recordings with mixed official membership: 5


,patient_id,recording_id,official_memberships,official_values
108,156,2b3,2,"test, train"
109,156,5b3,2,"test, train"
110,156,8b3,2,"test, train"
240,218,1b1,2,"test, train"
241,218,1p1,2,"test, train"



Official membership by current split, at recording level:


current_split,test,train,validation,All
official_split,,,,
test,22,62,18,102
train,33,107,19,159
All,55,169,37,261


In [70]:

from pathlib import Path
import pandas as pd

root = PROJECT_ROOT
metadata_path = root / "data/interim/recordings.csv"

if not metadata_path.exists():
    raise FileNotFoundError(f"Metadata file not found: {metadata_path}")

recordings = pd.read_csv(metadata_path)

print("Metadata shape:", recordings.shape)
print("Columns:", recordings.columns.tolist())
display(recordings.head())

# Identify relevant columns without assuming exact column naming.
patient_col = next(
    (c for c in recordings.columns if c.lower() in {"patient_id", "patient"}),
    None,
)
recording_col = next(
    (c for c in recordings.columns if c.lower() in {"recording_id", "recording"}),
    None,
)
split_col = next(
    (c for c in recordings.columns if c.lower() == "official_split"),
    None,
)

print("\nDetected columns:")
print("patient:", patient_col)
print("recording:", recording_col)
print("official split:", split_col)

if not all([patient_col, recording_col, split_col]):
    print(
        "\nCould not identify all expected columns. "
        "Review the displayed column names before continuing."
    )
else:
    focus = recordings[
        recordings[patient_col].astype(str).isin(["156", "218"])
    ].copy()

    print("\nOriginal metadata rows for patients 156 and 218:")
    display(
        focus.sort_values([patient_col, recording_col, split_col])
    )

    print("\nDuplicate patient-recording keys:")
    duplicate_keys = recordings[
        recordings.duplicated(
            subset=[patient_col, recording_col], keep=False
        )
    ].sort_values([patient_col, recording_col])

    print("Rows involved:", len(duplicate_keys))
    display(duplicate_keys)

    print("\nOfficial split assignments per patient-recording key:")
    key_summary = (
        recordings.groupby([patient_col, recording_col], dropna=False)
        .agg(
            row_count=(split_col, "size"),
            split_count=(split_col, "nunique"),
            split_values=(
                split_col,
                lambda s: ", ".join(
                    sorted(s.dropna().astype(str).unique())
                ),
            ),
        )
        .reset_index()
    )

    display(
        key_summary[key_summary["split_count"] > 1]
        .sort_values([patient_col, recording_col])
    )


Metadata shape: (920, 7)
Columns: ['patient_id', 'recording_id', 'chest_location', 'acquisition_mode', 'equipment', 'wav_path', 'official_split']


,patient_id,recording_id,chest_location,acquisition_mode,equipment,wav_path,official_split
0,101,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
1,101,1b1,Pr,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
2,102,1b1,Ar,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
3,103,2b2,Ar,mc,LittC2SE,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,train
4,104,1b1,Al,sc,Litt3200,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test



Detected columns:
patient: patient_id
recording: recording_id
official split: official_split

Original metadata rows for patients 156 and 218:


,patient_id,recording_id,chest_location,acquisition_mode,equipment,wav_path,official_split
362,156,2b3,Al,mc,AKGC417L,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
365,156,2b3,Lr,mc,AKGC417L,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
366,156,2b3,Pl,mc,AKGC417L,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
363,156,2b3,Ar,mc,AKGC417L,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,train
364,156,2b3,Ll,mc,AKGC417L,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,train
367,156,2b3,Pr,mc,AKGC417L,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,train
369,156,5b3,Ar,mc,AKGC417L,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
370,156,5b3,Ll,mc,AKGC417L,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
372,156,5b3,Pl,mc,AKGC417L,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
368,156,5b3,Al,mc,AKGC417L,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,train



Duplicate patient-recording keys:
Rows involved: 837


,patient_id,recording_id,chest_location,acquisition_mode,equipment,wav_path,official_split
0,101,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
1,101,1b1,Pr,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
4,104,1b1,Al,sc,Litt3200,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
5,104,1b1,Ar,sc,Litt3200,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
6,104,1b1,Ll,sc,Litt3200,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
...,...,...,...,...,...,...,...
912,223,1b1,Pl,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
913,223,1b1,Pr,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
917,226,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,train
918,226,1b1,Ll,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,train



Official split assignments per patient-recording key:


,patient_id,recording_id,row_count,split_count,split_values
108,156,2b3,6,2,"test, train"
109,156,5b3,6,2,"test, train"
110,156,8b3,5,2,"test, train"
240,218,1b1,5,2,"test, train"
241,218,1p1,3,2,"test, train"


In [71]:

from pathlib import Path
import pandas as pd

root = PROJECT_ROOT
metadata_path = root / "data/interim/recordings.csv"
manifest_dir = root / "data/processed/deep_learning/cnn_baseline"

metadata = pd.read_csv(metadata_path)

manifest_paths = [
    manifest_dir / "train_cnn_manifest.csv",
    manifest_dir / "validation_cnn_manifest.csv",
    manifest_dir / "test_cnn_manifest.csv",
]

manifests = []
for path in manifest_paths:
    df = pd.read_csv(path)
    if "cnn_split" in df.columns and "current_split" not in df.columns:
        df = df.rename(columns={"cnn_split": "current_split"})
    manifests.append(df)

cycles = pd.concat(manifests, ignore_index=True)

print("Metadata rows:", len(metadata))
print("Manifest cycle rows:", len(cycles))
print("Metadata unique WAV paths:", metadata["wav_path"].nunique())

# Check whether WAV paths exist and whether the metadata path is duplicated.
metadata_paths = metadata["wav_path"].dropna().astype(str)
missing_wav_paths = [
    path for path in metadata_paths.unique()
    if not Path(path).exists()
]
print("Unique metadata WAV paths missing on disk:", len(missing_wav_paths))
if missing_wav_paths:
    print("First missing paths:", missing_wav_paths[:10])

# Show how many chest-location rows map to each patient/recording pair.
recording_summary = (
    metadata.groupby(["patient_id", "recording_id"], dropna=False)
    .agg(
        metadata_rows=("wav_path", "size"),
        unique_wav_paths=("wav_path", "nunique"),
        official_memberships=("official_split", "nunique"),
        official_splits=(
            "official_split",
            lambda s: ", ".join(sorted(s.dropna().astype(str).unique()))
        ),
    )
    .reset_index()
)

print("\nRecording IDs with multiple official memberships:")
display(
    recording_summary[recording_summary["official_memberships"] > 1]
)

# Inspect the manifest's available audio-path column.
print("\nManifest columns:", cycles.columns.tolist())

audio_col = next(
    (c for c in ["audio_path", "wav_path", "path"] if c in cycles.columns),
    None,
)

if audio_col is None:
    print("No recognized audio path column found in the cycle manifests.")
else:
    cycle_paths = cycles[audio_col].dropna().astype(str)
    print("Cycle rows with non-null audio paths:", len(cycle_paths))
    print("Unique cycle audio paths:", cycle_paths.nunique())

    # Resolve relative paths against the project root.
    def resolve_path(value):
        p = Path(value).expanduser()
        return p.resolve() if p.is_absolute() else (root / p).resolve()

    metadata_resolved = {
        resolve_path(p) for p in metadata_paths
    }
    cycle_resolved = cycle_paths.map(resolve_path)

    print(
        "Unique cycle paths present in original metadata:",
        cycle_resolved.isin(metadata_resolved).sum(),
        "of",
        len(cycle_resolved),
    )

    missing_from_metadata = sorted(
        set(cycle_resolved) - metadata_resolved
    )
    print(
        "Unique cycle paths absent from original metadata:",
        len(missing_from_metadata),
    )
    if missing_from_metadata:
        print("Examples:", [str(p) for p in missing_from_metadata[:10]])


Metadata rows: 920
Manifest cycle rows: 6898
Metadata unique WAV paths: 920
Unique metadata WAV paths missing on disk: 0

Recording IDs with multiple official memberships:


,patient_id,recording_id,metadata_rows,unique_wav_paths,official_memberships,official_splits
108,156,2b3,6,6,2,"test, train"
109,156,5b3,6,6,2,"test, train"
110,156,8b3,5,5,2,"test, train"
240,218,1b1,5,5,2,"test, train"
241,218,1p1,3,3,2,"test, train"



Manifest columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'current_split']
Cycle rows with non-null audio paths: 6898
Unique cycle audio paths: 6898
Unique cycle paths present in original metadata: 0 of 6898
Unique cycle paths absent from original metadata: 6898
Examples: ['/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/audio/cycles/101/1b1_Al_001.wav', '/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/audio/cycles/101/1b1_Al_002.wav', '/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/audio/cycles/101/1b1_Al_003.wav', '/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/audio/cycles/101/1b1_Al_004.wav', '/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/audio/cycles/101/1b1_Al_005.wav', '/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/audio/cycles/101/1b1_Al_006.wav', '/Users/vs/Sem 5/Respiratory-Disease-Classificat

In [72]:
from pathlib import Path
import pandas as pd

root = PROJECT_ROOT

manifest_dir = root / "data/processed/deep_learning/cnn_baseline"
manifest_files = [
    manifest_dir / "train_cnn_manifest.csv",
    manifest_dir / "validation_cnn_manifest.csv",
    manifest_dir / "test_cnn_manifest.csv",
]

manifest_frames = [pd.read_csv(path) for path in manifest_files]
cycles = pd.concat(manifest_frames, ignore_index=True)

audio_col = "audio_path"
cycle_paths = cycles[audio_col].dropna().astype(str)

# Check the actual cycle audio files on disk
exists = cycle_paths.map(lambda p: Path(p).exists())

print("Total cycle paths:", len(cycle_paths))
print("Unique cycle paths:", cycle_paths.nunique())
print("Existing cycle files:", int(exists.sum()))
print("Missing cycle files:", int((~exists).sum()))

if (~exists).any():
    print("\nFirst missing cycle paths:")
    display(cycle_paths.loc[~exists].head(20).to_frame("missing_audio_path"))
else:
    print("\nPASS: Every cycle path listed in the manifests exists on disk.")
    

Total cycle paths: 6898
Unique cycle paths: 6898
Existing cycle files: 6898
Missing cycle files: 0

PASS: Every cycle path listed in the manifests exists on disk.


In [74]:
import pandas as pd

# Normalize the split-column name in each manifest
manifest_frames = []

for path in manifest_files:
    df = pd.read_csv(path)

    if "current_split" not in df.columns:
        if "cnn_split" in df.columns:
            df = df.rename(columns={"cnn_split": "current_split"})
        else:
            raise ValueError(
                f"No recognized split column in {path.name}. "
                f"Columns: {df.columns.tolist()}"
            )

    manifest_frames.append(df)

cycles = pd.concat(manifest_frames, ignore_index=True)

print("Columns:", cycles.columns.tolist())
print("\nRows by split:")
display(
    cycles["current_split"]
    .value_counts(dropna=False)
    .rename_axis("split")
    .to_frame("rows")
)

Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'current_split']

Rows by split:


,rows
split,
train,4423
test,1561
validation,914


In [75]:
check_cols = ["patient_id", "recording_id", "audio_path"]

for col in check_cols:
    split_counts = cycles.groupby(col, dropna=False)["current_split"].nunique()
    conflicts = split_counts[split_counts > 1]

    print(
        f"\n{col}: identifiers appearing in multiple splits = "
        f"{len(conflicts)}"
    )

    if len(conflicts):
        display(
            cycles.loc[
                cycles[col].isin(conflicts.index),
                [col, "current_split"]
            ]
            .drop_duplicates()
            .sort_values([col, "current_split"])
            .head(20)
        )
    else:
        print(f"PASS: No {col} identifier crosses split boundaries.")


patient_id: identifiers appearing in multiple splits = 0
PASS: No patient_id identifier crosses split boundaries.

recording_id: identifiers appearing in multiple splits = 21


,recording_id,current_split
5337,1b1,test
0,1b1,train
4423,1b1,validation
5727,1b2,test
90,1b2,train
4536,1b2,validation
5739,1b3,test
98,1b3,train
4552,1b3,validation
5750,1b4,test



audio_path: identifiers appearing in multiple splits = 0
PASS: No audio_path identifier crosses split boundaries.


In [76]:
# Check the composite patient + recording identifier
recording_key = ["patient_id", "recording_id"]

recording_split_counts = (
    cycles.groupby(recording_key, dropna=False)["current_split"]
    .nunique()
)

recording_conflicts = recording_split_counts[
    recording_split_counts > 1
]

print(
    "Patient-recording combinations appearing in multiple splits:",
    len(recording_conflicts)
)

if len(recording_conflicts):
    conflict_keys = recording_conflicts.index.to_frame(index=False)

    display(
        cycles.merge(
            conflict_keys,
            on=recording_key,
            how="inner"
        )[
            recording_key + ["current_split", "audio_path"]
        ]
        .drop_duplicates()
        .sort_values(recording_key + ["current_split"])
        .head(30)
    )
else:
    print(
        "PASS: No patient-recording combination crosses "
        "train, validation, and test boundaries."
    )

Patient-recording combinations appearing in multiple splits: 0
PASS: No patient-recording combination crosses train, validation, and test boundaries.


In [77]:
import pandas as pd

# Count unique patients and cycle samples for each disease in each split
support = (
    cycles.groupby(
        ["current_split", "disease_label"],
        observed=False
    )
    .agg(
        cycles=("audio_path", "nunique"),
        patients=("patient_id", "nunique"),
        recordings=("recording_id", "nunique"),
    )
    .reset_index()
    .sort_values(["disease_label", "current_split"])
)

display(support)

print("\nTest-set disease support:")
display(
    support[support["current_split"] == "test"]
    .sort_values("patients")
    .reset_index(drop=True)
)

,current_split,disease_label,cycles,patients,recordings
6,train,0,6,1,1
0,test,1,20,2,2
7,train,1,76,4,3
14,validation,1,8,1,1
1,test,2,19,1,1
8,train,2,114,4,1
15,validation,2,27,1,1
2,test,3,1337,13,19
9,train,3,3713,41,32
16,validation,3,696,10,17



Test-set disease support:


,current_split,disease_label,cycles,patients,recordings
0,test,2,19,1,1
1,test,6,52,1,3
2,test,1,20,2,2
3,test,7,49,3,2
4,test,4,84,5,2
5,test,3,1337,13,19


In [78]:
import pandas as pd
from pathlib import Path

root = PROJECT_ROOT

mapping_path = (
    root / "data/processed/deep_learning/disease_class_mapping.csv"
)

mapping = pd.read_csv(mapping_path)
print("Saved class mapping:")
display(mapping)

support_check = (
    cycles.groupby(["current_split", "disease_label"])
    .agg(
        cycles=("audio_path", "nunique"),
        patients=("patient_id", "nunique"),
        recordings=("recording_id", "nunique"),
    )
    .reset_index()
)

print("\nTest-set support, sorted by class ID:")
display(
    support_check[support_check["current_split"] == "test"]
    .sort_values("disease_label")
    .reset_index(drop=True)
)

print("\nUnique patients per disease across the entire dataset:")
display(
    cycles.groupby("disease_label")
    .agg(
        total_cycles=("audio_path", "nunique"),
        total_patients=("patient_id", "nunique"),
    )
    .reset_index()
    .sort_values("disease_label")
)

Saved class mapping:


,disease_label,diagnosis
0,0,Asthma
1,1,Bronchiectasis
2,2,Bronchiolitis
3,3,COPD
4,4,Healthy
5,5,LRTI
6,6,Pneumonia
7,7,URTI



Test-set support, sorted by class ID:


,current_split,disease_label,cycles,patients,recordings
0,test,1,20,2,2
1,test,2,19,1,1
2,test,3,1337,13,19
3,test,4,84,5,2
4,test,6,52,1,3
5,test,7,49,3,2



Unique patients per disease across the entire dataset:


,disease_label,total_cycles,total_patients
0,0,6,1
1,1,104,7
2,2,160,6
3,3,5746,64
4,4,322,26
5,5,32,2
6,6,285,6
7,7,243,14


In [79]:
import pandas as pd

patient_class = (
    cycles[["patient_id", "disease_label"]]
    .drop_duplicates()
)

patient_support = (
    patient_class.groupby("disease_label")["patient_id"]
    .nunique()
    .reset_index(name="unique_patients")
)

patient_support = patient_support.merge(
    mapping[["disease_label", "diagnosis"]],
    on="disease_label",
    how="left",
)

patient_support["max_theoretical_folds"] = (
    patient_support["unique_patients"]
)

patient_support["assessment"] = patient_support[
    "unique_patients"
].map(
    lambda n: (
        "No independent held-out evaluation possible"
        if n == 1
        else "Extremely limited"
        if n <= 2
        else "Limited"
        if n <= 7
        else "More feasible"
    )
)

display(
    patient_support[
        [
            "diagnosis",
            "unique_patients",
            "max_theoretical_folds",
            "assessment",
        ]
    ].sort_values("unique_patients")
)

,diagnosis,unique_patients,max_theoretical_folds,assessment
0,Asthma,1,1,No independent held-out evaluation possible
5,LRTI,2,2,Extremely limited
2,Bronchiolitis,6,6,Limited
6,Pneumonia,6,6,Limited
1,Bronchiectasis,7,7,Limited
7,URTI,14,14,More feasible
4,Healthy,26,26,More feasible
3,COPD,64,64,More feasible


In [80]:
# Patient-level disease distribution in each existing split

patient_split_support = (
    cycles[["patient_id", "disease_label", "current_split"]]
    .drop_duplicates()
    .groupby(["current_split", "disease_label"])
    .agg(
        patients=("patient_id", "nunique")
    )
    .reset_index()
    .merge(
        mapping[["disease_label", "diagnosis"]],
        on="disease_label",
        how="left",
    )
)

support_pivot = patient_split_support.pivot(
    index="diagnosis",
    columns="current_split",
    values="patients",
).fillna(0).astype(int)

for split in ["train", "validation", "test"]:
    if split not in support_pivot.columns:
        support_pivot[split] = 0

support_pivot = support_pivot[["train", "validation", "test"]]
support_pivot["total"] = support_pivot.sum(axis=1)

display(support_pivot)

print("\nTotal unique patients:", cycles["patient_id"].nunique())
print(
    "Patients per split:",
    cycles.groupby("current_split")["patient_id"].nunique().to_dict()
)

current_split,train,validation,test,total
diagnosis,,,,
Asthma,1,0,0,1
Bronchiectasis,4,1,2,7
Bronchiolitis,4,1,1,6
COPD,41,10,13,64
Healthy,17,4,5,26
LRTI,2,0,0,2
Pneumonia,4,1,1,6
URTI,9,2,3,14



Total unique patients: 126
Patients per split: {'test': 25, 'train': 82, 'validation': 19}


In [81]:
# Audit the development data without touching the test set

development = cycles[
    cycles["current_split"].isin(["train", "validation"])
].copy()

development_patient_support = (
    development[["patient_id", "disease_label"]]
    .drop_duplicates()
    .groupby("disease_label")
    .agg(
        development_patients=("patient_id", "nunique")
    )
    .reset_index()
    .merge(
        mapping[["disease_label", "diagnosis"]],
        on="disease_label",
        how="left",
    )
)

development_cycle_support = (
    development.groupby("disease_label")
    .agg(development_cycles=("audio_path", "nunique"))
    .reset_index()
)

development_support = development_patient_support.merge(
    development_cycle_support,
    on="disease_label",
    how="left",
)

display(
    development_support[
        ["diagnosis", "development_patients", "development_cycles"]
    ].sort_values("development_patients")
)

print("Development patients:", development["patient_id"].nunique())
print("Development cycles:", development["audio_path"].nunique())
print("Test patients reserved:", cycles.loc[
    cycles["current_split"] == "test", "patient_id"
].nunique())

,diagnosis,development_patients,development_cycles
0,Asthma,1,6
5,LRTI,2,32
1,Bronchiectasis,5,84
2,Bronchiolitis,5,141
6,Pneumonia,5,233
7,URTI,11,194
4,Healthy,21,238
3,COPD,51,4409


Development patients: 101
Development cycles: 5337
Test patients reserved: 25


In [82]:
from sklearn.model_selection import StratifiedGroupKFold

# Use development data only; reserve the current test set.
development = cycles[
    cycles["current_split"].isin(["train", "validation"])
].copy()

# Each patient must have exactly one disease label.
patient_labels = development[
    ["patient_id", "disease_label"]
].drop_duplicates()

print("Patient-label rows:", len(patient_labels))
print("Unique development patients:", patient_labels["patient_id"].nunique())
print("Unique disease labels:", patient_labels["disease_label"].nunique())

# Check whether each patient has only one disease label.
labels_per_patient = patient_labels.groupby("patient_id")["disease_label"].nunique()
print("Patients with multiple disease labels:", int((labels_per_patient > 1).sum()))

# Summarize patient support before constructing folds.
display(
    patient_labels.groupby("disease_label")
    .agg(patients=("patient_id", "nunique"))
    .reset_index()
    .merge(mapping, on="disease_label", how="left")
    [["diagnosis", "patients"]]
    .sort_values("patients")
)

Patient-label rows: 101
Unique development patients: 101
Unique disease labels: 8
Patients with multiple disease labels: 0


,diagnosis,patients
0,Asthma,1
5,LRTI,2
1,Bronchiectasis,5
2,Bronchiolitis,5
6,Pneumonia,5
7,URTI,11
4,Healthy,21
3,COPD,51


In [83]:
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np
import pandas as pd

development = cycles[
    cycles["current_split"].isin(["train", "validation"])
].copy().reset_index(drop=True)

# Every patient must have one disease label.
patient_labels = (
    development[["patient_id", "disease_label"]]
    .drop_duplicates()
)

assert patient_labels["patient_id"].is_unique, (
    "A patient has multiple disease labels."
)

X = patient_labels[["patient_id"]]
y = patient_labels["disease_label"].to_numpy()
groups = patient_labels["patient_id"].to_numpy()

cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

fold_rows = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X, y, groups=groups), start=1
):
    train_patients = set(groups[train_idx])
    val_patients = set(groups[val_idx])

    assert train_patients.isdisjoint(val_patients)

    train_labels = patient_labels.iloc[train_idx]
    val_labels = patient_labels.iloc[val_idx]

    for _, row in mapping.iterrows():
        class_id = row["disease_label"]
        diagnosis = row["diagnosis"]

        fold_rows.append({
            "fold": fold,
            "diagnosis": diagnosis,
            "train_patients": int(
                (train_labels["disease_label"] == class_id).sum()
            ),
            "validation_patients": int(
                (val_labels["disease_label"] == class_id).sum()
            ),
        })

fold_support = pd.DataFrame(fold_rows)

print("Patient counts per fold:")
display(
    fold_support.groupby("fold")[
        ["train_patients", "validation_patients"]
    ].sum()
)

print("\nClass support by fold:")
display(
    fold_support.sort_values(["diagnosis", "fold"])
    .reset_index(drop=True)
)

print("\nPatient overlap between train and validation: 0 in every fold")

Patient counts per fold:


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:1036: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


,train_patients,validation_patients
fold,,
1,67,34
2,67,34
3,68,33



Class support by fold:


,fold,diagnosis,train_patients,validation_patients
0,1,Asthma,1,0
1,2,Asthma,0,1
2,3,Asthma,1,0
3,1,Bronchiectasis,3,2
4,2,Bronchiectasis,4,1
5,3,Bronchiectasis,3,2
6,1,Bronchiolitis,4,1
7,2,Bronchiolitis,3,2
8,3,Bronchiolitis,3,2
9,1,COPD,34,17



Patient overlap between train and validation: 0 in every fold


In [84]:
# Expand the patient-level folds to all development cycles.
# No model training and no changes to saved manifests.

from sklearn.model_selection import StratifiedGroupKFold
import pandas as pd

development = cycles[
    cycles["current_split"].isin(["train", "validation"])
].copy().reset_index(drop=True)

cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

fold_cycle_rows = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(
        development,
        development["disease_label"],
        groups=development["patient_id"],
    ),
    start=1,
):
    train = development.iloc[train_idx]
    val = development.iloc[val_idx]

    assert set(train["patient_id"]).isdisjoint(set(val["patient_id"]))

    for _, row in mapping.iterrows():
        class_id = row["disease_label"]
        diagnosis = row["diagnosis"]

        fold_cycle_rows.append({
            "fold": fold,
            "diagnosis": diagnosis,
            "train_patients": train.loc[
                train["disease_label"] == class_id, "patient_id"
            ].nunique(),
            "validation_patients": val.loc[
                val["disease_label"] == class_id, "patient_id"
            ].nunique(),
            "train_cycles": train.loc[
                train["disease_label"] == class_id, "audio_path"
            ].nunique(),
            "validation_cycles": val.loc[
                val["disease_label"] == class_id, "audio_path"
            ].nunique(),
        })

fold_cycle_support = pd.DataFrame(fold_cycle_rows)

display(
    fold_cycle_support.sort_values(["diagnosis", "fold"])
    .reset_index(drop=True)
)

,fold,diagnosis,train_patients,validation_patients,train_cycles,validation_cycles
0,1,Asthma,0,1,0,6
1,2,Asthma,1,0,6,0
2,3,Asthma,1,0,6,0
3,1,Bronchiectasis,3,2,60,24
4,2,Bronchiectasis,4,1,43,41
5,3,Bronchiectasis,3,2,65,19
6,1,Bronchiolitis,3,2,100,41
7,2,Bronchiolitis,4,1,90,51
8,3,Bronchiolitis,3,2,92,49
9,1,COPD,35,16,2938,1471


In [85]:
# Verify the exact fold assignments used in the latest feasibility check.

fold_assignments = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(
        development,
        development["disease_label"],
        groups=development["patient_id"],
    ),
    start=1,
):
    train_patients = set(development.iloc[train_idx]["patient_id"])
    val_patients = set(development.iloc[val_idx]["patient_id"])

    assert train_patients.isdisjoint(val_patients)

    fold_assignments.extend(
        {
            "fold": fold,
            "patient_id": patient_id,
            "disease_label": disease_label,
        }
        for patient_id, disease_label in (
            development.iloc[val_idx][
                ["patient_id", "disease_label"]
            ].drop_duplicates().itertuples(index=False, name=None)
        )
    )

fold_assignments = pd.DataFrame(fold_assignments)

print("Total validation patient-fold assignments:", len(fold_assignments))
print("Unique patients assigned to validation:", fold_assignments["patient_id"].nunique())
print("Patients assigned to more than one validation fold:",
      int((fold_assignments.groupby("patient_id")["fold"].nunique() > 1).sum()))

print("\nValidation patients per disease and fold:")
display(
    fold_assignments.groupby(["fold", "disease_label"])
    .agg(validation_patients=("patient_id", "nunique"))
    .reset_index()
    .merge(mapping, on="disease_label", how="left")
    [["fold", "diagnosis", "validation_patients"]]
    .sort_values(["diagnosis", "fold"])
    .reset_index(drop=True)
)

Total validation patient-fold assignments: 101
Unique patients assigned to validation: 101
Patients assigned to more than one validation fold: 0

Validation patients per disease and fold:


,fold,diagnosis,validation_patients
0,1,Asthma,1
1,1,Bronchiectasis,2
2,2,Bronchiectasis,1
3,3,Bronchiectasis,2
4,1,Bronchiolitis,2
5,2,Bronchiolitis,1
6,3,Bronchiolitis,2
7,1,COPD,16
8,2,COPD,17
9,3,COPD,18


In [87]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# Keep the existing test set completely separate.
development = cycles[
    cycles["current_split"].isin(["train", "validation"])
].copy()

# One row per patient; every patient must have exactly one disease label.
patient_table = (
    development[["patient_id", "disease_label"]]
    .drop_duplicates()
    .sort_values(["patient_id"])
    .reset_index(drop=True)
)

assert patient_table["patient_id"].is_unique, (
    "A patient has multiple disease labels."
)

# Assign folds at the patient level.
# Asthma has one patient, so stratification across all classes is impossible.
# We use a reproducible patient-level fold assignment and audit class support.
patient_table["fold"] = -1

rare_classes = patient_table["disease_label"].value_counts()
stratifiable_classes = rare_classes[rare_classes >= 3].index

stratifiable_mask = patient_table["disease_label"].isin(stratifiable_classes)
stratifiable_patients = patient_table.loc[stratifiable_mask].copy()
unstratifiable_patients = patient_table.loc[~stratifiable_mask].copy()

# Stratify classes with at least three independent patients.
skf = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

for fold, (_, val_idx) in enumerate(
    skf.split(
        stratifiable_patients[["patient_id"]],
        stratifiable_patients["disease_label"],
    ),
    start=1,
):
    fold_patient_ids = stratifiable_patients.iloc[val_idx]["patient_id"]
    patient_table.loc[
        patient_table["patient_id"].isin(fold_patient_ids), "fold"
    ] = fold

# Assign patients from classes with fewer than three patients
# to the currently smallest fold, deterministically.
fold_sizes = patient_table.loc[
    patient_table["fold"] > 0
].groupby("fold")["patient_id"].nunique().to_dict()

for _, patient in unstratifiable_patients.sort_values(
    ["disease_label", "patient_id"]
).iterrows():
    target_fold = min(
        [1, 2, 3],
        key=lambda f: (fold_sizes.get(f, 0), f),
    )
    patient_table.loc[
        patient_table["patient_id"] == patient["patient_id"], "fold"
    ] = target_fold
    fold_sizes[target_fold] = fold_sizes.get(target_fold, 0) + 1

print("Patients:", len(patient_table))
print("Fold assignments:")
display(
    patient_table.groupby("fold")
    .agg(patients=("patient_id", "nunique"))
)

print("\nPatient-level class support per fold:")
display(
    patient_table.groupby(["fold", "disease_label"])
    .agg(patients=("patient_id", "nunique"))
    .reset_index()
    .merge(mapping, on="disease_label", how="left")
    [["fold", "diagnosis", "patients"]]
    .sort_values(["diagnosis", "fold"])
    .reset_index(drop=True)
)

assert patient_table["fold"].between(1, 3).all()
assert patient_table["patient_id"].nunique() == 101
assert patient_table["fold"].notna().all()

print("\nPASS: Every development patient has exactly one fold assignment.")

Patients: 101
Fold assignments:


,patients
fold,
1,34
2,34
3,33



Patient-level class support per fold:


,fold,diagnosis,patients
0,3,Asthma,1
1,1,Bronchiectasis,2
2,2,Bronchiectasis,1
3,3,Bronchiectasis,2
4,1,Bronchiolitis,2
5,2,Bronchiolitis,2
6,3,Bronchiolitis,1
7,1,COPD,17
8,2,COPD,17
9,3,COPD,17



PASS: Every development patient has exactly one fold assignment.


In [88]:
# Expand patient-level folds to the cycle-level development data.
# This creates an in-memory table only; no files are modified.

development = cycles[
    cycles["current_split"].isin(["train", "validation"])
].copy()

folded_development = development.merge(
    patient_table[["patient_id", "fold"]],
    on="patient_id",
    how="left",
    validate="many_to_one",
)

print("Development cycles:", len(development))
print("Folded development cycles:", len(folded_development))
print("Cycles without a fold:", int(folded_development["fold"].isna().sum()))
print("Unique patients:", folded_development["patient_id"].nunique())

print("\nCycle counts per validation fold:")
display(
    folded_development.groupby("fold")
    .agg(
        cycles=("audio_path", "nunique"),
        patients=("patient_id", "nunique"),
    )
)

# Verify that each patient maps to exactly one fold.
patient_fold_counts = (
    folded_development.groupby("patient_id")["fold"].nunique()
)

print(
    "\nPatients assigned to multiple folds:",
    int((patient_fold_counts > 1).sum()),
)

assert folded_development["fold"].notna().all()
assert len(folded_development) == len(development)
assert (patient_fold_counts == 1).all()

print("\nPASS: Cycle rows inherit a single patient-level fold.")

Development cycles: 5337
Folded development cycles: 5337
Cycles without a fold: 0
Unique patients: 101

Cycle counts per validation fold:


,cycles,patients
fold,,
1,1395,34
2,2011,34
3,1931,33



Patients assigned to multiple folds: 0

PASS: Cycle rows inherit a single patient-level fold.


In [89]:
from pathlib import Path

root = PROJECT_ROOT

candidates = [
    root / "Notebooks/09_classical_ml.ipynb",
    root / "Notebooks/10_deep_learning_data_preparation.ipynb",
    root / "Notebooks/11_cnn_baseline.ipynb",
    root / "Notebooks/12_multifeature_cnn.ipynb",
    root / "Notebooks 2/02_Leakage_Safe_Preprocessing.ipynb",
    root / "Notebooks 2/03_Multidataset_Phenotype_Pretraining_ICBHI_Disease.ipynb",
    root / "Notebooks 2/04_ICBHI_Disease_Transfer_Learning.ipynb",
]

for path in candidates:
    print(f"{'FOUND' if path.exists() else 'MISSING'}: {path.relative_to(root)}")

normalization_path = (
    root
    / "data/processed/deep_learning/multifeature_cnn/"
      "multifeature_train_normalization.csv"
)

print("\nTraining normalization artifact exists:", normalization_path.exists())

if normalization_path.exists():
    import pandas as pd
    display(pd.read_csv(normalization_path))

FOUND: Notebooks/09_classical_ml.ipynb
FOUND: Notebooks/10_deep_learning_data_preparation.ipynb
FOUND: Notebooks/11_cnn_baseline.ipynb
FOUND: Notebooks/12_multifeature_cnn.ipynb
FOUND: Notebooks 2/02_Leakage_Safe_Preprocessing.ipynb
FOUND: Notebooks 2/03_Multidataset_Phenotype_Pretraining_ICBHI_Disease.ipynb
FOUND: Notebooks 2/04_ICBHI_Disease_Transfer_Learning.ipynb

Training normalization artifact exists: True


,feature,count,mean,std
0,logmel,44442304,-66.230481,17.661886
1,mfcc,9027343,-22.509034,137.274416
2,chroma,8332932,0.385656,0.389059


In [90]:
import json
from pathlib import Path

root = PROJECT_ROOT

notebook_paths = [
    root / "Notebooks/09_classical_ml.ipynb",
    root / "Notebooks/10_deep_learning_data_preparation.ipynb",
    root / "Notebooks/11_cnn_baseline.ipynb",
    root / "Notebooks/12_multifeature_cnn.ipynb",
    root / "Notebooks 2/02_Leakage_Safe_Preprocessing.ipynb",
    root / "Notebooks 2/03_Multidataset_Phenotype_Pretraining_ICBHI_Disease.ipynb",
    root / "Notebooks 2/04_ICBHI_Disease_Transfer_Learning.ipynb",
]

keywords = [
    "StandardScaler",
    "MinMaxScaler",
    "fit_transform",
    "normalization",
    "feature_selection",
    "SelectKBest",
    "class_weight",
    "compute_class_weight",
    "train_test_split",
    "multifeature_train_normalization",
]

for path in notebook_paths:
    if not path.exists():
        continue

    notebook = json.loads(path.read_text(encoding="utf-8"))
    matches = []

    for cell_index, cell in enumerate(notebook.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        found = [word for word in keywords if word.lower() in source.lower()]

        if found:
            matches.append((cell_index, found, source))

    print(f"\n{'=' * 80}\nNOTEBOOK: {path.relative_to(root)}")

    for cell_index, found, source in matches:
        print(f"\n--- Cell {cell_index}; matches: {', '.join(found)} ---")
        print(source[:3500])


NOTEBOOK: Notebooks/09_classical_ml.ipynb

--- Cell 2; matches: StandardScaler ---
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

print("Imports completed.")

--- Cell 3; matches: feature_selection ---
PROJECT_ROOT = Path("/Users/vs/Sem 5/Respiratory-Disease-Classification")

FEATURES_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "features"
    / "respiratory_cycle_features.csv"
)

SELECTED_FEATURES_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "feature_selection"
    / "selected_features.csv"
)

RECORDINGS_PAT

In [91]:
import json
from pathlib import Path

root = PROJECT_ROOT

# Locate the feature-selection notebook without assuming capitalization.
candidates = [
    p for p in root.rglob("*.ipynb")
    if "feature_selection" in p.stem.lower()
]

print("Feature-selection notebooks found:")
for p in candidates:
    print(" -", p.relative_to(root))

keywords = [
    "SelectKBest",
    "mutual_info",
    "f_classif",
    "feature_importances_",
    "selected_features",
    "train_test_split",
    "official_split",
    "fit_transform",
    "get_support",
]

for path in candidates:
    notebook = json.loads(path.read_text(encoding="utf-8"))

    print(f"\n{'=' * 80}")
    print("NOTEBOOK:", path.relative_to(root))

    for i, cell in enumerate(notebook.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        matched = [
            word for word in keywords
            if word.lower() in source.lower()
        ]

        if matched:
            print(f"\n--- Cell {i}; matches: {', '.join(matched)} ---")
            print(source[:5000])

Feature-selection notebooks found:
 - notebooks/07_feature_selection.ipynb

NOTEBOOK: notebooks/07_feature_selection.ipynb

--- Cell 1; matches: mutual_info, f_classif ---
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_selection import (
    f_classif,
    mutual_info_classif
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.preprocessing import LabelEncoder

print("Libraries imported successfully.")

--- Cell 7; matches: official_split ---
print("Official split distribution:")

display(
    recordings_df["official_split"]
    .value_counts()
)

--- Cell 8; matches: official_split ---
# ============================================================
# Attach Official Train/Test Split to Cycle Features
# ============================================================

split_columns = [
    "patient_id",
    "recording_id",
    "chest_location",
    "official_split"
]

features_with_split = features_df.m

In [92]:
import json
from pathlib import Path

path = PROJECT_ROOT / "notebooks" / "07_feature_selection.ipynb"
nb = json.loads(path.read_text(encoding="utf-8"))

keywords = [
    "feature_columns",
    "train_correlation_matrix",
    "corr(",
    "correlation",
    "features_df",
]

for i, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    if any(word.lower() in source.lower() for word in keywords):
        print(f"\n{'=' * 70}\nCELL {i}\n{'=' * 70}")
        print(source[:6000])


CELL 2
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA_DIR = (
    PROJECT_ROOT / "data" / "processed"
)

FEATURES_DIR = (
    PROCESSED_DATA_DIR / "features"
)

ANALYSIS_DIR = (
    PROCESSED_DATA_DIR / "analysis"
)

features_path = (
    FEATURES_DIR /
    "respiratory_cycle_features.csv"
)

features_df = pd.read_csv(features_path)

print("Feature dataset shape:", features_df.shape)

CELL 3
metadata_columns = [
    "patient_id",
    "recording_id",
    "chest_location",
    "acquisition_mode",
    "equipment",
    "cycle_number",
    "start_time",
    "end_time",
    "sound_label",
    "crackles",
    "wheezes",
    "diagnosis",
    "processed_path",
]

feature_columns = [
    col
    for col in features_df.columns
    if col not in metadata_columns
]

print("Number of features:", len(feature_columns))

CELL 8
# ============================================================
# Attach Official Train/Test Split to Cycle Features
# ========================================================

In [93]:
import json
from pathlib import Path

path = PROJECT_ROOT / "Notebooks" / "09_classical_ml.ipynb"

if not path.exists():
    path = PROJECT_ROOT / "notebooks" / "09_classical_ml.ipynb"

print("Notebook found:", path.exists())

if path.exists():
    nb = json.loads(path.read_text(encoding="utf-8"))

    keywords = [
        "selected_features",
        "train_test_split",
        "official_split",
        "patient_id",
        "StandardScaler",
        "LogisticRegression",
        "SVC(",
        "RandomForestClassifier",
        "classification_report",
        "fit(",
    ]

    for i, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        matches = [
            word for word in keywords
            if word.lower() in source.lower()
        ]

        if matches:
            print(f"\n{'=' * 70}")
            print(f"CELL {i} | Matches: {', '.join(matches)}")
            print("=" * 70)
            print(source[:4500])

Notebook found: True

CELL 1 | Matches: selected_features

required_objects = [
    "PROJECT_ROOT",
    "train_df",
    "test_df",
    "selected_features_df",
    "selected_feature_names",
    "X_train",
    "X_test",
    "scaler",
    "X_train_scaled",
    "X_test_scaled",
    "label_encoder",
    "y_train",
    "y_test",
    "logistic_model",
    "svm_model",
    "rf_model",
]

status = []

for name in required_objects:
    exists = name in globals()
    obj = globals().get(name) if exists else None

    status.append({
        "variable": name,
        "available": exists,
        "type": type(obj).__name__ if exists else "NOT AVAILABLE",
    })

import pandas as pd
display(pd.DataFrame(status))


CELL 2 | Matches: StandardScaler, LogisticRegression, RandomForestClassifier, classification_report
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model imp

In [94]:
from pathlib import Path
import pandas as pd

root = PROJECT_ROOT

manifest_paths = {
    "train": root / "data/processed/deep_learning/cnn_baseline/train_cnn_manifest.csv",
    "validation": root / "data/processed/deep_learning/cnn_baseline/validation_cnn_manifest.csv",
    "test": root / "data/processed/deep_learning/cnn_baseline/test_cnn_manifest.csv",
}

frames = {}

for split, path in manifest_paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {split} manifest: {path}")

    df = pd.read_csv(path)
    frames[split] = df

    print(f"\n{'=' * 65}")
    print(f"{split.upper()} — {len(df):,} cycles")
    print("Columns:", df.columns.tolist())

    patient_col = "patient_id"
    diagnosis_col = "diagnosis"

    print("Unique patients:", df[patient_col].nunique())
    print("\nCycles per diagnosis:")
    print(df[diagnosis_col].value_counts().sort_index())

    print("\nPatients per diagnosis:")
    print(
        df.groupby(diagnosis_col)[patient_col]
        .nunique()
        .sort_index()
    )

# Verify patient-level separation across partitions.
print(f"\n{'=' * 65}")
print("PATIENT OVERLAP CHECK")

for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    pa = set(frames[a]["patient_id"].astype(str))
    pb = set(frames[b]["patient_id"].astype(str))
    print(f"{a} vs {b}: {len(pa & pb)} overlapping patients")


TRAIN — 4,423 cycles
Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split']
Unique patients: 82

Cycles per diagnosis:
diagnosis
Asthma               6
Bronchiectasis      76
Bronchiolitis      114
COPD              3713
Healthy            194
LRTI                32
Pneumonia          121
URTI               167
Name: count, dtype: int64

Patients per diagnosis:
diagnosis
Asthma             1
Bronchiectasis     4
Bronchiolitis      4
COPD              41
Healthy           17
LRTI               2
Pneumonia          4
URTI               9
Name: patient_id, dtype: int64

VALIDATION — 914 cycles
Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split']
Unique patients: 19

Cycles per diagnosis:
diagnosis
Bronchiectasis      8
Bronchiolitis      27
COPD              696
Healthy            44
Pneumonia         112
URTI               27
Name:

In [95]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# Combine only the existing development partitions.
development_df = pd.concat(
    [frames["train"], frames["validation"]],
    ignore_index=True
).copy()

reserved_test_df = frames["test"].copy()

# Each patient must have exactly one disease label.
patient_table = (
    development_df.groupby("patient_id")
    .agg(
        diagnosis_count=("diagnosis", "nunique"),
        diagnosis=("diagnosis", "first"),
        cycle_count=("diagnosis", "size"),
    )
    .reset_index()
)

assert (patient_table["diagnosis_count"] == 1).all(), (
    "At least one patient has multiple diagnoses."
)

assert development_df["patient_id"].nunique() == 101
assert reserved_test_df["patient_id"].nunique() == 25
assert set(development_df["patient_id"]).isdisjoint(
    set(reserved_test_df["patient_id"])
)

# Assign each patient to exactly one validation fold.
cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

patient_table["fold"] = -1

for fold, (_, val_idx) in enumerate(
    cv.split(
        X=patient_table,
        y=patient_table["diagnosis"],
        groups=patient_table["patient_id"],
    ),
    start=1,
):
    patient_table.loc[val_idx, "fold"] = fold

assert patient_table["fold"].between(1, 3).all()
assert patient_table["patient_id"].is_unique

# Attach the patient fold to every development cycle.
folded_development = development_df.merge(
    patient_table[["patient_id", "fold"]],
    on="patient_id",
    how="left",
    validate="many_to_one",
)

assert len(folded_development) == len(development_df)
assert folded_development["fold"].notna().all()

print("Development patients:", patient_table["patient_id"].nunique())
print("Development cycles:", len(folded_development))
print("Reserved test patients:", reserved_test_df["patient_id"].nunique())
print("Reserved test cycles:", len(reserved_test_df))
print("\nPatients per validation fold:")
print(patient_table.groupby("fold")["patient_id"].nunique())
print("\nCycles per validation fold:")
print(folded_development.groupby("fold").size())
print("\nPatients per disease and fold:")
print(pd.crosstab(patient_table["diagnosis"], patient_table["fold"]))
print("\nCycles per disease and fold:")
print(pd.crosstab(folded_development["diagnosis"], folded_development["fold"]))

Development patients: 101
Development cycles: 5337
Reserved test patients: 25
Reserved test cycles: 1561

Patients per validation fold:
fold
1    34
2    34
3    33
Name: patient_id, dtype: int64

Cycles per validation fold:
fold
1    1948
2    1262
3    2127
dtype: int64

Patients per disease and fold:
fold             1   2   3
diagnosis                 
Asthma           0   1   0
Bronchiectasis   2   1   2
Bronchiolitis    1   2   2
COPD            17  17  17
Healthy          7   7   7
LRTI             1   1   0
Pneumonia        2   2   1
URTI             4   3   4

Cycles per disease and fold:
fold               1     2     3
diagnosis                       
Asthma             0     6     0
Bronchiectasis    49    17    18
Bronchiolitis     27    35    79
COPD            1506  1025  1878
Healthy           73    77    88
LRTI               8    24     0
Pneumonia        152    57    24
URTI             133    21    40


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:1036: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


In [96]:
# Compare the current grouped folds with a patient-level stratified assignment.
# One row per patient is used, so all cycles belonging to a patient
# inherit that patient's fold.

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

patient_table_comparison = patient_table[
    ["patient_id", "diagnosis", "cycle_count"]
].copy().reset_index(drop=True)

# Classes with fewer than three patients cannot be represented
# in all three validation folds through ordinary stratification.
class_support = patient_table_comparison["diagnosis"].value_counts()
regular_classes = class_support[class_support >= 3].index

regular_mask = patient_table_comparison["diagnosis"].isin(
    regular_classes
)

regular = patient_table_comparison.loc[regular_mask].copy()
rare = patient_table_comparison.loc[~regular_mask].copy()

skf = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

patient_table_comparison["alternative_fold"] = 0

for fold, (_, val_idx) in enumerate(
    skf.split(
        regular,
        regular["diagnosis"]
    ),
    start=1,
):
    patient_table_comparison.loc[
        regular.index[val_idx],
        "alternative_fold"
    ] = fold

# Place rare-class patients into folds with the fewest patients so far.
fold_sizes = {
    fold: int(
        (patient_table_comparison["alternative_fold"] == fold).sum()
    )
    for fold in (1, 2, 3)
}

for idx, row in rare.sort_values(
    ["cycle_count", "patient_id"]
).iterrows():
    target_fold = min(fold_sizes, key=lambda f: (fold_sizes[f], f))
    patient_table_comparison.loc[idx, "alternative_fold"] = target_fold
    fold_sizes[target_fold] += 1

print("Alternative patients per fold:")
print(
    patient_table_comparison.groupby("alternative_fold")["patient_id"]
    .nunique()
)

print("\nAlternative patients per disease and fold:")
print(
    pd.crosstab(
        patient_table_comparison["diagnosis"],
        patient_table_comparison["alternative_fold"]
    )
)

print("\nCurrent grouped-fold patients per disease and fold:")
print(
    pd.crosstab(
        patient_table["diagnosis"],
        patient_table["fold"]
    )
)

print("\nRare-class patients:")
print(
    patient_table_comparison[
        patient_table_comparison["diagnosis"].isin(
            ["Asthma", "LRTI"]
        )
    ][["patient_id", "diagnosis", "cycle_count", "alternative_fold"]]
)

Alternative patients per fold:
alternative_fold
1    34
2    34
3    33
Name: patient_id, dtype: int64

Alternative patients per disease and fold:
alternative_fold   1   2   3
diagnosis                   
Asthma             0   0   1
Bronchiectasis     2   1   2
Bronchiolitis      2   2   1
COPD              17  17  17
Healthy            7   7   7
LRTI               1   1   0
Pneumonia          1   2   2
URTI               4   4   3

Current grouped-fold patients per disease and fold:
fold             1   2   3
diagnosis                 
Asthma           0   1   0
Bronchiectasis   2   1   2
Bronchiolitis    1   2   2
COPD            17  17  17
Healthy          7   7   7
LRTI             1   1   0
Pneumonia        2   2   1
URTI             4   3   4

Rare-class patients:
    patient_id diagnosis  cycle_count  alternative_fold
1          103    Asthma            6                 3
4          108      LRTI            8                 1
11         115      LRTI           24             

In [97]:
import pandas as pd

# Freeze the current grouped fold assignment.
final_patient_folds = patient_table[
    ["patient_id", "diagnosis", "fold"]
].copy()

final_patient_folds["fold"] = final_patient_folds["fold"].astype(int)

# Attach fold IDs to every development cycle.
final_folded_development = development_df.merge(
    final_patient_folds[["patient_id", "fold"]],
    on="patient_id",
    how="left",
    validate="many_to_one",
)

# Integrity checks.
assert final_patient_folds["patient_id"].is_unique
assert final_patient_folds["fold"].isin([1, 2, 3]).all()
assert final_folded_development["fold"].notna().all()
assert len(final_folded_development) == len(development_df)

assert set(final_folded_development["patient_id"]).isdisjoint(
    set(reserved_test_df["patient_id"])
)

# Every patient must map to exactly one fold.
assert (
    final_folded_development.groupby("patient_id")["fold"].nunique().max()
    == 1
)

print("FOLD ASSIGNMENT VALIDATION PASSED")
print("Development patients:", final_patient_folds["patient_id"].nunique())
print("Development cycles:", len(final_folded_development))
print("\nPatients per fold:")
print(final_patient_folds.groupby("fold")["patient_id"].nunique())
print("\nCycles per fold:")
print(final_folded_development.groupby("fold").size())
print("\nPatients per disease and fold:")
print(pd.crosstab(
    final_patient_folds["diagnosis"],
    final_patient_folds["fold"]
))

FOLD ASSIGNMENT VALIDATION PASSED
Development patients: 101
Development cycles: 5337

Patients per fold:
fold
1    34
2    34
3    33
Name: patient_id, dtype: int64

Cycles per fold:
fold
1    1948
2    1262
3    2127
dtype: int64

Patients per disease and fold:
fold             1   2   3
diagnosis                 
Asthma           0   1   0
Bronchiectasis   2   1   2
Bronchiolitis    1   2   2
COPD            17  17  17
Healthy          7   7   7
LRTI             1   1   0
Pneumonia        2   2   1
URTI             4   3   4


In [98]:
import numpy as np
import pandas as pd

# Use the verified in-memory objects.
cv_data = final_folded_development.copy()
cv_test_data = reserved_test_df.copy()

# Record the original feature list for historical comparison only.
selected_features_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "feature_selection"
    / "selected_features.csv"
)

original_selected_features = pd.read_csv(
    selected_features_path
)["feature"].tolist()

metadata_columns = [
    "patient_id",
    "recording_id",
    "chest_location",
    "diagnosis",
    "disease_label",
    "official_split",
    "audio_path",
    "cnn_split",
    "fold",
]

candidate_features = [
    col for col in cv_data.columns
    if col not in metadata_columns
]

# Check that features are numeric.
non_numeric = cv_data[candidate_features].select_dtypes(
    exclude=[np.number]
).columns.tolist()

print("Development rows:", len(cv_data))
print("Development patients:", cv_data["patient_id"].nunique())
print("Reserved test rows:", len(cv_test_data))
print("Reserved test patients:", cv_test_data["patient_id"].nunique())
print("Candidate feature count:", len(candidate_features))
print("Original selected feature count:", len(original_selected_features))
print("Non-numeric candidate features:", non_numeric)

print("\nMissing values in candidate features:",
      int(cv_data[candidate_features].isna().sum().sum()))

print("Infinite values in candidate features:",
      int(np.isinf(
          cv_data[candidate_features].to_numpy(dtype=float)
      ).sum()))

print("\nDisease labels:")
print(sorted(cv_data["diagnosis"].dropna().unique()))

# Verify fold separation.
for fold in (1, 2, 3):
    train_patients = set(
        cv_data.loc[cv_data["fold"] != fold, "patient_id"]
    )
    validation_patients = set(
        cv_data.loc[cv_data["fold"] == fold, "patient_id"]
    )

    assert train_patients.isdisjoint(validation_patients)
    assert validation_patients.isdisjoint(
        set(cv_test_data["patient_id"])
    )

print("\nAll fold separation checks passed.")

Development rows: 5337
Development patients: 101
Reserved test rows: 1561
Reserved test patients: 25
Candidate feature count: 0
Original selected feature count: 33
Non-numeric candidate features: []

Missing values in candidate features: 0
Infinite values in candidate features: 0

Disease labels:
['Asthma', 'Bronchiectasis', 'Bronchiolitis', 'COPD', 'Healthy', 'LRTI', 'Pneumonia', 'URTI']

All fold separation checks passed.


In [99]:
import pandas as pd
from pathlib import Path

features_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "features"
    / "respiratory_cycle_features.csv"
)

features_df = pd.read_csv(features_path, nrows=5)

print("Feature file:", features_path)
print("File exists:", features_path.exists())
print("Columns:", len(features_df.columns))
print("\nAll columns:")
for col in features_df.columns:
    print(f" - {col}")

metadata_columns = [
    "patient_id",
    "recording_id",
    "chest_location",
    "acquisition_mode",
    "equipment",
    "cycle_number",
    "start_time",
    "end_time",
    "sound_label",
    "crackles",
    "wheezes",
    "diagnosis",
    "processed_path",
]

feature_columns = [
    col for col in features_df.columns
    if col not in metadata_columns
]

print("\nCandidate numerical feature columns:", len(feature_columns))
print(feature_columns[:20])

Feature file: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/features/respiratory_cycle_features.csv
File exists: True
Columns: 52

All columns:
 - patient_id
 - recording_id
 - chest_location
 - acquisition_mode
 - equipment
 - cycle_number
 - start_time
 - end_time
 - sound_label
 - crackles
 - wheezes
 - diagnosis
 - processed_path
 - mean
 - std
 - variance
 - rms
 - max
 - min
 - peak_to_peak
 - median
 - skewness
 - kurtosis
 - energy
 - zero_crossing_rate
 - spectral_centroid
 - spectral_bandwidth
 - spectral_rolloff
 - spectral_flatness
 - spectral_energy
 - dominant_frequency
 - spectral_entropy
 - band_energy_0_500
 - band_energy_500_1000
 - band_energy_1000_1500
 - band_energy_1500_2000
 - mfcc_1_mean
 - mfcc_2_mean
 - mfcc_3_mean
 - mfcc_4_mean
 - mfcc_5_mean
 - mfcc_6_mean
 - mfcc_7_mean
 - mfcc_8_mean
 - mfcc_9_mean
 - mfcc_10_mean
 - mfcc_11_mean
 - mfcc_12_mean
 - mfcc_13_mean
 - wavelet_energy
 - wavelet_entropy
 - wavelet_std

Candidate numerical fe

In [100]:
import pandas as pd
import numpy as np

features_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "features"
    / "respiratory_cycle_features.csv"
)

all_features = pd.read_csv(features_path)

# Use the cycle number if the manifest has it; otherwise inspect the
# available identifiers before attempting a less specific join.
join_keys = [
    "patient_id",
    "recording_id",
    "chest_location",
]

if "cycle_number" in cv_data.columns and "cycle_number" in all_features.columns:
    join_keys.append("cycle_number")

print("Join keys:", join_keys)
print("Manifest rows:", len(cv_data))
print("Feature-table rows:", len(all_features))

print("\nDuplicate key rows in manifests:",
      int(cv_data.duplicated(join_keys).sum()))
print("Duplicate key rows in feature table:",
      int(all_features.duplicated(join_keys).sum()))

# Verify key uniqueness before joining.
if cv_data.duplicated(join_keys).any():
    print("\nDuplicate manifest examples:")
    display(cv_data.loc[
        cv_data.duplicated(join_keys, keep=False),
        join_keys + ["diagnosis", "audio_path", "fold"]
    ].head(10))

if all_features.duplicated(join_keys).any():
    print("\nDuplicate feature-table examples:")
    display(all_features.loc[
        all_features.duplicated(join_keys, keep=False),
        join_keys + ["diagnosis", "cycle_number", "processed_path"]
    ].head(10))

# Do not proceed with a many-to-many or ambiguous join.
if cv_data.duplicated(join_keys).any() or all_features.duplicated(join_keys).any():
    print("\nSTOP: join keys are not unique. We need to inspect the duplicates.")
else:
    feature_cols = [
        c for c in all_features.columns
        if c not in [
            "patient_id", "recording_id", "chest_location",
            "acquisition_mode", "equipment", "cycle_number",
            "start_time", "end_time", "sound_label",
            "crackles", "wheezes", "diagnosis", "processed_path"
        ]
    ]

    joined = cv_data.merge(
        all_features[join_keys + feature_cols + ["diagnosis"]],
        on=join_keys,
        how="left",
        suffixes=("_manifest", "_features"),
        validate="one_to_one",
        indicator=True,
    )

    print("\nJoin result rows:", len(joined))
    print("Join status:")
    print(joined["_merge"].value_counts(dropna=False))

    if "diagnosis_manifest" in joined.columns:
        matched_labels = joined["_merge"].eq("both")
        label_mismatch = (
            joined.loc[matched_labels, "diagnosis_manifest"].astype(str)
            != joined.loc[matched_labels, "diagnosis_features"].astype(str)
        )
        print("Diagnosis mismatches among matched rows:", int(label_mismatch.sum()))

    print("Numerical feature columns:", len(feature_cols))
    print("Missing feature values:", int(joined[feature_cols].isna().sum().sum()))

Join keys: ['patient_id', 'recording_id', 'chest_location']
Manifest rows: 5337
Feature-table rows: 6898

Duplicate key rows in manifests: 4616
Duplicate key rows in feature table: 5978

Duplicate manifest examples:


,patient_id,recording_id,chest_location,diagnosis,audio_path,fold
0,102,1b1,Ar,Healthy,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,1
1,102,1b1,Ar,Healthy,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,1
2,102,1b1,Ar,Healthy,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,1
3,102,1b1,Ar,Healthy,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,1
4,102,1b1,Ar,Healthy,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,1
5,102,1b1,Ar,Healthy,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,1
6,102,1b1,Ar,Healthy,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,1
7,102,1b1,Ar,Healthy,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,1
8,102,1b1,Ar,Healthy,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,1
9,102,1b1,Ar,Healthy,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,1



Duplicate feature-table examples:


,patient_id,recording_id,chest_location,diagnosis,cycle_number,processed_path
0,101,1b1,Al,URTI,1,audio/cycles/101/1b1_Al_001.wav
1,101,1b1,Al,URTI,2,audio/cycles/101/1b1_Al_002.wav
2,101,1b1,Al,URTI,3,audio/cycles/101/1b1_Al_003.wav
3,101,1b1,Al,URTI,4,audio/cycles/101/1b1_Al_004.wav
4,101,1b1,Al,URTI,5,audio/cycles/101/1b1_Al_005.wav
5,101,1b1,Al,URTI,6,audio/cycles/101/1b1_Al_006.wav
6,101,1b1,Al,URTI,7,audio/cycles/101/1b1_Al_007.wav
7,101,1b1,Al,URTI,8,audio/cycles/101/1b1_Al_008.wav
8,101,1b1,Al,URTI,9,audio/cycles/101/1b1_Al_009.wav
9,101,1b1,Al,URTI,10,audio/cycles/101/1b1_Al_010.wav



STOP: join keys are not unique. We need to inspect the duplicates.


In [101]:
import pandas as pd
from pathlib import Path

features_path = (
    PROJECT_ROOT / "data" / "processed" / "features"
    / "respiratory_cycle_features.csv"
)

all_features = pd.read_csv(features_path)

print("Feature table columns:")
print(all_features.columns.tolist())

print("\nManifest sample:")
display(
    cv_data[
        [
            "patient_id", "recording_id", "chest_location",
            "audio_path", "diagnosis", "fold"
        ]
    ].head(10)
)

print("\nFeature-table sample:")
display(
    all_features[
        [
            "patient_id", "recording_id", "chest_location",
            "cycle_number", "processed_path", "diagnosis"
        ]
    ].head(10)
)

# Compare normalized cycle paths.
def normalize_cycle_path(value):
    if pd.isna(value):
        return None
    return str(value).replace("\\", "/").strip().lstrip("./")

manifest_paths = set(
    cv_data["audio_path"].map(normalize_cycle_path).dropna()
)
feature_paths = set(
    all_features["processed_path"].map(normalize_cycle_path).dropna()
)

print("\nUnique manifest audio paths:", len(manifest_paths))
print("Unique feature processed paths:", len(feature_paths))
print("Exact normalized path overlap:", len(manifest_paths & feature_paths))
print("Manifest paths not found in features:", len(manifest_paths - feature_paths))
print("Feature paths not found in manifest:", len(feature_paths - manifest_paths))

print("\nExample manifest path:")
print(next(iter(manifest_paths), None))

print("\nExample feature path:")
print(next(iter(feature_paths), None))

Feature table columns:
['patient_id', 'recording_id', 'chest_location', 'acquisition_mode', 'equipment', 'cycle_number', 'start_time', 'end_time', 'sound_label', 'crackles', 'wheezes', 'diagnosis', 'processed_path', 'mean', 'std', 'variance', 'rms', 'max', 'min', 'peak_to_peak', 'median', 'skewness', 'kurtosis', 'energy', 'zero_crossing_rate', 'spectral_centroid', 'spectral_bandwidth', 'spectral_rolloff', 'spectral_flatness', 'spectral_energy', 'dominant_frequency', 'spectral_entropy', 'band_energy_0_500', 'band_energy_500_1000', 'band_energy_1000_1500', 'band_energy_1500_2000', 'mfcc_1_mean', 'mfcc_2_mean', 'mfcc_3_mean', 'mfcc_4_mean', 'mfcc_5_mean', 'mfcc_6_mean', 'mfcc_7_mean', 'mfcc_8_mean', 'mfcc_9_mean', 'mfcc_10_mean', 'mfcc_11_mean', 'mfcc_12_mean', 'mfcc_13_mean', 'wavelet_energy', 'wavelet_entropy', 'wavelet_std']

Manifest sample:


,patient_id,recording_id,chest_location,audio_path,diagnosis,fold
0,102,1b1,Ar,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,Healthy,1
1,102,1b1,Ar,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,Healthy,1
2,102,1b1,Ar,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,Healthy,1
3,102,1b1,Ar,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,Healthy,1
4,102,1b1,Ar,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,Healthy,1
5,102,1b1,Ar,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,Healthy,1
6,102,1b1,Ar,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,Healthy,1
7,102,1b1,Ar,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,Healthy,1
8,102,1b1,Ar,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,Healthy,1
9,102,1b1,Ar,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,Healthy,1



Feature-table sample:


,patient_id,recording_id,chest_location,cycle_number,processed_path,diagnosis
0,101,1b1,Al,1,audio/cycles/101/1b1_Al_001.wav,URTI
1,101,1b1,Al,2,audio/cycles/101/1b1_Al_002.wav,URTI
2,101,1b1,Al,3,audio/cycles/101/1b1_Al_003.wav,URTI
3,101,1b1,Al,4,audio/cycles/101/1b1_Al_004.wav,URTI
4,101,1b1,Al,5,audio/cycles/101/1b1_Al_005.wav,URTI
5,101,1b1,Al,6,audio/cycles/101/1b1_Al_006.wav,URTI
6,101,1b1,Al,7,audio/cycles/101/1b1_Al_007.wav,URTI
7,101,1b1,Al,8,audio/cycles/101/1b1_Al_008.wav,URTI
8,101,1b1,Al,9,audio/cycles/101/1b1_Al_009.wav,URTI
9,101,1b1,Al,10,audio/cycles/101/1b1_Al_010.wav,URTI



Unique manifest audio paths: 5337
Unique feature processed paths: 6898
Exact normalized path overlap: 0
Manifest paths not found in features: 5337
Feature paths not found in manifest: 6898

Example manifest path:
Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/audio/cycles/149/1b1_Pl_016.wav

Example feature path:
audio/cycles/166/1p1_Ll_002.wav


In [102]:
from pathlib import Path
import pandas as pd

def filename_only(value):
    if pd.isna(value):
        return None
    return str(value).replace("\\", "/").rstrip("/").split("/")[-1]

manifest_check = cv_data.copy()
feature_check = all_features.copy()

manifest_check["cycle_filename"] = (
    manifest_check["audio_path"].map(filename_only)
)
feature_check["cycle_filename"] = (
    feature_check["processed_path"].map(filename_only)
)

print("Manifest rows:", len(manifest_check))
print("Feature rows:", len(feature_check))

print("\nUnique filenames:")
print("Manifest:", manifest_check["cycle_filename"].nunique())
print("Features:", feature_check["cycle_filename"].nunique())

print("\nDuplicate filenames:")
print("Manifest:", manifest_check["cycle_filename"].duplicated().sum())
print("Features:", feature_check["cycle_filename"].duplicated().sum())

manifest_names = set(manifest_check["cycle_filename"].dropna())
feature_names = set(feature_check["cycle_filename"].dropna())

print("\nUnique filename overlap:", len(manifest_names & feature_names))
print("Manifest filenames without feature match:",
      len(manifest_names - feature_names))
print("Feature filenames without manifest match:",
      len(feature_names - manifest_names))

print("\nMatching filename examples:")
matching_names = sorted(manifest_names & feature_names)
print(matching_names[:15])

print("\nManifest example:")
display(
    manifest_check[
        ["patient_id", "recording_id", "chest_location",
         "cycle_filename", "diagnosis", "fold"]
    ].head(8)
)

print("\nFeature example:")
display(
    feature_check[
        ["patient_id", "recording_id", "chest_location",
         "cycle_number", "cycle_filename", "diagnosis"]
    ].head(8)
)

Manifest rows: 5337
Feature rows: 6898

Unique filenames:
Manifest: 1949
Features: 2073

Duplicate filenames:
Manifest: 3388
Features: 4825

Unique filename overlap: 1949
Manifest filenames without feature match: 0
Feature filenames without manifest match: 124

Matching filename examples:
['1b1_Al_001.wav', '1b1_Al_002.wav', '1b1_Al_003.wav', '1b1_Al_004.wav', '1b1_Al_005.wav', '1b1_Al_006.wav', '1b1_Al_007.wav', '1b1_Al_008.wav', '1b1_Al_009.wav', '1b1_Al_010.wav', '1b1_Al_011.wav', '1b1_Al_012.wav', '1b1_Al_013.wav', '1b1_Al_014.wav', '1b1_Al_015.wav']

Manifest example:


,patient_id,recording_id,chest_location,cycle_filename,diagnosis,fold
0,102,1b1,Ar,1b1_Ar_001.wav,Healthy,1
1,102,1b1,Ar,1b1_Ar_002.wav,Healthy,1
2,102,1b1,Ar,1b1_Ar_003.wav,Healthy,1
3,102,1b1,Ar,1b1_Ar_004.wav,Healthy,1
4,102,1b1,Ar,1b1_Ar_005.wav,Healthy,1
5,102,1b1,Ar,1b1_Ar_006.wav,Healthy,1
6,102,1b1,Ar,1b1_Ar_007.wav,Healthy,1
7,102,1b1,Ar,1b1_Ar_008.wav,Healthy,1



Feature example:


,patient_id,recording_id,chest_location,cycle_number,cycle_filename,diagnosis
0,101,1b1,Al,1,1b1_Al_001.wav,URTI
1,101,1b1,Al,2,1b1_Al_002.wav,URTI
2,101,1b1,Al,3,1b1_Al_003.wav,URTI
3,101,1b1,Al,4,1b1_Al_004.wav,URTI
4,101,1b1,Al,5,1b1_Al_005.wav,URTI
5,101,1b1,Al,6,1b1_Al_006.wav,URTI
6,101,1b1,Al,7,1b1_Al_007.wav,URTI
7,101,1b1,Al,8,1b1_Al_008.wav,URTI


In [103]:
import pandas as pd

# Use the existing in-memory dataframes:
# cv_data, all_features

manifest = cv_data.copy()
features = all_features.copy()

manifest["cycle_filename"] = (
    manifest["audio_path"]
    .astype(str)
    .str.replace("\\", "/", regex=False)
    .str.split("/")
    .str[-1]
)

features["cycle_filename"] = (
    features["processed_path"]
    .astype(str)
    .str.replace("\\", "/", regex=False)
    .str.split("/")
    .str[-1]
)

# Composite key: patient + recording + chest location + cycle filename.
keys = [
    "patient_id",
    "recording_id",
    "chest_location",
    "cycle_filename",
]

print("Duplicate manifest composite keys:",
      int(manifest.duplicated(keys).sum()))
print("Duplicate feature composite keys:",
      int(features.duplicated(keys).sum()))

if manifest.duplicated(keys).any() or features.duplicated(keys).any():
    print("\nSTOP: composite keys are not unique.")
else:
    feature_columns = [
        c for c in features.columns
        if c not in [
            "patient_id", "recording_id", "chest_location",
            "acquisition_mode", "equipment", "cycle_number",
            "start_time", "end_time", "sound_label",
            "crackles", "wheezes", "diagnosis",
            "processed_path", "cycle_filename",
        ]
    ]

    checked = manifest.merge(
        features[
            keys + ["cycle_number", "diagnosis"] + feature_columns
        ],
        on=keys,
        how="left",
        validate="one_to_one",
        indicator=True,
        suffixes=("_manifest", "_feature"),
    )

    print("\nMatched rows:")
    print(checked["_merge"].value_counts())

    matched = checked["_merge"].eq("both")

    diagnosis_mismatch = (
        checked.loc[matched, "diagnosis_manifest"].astype(str)
        != checked.loc[matched, "diagnosis_feature"].astype(str)
    )

    print("\nDiagnosis mismatches:", int(diagnosis_mismatch.sum()))
    print("Unmatched manifest cycles:", int((~matched).sum()))
    print("Missing feature values:",
          int(checked.loc[matched, feature_columns].isna().sum().sum()))

    print("\nCycle number vs filename-derived cycle number:")
    checked["filename_cycle_number"] = (
        checked["cycle_filename"]
        .str.extract(r"_(\d+)\.wav$", expand=False)
    )
    checked["filename_cycle_number"] = pd.to_numeric(
        checked["filename_cycle_number"], errors="coerce"
    )

    print(
        "Mismatches:",
        int((
            matched
            & (
                checked["cycle_number"]
                != checked["filename_cycle_number"]
            )
        ).sum())
    )

Duplicate manifest composite keys: 0
Duplicate feature composite keys: 0

Matched rows:
_merge
both          5337
left_only        0
right_only       0
Name: count, dtype: int64

Diagnosis mismatches: 0
Unmatched manifest cycles: 0
Missing feature values: 0

Cycle number vs filename-derived cycle number:
Mismatches: 0


In [104]:
import numpy as np
import pandas as pd

# Recreate the validated cycle-level mapping.
manifest = cv_data.copy()
features = all_features.copy()

manifest["cycle_filename"] = (
    manifest["audio_path"]
    .astype(str)
    .str.replace("\\", "/", regex=False)
    .str.split("/")
    .str[-1]
)

features["cycle_filename"] = (
    features["processed_path"]
    .astype(str)
    .str.replace("\\", "/", regex=False)
    .str.split("/")
    .str[-1]
)

join_keys = [
    "patient_id",
    "recording_id",
    "chest_location",
    "cycle_filename",
]

metadata_columns = [
    "patient_id", "recording_id", "chest_location",
    "acquisition_mode", "equipment", "cycle_number",
    "start_time", "end_time", "sound_label",
    "crackles", "wheezes", "diagnosis",
    "processed_path", "cycle_filename",
]

candidate_features = [
    col for col in features.columns
    if col not in metadata_columns
]

cv_dataset = manifest.merge(
    features[join_keys + ["diagnosis"] + candidate_features],
    on=join_keys,
    how="left",
    validate="one_to_one",
    suffixes=("_manifest", "_feature"),
)

# Keep the manifest's diagnosis and fold as authoritative.
cv_dataset = cv_dataset.rename(
    columns={"diagnosis_manifest": "diagnosis"}
)

if "diagnosis" not in cv_dataset.columns:
    raise RuntimeError("Could not resolve the diagnosis column.")

# Confirm the expected dataset dimensions and values.
assert len(cv_dataset) == 5337
assert cv_dataset["patient_id"].nunique() == 101
assert cv_dataset["fold"].notna().all()
assert len(candidate_features) == 39
assert not cv_dataset[candidate_features].isna().any().any()
assert np.isfinite(
    cv_dataset[candidate_features].to_numpy(dtype=float)
).all()

print("CV dataset prepared successfully.")
print("Rows:", len(cv_dataset))
print("Unique patients:", cv_dataset["patient_id"].nunique())
print("Feature count:", len(candidate_features))
print("Fold counts:")
print(cv_dataset.groupby("fold").size())
print("\nClass counts:")
print(cv_dataset["diagnosis"].value_counts().sort_index())
print("\nFeature matrix shape:",
      cv_dataset[candidate_features].shape)

CV dataset prepared successfully.
Rows: 5337
Unique patients: 101
Feature count: 39
Fold counts:
fold
1    1948
2    1262
3    2127
dtype: int64

Class counts:
diagnosis
Asthma               6
Bronchiectasis      84
Bronchiolitis      141
COPD              4409
Healthy            238
LRTI                32
Pneumonia          233
URTI               194
Name: count, dtype: int64

Feature matrix shape: (5337, 39)


In [107]:

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

X = cv_dataset[candidate_features].astype(float)
y = cv_dataset["diagnosis"].astype(str)
groups = cv_dataset["patient_id"]
fold_ids = cv_dataset["fold"].astype(int)

class_names = sorted(y.unique())
n_features_to_select = min(20, len(candidate_features))

model_factories = {
    "Logistic Regression": lambda: Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("feature_selection", SelectKBest(f_classif, k=n_features_to_select)),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=3000, class_weight="balanced", random_state=42
        )),
    ]),
    "SVM": lambda: Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("feature_selection", SelectKBest(f_classif, k=n_features_to_select)),
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf", class_weight="balanced",
            probability=False, random_state=42
        )),
    ]),
    "Random Forest": lambda: Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("feature_selection", SelectKBest(f_classif, k=n_features_to_select)),
        ("model", RandomForestClassifier(
            n_estimators=300, class_weight="balanced",
            random_state=42, n_jobs=-1
        )),
    ]),
}

fold_results = []
oof_predictions = []

for model_name, make_model in model_factories.items():
    print(f"\n{'=' * 65}\n{model_name}\n{'=' * 65}")

    model_oof = np.empty(len(cv_dataset), dtype=object)
    model_oof[:] = None

    for fold in (1, 2, 3):
        train_mask = fold_ids != fold
        val_mask = fold_ids == fold

        X_train = X.loc[train_mask]
        X_val = X.loc[val_mask]
        y_train = y.loc[train_mask]
        y_val = y.loc[val_mask]

        train_patients = set(groups.loc[train_mask])
        val_patients = set(groups.loc[val_mask])
        assert train_patients.isdisjoint(val_patients)

        model = make_model()
        model.fit(X_train, y_train)

        predictions = model.predict(X_val)
        model_oof[val_mask.to_numpy()] = predictions

        fold_accuracy = accuracy_score(y_val, predictions)
        fold_balanced = balanced_accuracy_score(y_val, predictions)
        fold_macro_f1 = f1_score(
            y_val, predictions,
            labels=class_names, average="macro", zero_division=0
        )

        fold_results.append({
            "model": model_name,
            "fold": fold,
            "train_patients": len(train_patients),
            "validation_patients": len(val_patients),
            "validation_cycles": int(val_mask.sum()),
            "accuracy": fold_accuracy,
            "balanced_accuracy": fold_balanced,
            "macro_precision": precision_score(
                y_val, predictions,
                labels=class_names, average="macro", zero_division=0
            ),
            "macro_recall": recall_score(
                y_val, predictions,
                labels=class_names, average="macro", zero_division=0
            ),
            "macro_f1": fold_macro_f1,
            "weighted_f1": f1_score(
                y_val, predictions, average="weighted", zero_division=0
            ),
        })

        print(
            f"Fold {fold}: cycles={int(val_mask.sum())}, "
            f"accuracy={fold_accuracy:.4f}, "
            f"balanced_accuracy={fold_balanced:.4f}, "
            f"macro_F1={fold_macro_f1:.4f}"
        )

    assert all(pred is not None for pred in model_oof)

    print("\nCombined out-of-fold classification report:")
    print(classification_report(
        y, model_oof,
        labels=class_names,
        target_names=class_names,
        zero_division=0,
    ))

    for idx, pred in enumerate(model_oof):
        oof_predictions.append({
            "model": model_name,
            "patient_id": cv_dataset.iloc[idx]["patient_id"],
            "fold": int(fold_ids.iloc[idx]),
            "true_diagnosis": y.iloc[idx],
            "predicted_diagnosis": pred,
        })

fold_results_df = pd.DataFrame(fold_results)
oof_predictions_df = pd.DataFrame(oof_predictions)

print("\nFOLD-LEVEL RESULTS")
display(fold_results_df.sort_values(["model", "fold"]).round(4))

print("\nMEAN ± STANDARD DEVIATION ACROSS FOLDS")
summary = (
    fold_results_df.groupby("model")[
        ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]
    ]
    .agg(["mean", "std"])
    .round(4)
)
display(summary)

print("\nOUT-OF-FOLD DATASET ROWS:", len(oof_predictions_df))



Logistic Regression
Fold 1: cycles=1948, accuracy=0.6283, balanced_accuracy=0.2873, macro_F1=0.2272


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Fold 2: cycles=1262, accuracy=0.4691, balanced_accuracy=0.3004, macro_F1=0.2030
Fold 3: cycles=2127, accuracy=0.7334, balanced_accuracy=0.4204, macro_F1=0.1983

Combined out-of-fold classification report:
                precision    recall  f1-score   support

        Asthma       0.00      0.00      0.00         6
Bronchiectasis       0.18      0.61      0.28        84
 Bronchiolitis       0.04      0.08      0.05       141
          COPD       0.95      0.71      0.81      4409
       Healthy       0.24      0.36      0.28       238
          LRTI       0.00      0.00      0.00        32
     Pneumonia       0.12      0.35      0.18       233
          URTI       0.07      0.10      0.08       194

      accuracy                           0.63      5337
     macro avg       0.20      0.28      0.21      5337
  weighted avg       0.80      0.63      0.70      5337


SVM


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Fold 1: cycles=1948, accuracy=0.6725, balanced_accuracy=0.2241, macro_F1=0.1923


...

Fold 3: cycles=2127, accuracy=0.8481, balanced_accuracy=0.3639, macro_F1=0.2251

Combined out-of-fold classification report:
                precision    recall  f1-score   support

        Asthma       0.00      0.00      0.00         6
Bronchiectasis       0.00      0.00      0.00        84
 Bronchiolitis       0.03      0.03      0.03       141
          COPD       0.90      0.88      0.89      4409
       Healthy       0.24      0.42      0.30       238
          LRTI       0.00      0.00      0.00        32
     Pneumonia       0.17      0.20      0.18       233
          URTI       0.07      0.05      0.06       194

      accuracy                           0.76      5337
     macro avg       0.18      0.20      0.18      5337
  weighted avg       0.77      0.76      0.76      5337


FOLD-LEVEL RESULTS


,model,fold,train_patients,validation_patients,validation_cycles,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1
0,Logistic Regression,1,67,34,1948,0.6283,0.2873,0.2243,0.2514,0.2272,0.6804
1,Logistic Regression,2,67,34,1262,0.4691,0.3004,0.2041,0.3004,0.2030,0.5736
2,Logistic Regression,3,68,33,2127,0.7334,0.4204,0.1886,0.3153,0.1983,0.7882
6,Random Forest,1,67,34,1948,0.7305,0.2190,0.2001,0.1917,0.1766,0.7019
7,Random Forest,2,67,34,1262,0.6513,0.1794,0.1505,0.1794,0.1528,0.6998
8,Random Forest,3,68,33,2127,0.8481,0.3639,0.2187,0.2729,0.2251,0.8526
3,SVM,1,67,34,1948,0.6725,0.2241,0.2032,0.1961,0.1923,0.7034
4,SVM,2,67,34,1262,0.6141,0.1847,0.1630,0.1847,0.1613,0.6840
5,SVM,3,68,33,2127,0.7551,0.4399,0.2048,0.3299,0.2169,0.8071



MEAN ± STANDARD DEVIATION ACROSS FOLDS


accuracy         balanced_accuracy         macro_f1  \
                        mean     std              mean     std     mean   
model                                                                     
Logistic Regression   0.6103  0.1331            0.3360  0.0734   0.2095   
Random Forest         0.7433  0.0990            0.2541  0.0971   0.1848   
SVM                   0.6805  0.0708            0.2829  0.1374   0.1902   

                            weighted_f1          
                        std        mean     std  
model                                            
Logistic Regression  0.0155      0.6807  0.1073  
Random Forest        0.0369      0.7514  0.0876  
SVM                  0.0279      0.7315  0.0662


OUT-OF-FOLD DATASET ROWS: 16011


In [108]:
try:
    import imblearn
    print("imbalanced-learn installed:", imblearn.__version__)
except ImportError:
    print("imbalanced-learn is NOT installed.")

imbalanced-learn installed: 0.14.2


In [109]:

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# Four experiments: two baselines and two oversampling variants
experiments = {
    "LR_balanced": {
        "model": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ),
        "oversample": False,
        "scale": True
    },
    "LR_oversampled": {
        "model": LogisticRegression(
            max_iter=3000,
            class_weight=None,
            random_state=42
        ),
        "oversample": True,
        "scale": True
    },
    "RF_balanced": {
        "model": RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
        "oversample": False,
        "scale": False
    },
    "RF_oversampled": {
        "model": RandomForestClassifier(
            n_estimators=300,
            class_weight=None,
            random_state=42,
            n_jobs=-1
        ),
        "oversample": True,
        "scale": False
    }
}

cv_results = []
cv_predictions = []

X_all = cv_dataset[candidate_features]
y_all = cv_dataset["diagnosis"]
fold_all = cv_dataset["fold"]
patient_all = cv_dataset["patient_id"]

for experiment_name, config in experiments.items():
    print(f"\nRunning: {experiment_name}")

    for fold in sorted(fold_all.unique()):
        train_mask = fold_all != fold
        val_mask = fold_all == fold

        X_train = X_all.loc[train_mask]
        y_train = y_all.loc[train_mask]
        X_val = X_all.loc[val_mask]
        y_val = y_all.loc[val_mask]

        train_patients = set(patient_all.loc[train_mask])
        val_patients = set(patient_all.loc[val_mask])

        assert train_patients.isdisjoint(val_patients), (
            f"Patient leakage detected in fold {fold}"
        )

        steps = [
            ("imputer", SimpleImputer(strategy="median")),
            ("selector", SelectKBest(score_func=f_classif, k=20))
        ]

        if config["scale"]:
            steps.append(("scaler", StandardScaler()))

        if config["oversample"]:
            steps.append((
                "oversampler",
                RandomOverSampler(random_state=42)
            ))

        steps.append(("model", clone(config["model"])))

        # Oversampling, if enabled, happens only on the training fold.
        if config["oversample"]:
            pipeline = ImbPipeline(steps)
        else:
            pipeline = SklearnPipeline(steps)

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_val)

        cv_results.append({
            "experiment": experiment_name,
            "fold": int(fold),
            "accuracy": accuracy_score(y_val, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_val, y_pred),
            "macro_f1": f1_score(
                y_val, y_pred, average="macro", zero_division=0
            ),
            "weighted_f1": f1_score(
                y_val, y_pred, average="weighted", zero_division=0
            ),
            "train_patients": len(train_patients),
            "validation_patients": len(val_patients)
        })

        cv_predictions.append(pd.DataFrame({
            "experiment": experiment_name,
            "fold": int(fold),
            "patient_id": patient_all.loc[val_mask].to_numpy(),
            "y_true": y_val.to_numpy(),
            "y_pred": y_pred
        }))

results_df = pd.DataFrame(cv_results)
predictions_df = pd.concat(cv_predictions, ignore_index=True)

summary_df = (
    results_df.groupby("experiment")
    .agg({
        "accuracy": ["mean", "std"],
        "balanced_accuracy": ["mean", "std"],
        "macro_f1": ["mean", "std"],
        "weighted_f1": ["mean", "std"]
    })
    .round(4)
)

print("\n========== 3-FOLD CV: MEAN ± STANDARD DEVIATION ==========")
display(summary_df)

print("\n========== SCORES BY FOLD ==========")
display(
    results_df.sort_values(["experiment", "fold"]).round(4)
)

print("\nPrediction rows:", len(predictions_df))
print("Expected rows:", len(cv_dataset) * len(experiments))
assert len(predictions_df) == len(cv_dataset) * len(experiments)
print("Patient-level fold isolation verified for every experiment.")



Running: LR_balanced


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



Running: LR_oversampled


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



Running: RF_balanced


...

/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


accuracy         balanced_accuracy         macro_f1          \
                   mean     std              mean     std     mean     std   
experiment                                                                   
LR_balanced      0.6103  0.1331            0.3360  0.0734   0.2095  0.0155   
LR_oversampled   0.6078  0.1283            0.3233  0.0783   0.2054  0.0085   
RF_balanced      0.7433  0.0990            0.2541  0.0971   0.1932  0.0369   
RF_oversampled   0.7838  0.0806            0.1831  0.0330   0.1635  0.0126   

               weighted_f1          
                      mean     std  
experiment                          
LR_balanced         0.6807  0.1073  
LR_oversampled      0.6790  0.1042  
RF_balanced         0.7514  0.0876  
RF_oversampled      0.7548  0.0809


========== SCORES BY FOLD ==========


,experiment,fold,accuracy,balanced_accuracy,macro_f1,weighted_f1,train_patients,validation_patients
0,LR_balanced,1,0.6283,0.2873,0.2272,0.6804,67,34
1,LR_balanced,2,0.4691,0.3004,0.2030,0.5736,67,34
2,LR_balanced,3,0.7334,0.4204,0.1983,0.7882,68,33
3,LR_oversampled,1,0.6165,0.2614,0.2115,0.6688,67,34
4,LR_oversampled,2,0.4754,0.2973,0.2091,0.5803,67,34
5,LR_oversampled,3,0.7315,0.4113,0.1956,0.7879,68,33
6,RF_balanced,1,0.7305,0.2190,0.2019,0.7019,67,34
7,RF_balanced,2,0.6513,0.1794,0.1528,0.6998,67,34
8,RF_balanced,3,0.8481,0.3639,0.2251,0.8526,68,33
9,RF_oversampled,1,0.7587,0.1632,0.1575,0.6889,67,34



Prediction rows: 21348
Expected rows: 21348
Patient-level fold isolation verified for every experiment.


In [110]:

from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# Inspect the two strongest candidates for minority-class evaluation.
# These predictions come only from their held-out validation folds.
for model_name in ["LR_balanced", "LR_oversampled"]:
    model_preds = predictions_df[
        predictions_df["experiment"] == model_name
    ]

    print(f"\n{'=' * 65}")
    print(f"OUT-OF-FOLD CLASSIFICATION REPORT: {model_name}")
    print("=" * 65)

    print(classification_report(
        model_preds["y_true"],
        model_preds["y_pred"],
        labels=sorted(cv_dataset["diagnosis"].unique()),
        zero_division=0,
        digits=3
    ))

    cm = confusion_matrix(
        model_preds["y_true"],
        model_preds["y_pred"],
        labels=sorted(cv_dataset["diagnosis"].unique())
    )

    class_names = sorted(cv_dataset["diagnosis"].unique())

    cm_df = pd.DataFrame(
        cm,
        index=[f"Actual: {c}" for c in class_names],
        columns=[f"Pred: {c}" for c in class_names]
    )

    print("\nConfusion matrix:")
    display(cm_df)

print("\nDisease support in development data:")
display(
    cv_dataset["diagnosis"]
    .value_counts()
    .rename_axis("disease")
    .to_frame("cycle_count")
)



OUT-OF-FOLD CLASSIFICATION REPORT: LR_balanced
                precision    recall  f1-score   support

        Asthma      0.000     0.000     0.000         6
Bronchiectasis      0.184     0.607     0.283        84
 Bronchiolitis      0.038     0.078     0.051       141
          COPD      0.946     0.709     0.811      4409
       Healthy      0.237     0.357     0.285       238
          LRTI      0.000     0.000     0.000        32
     Pneumonia      0.119     0.348     0.177       233
          URTI      0.068     0.103     0.082       194

      accuracy                          0.633      5337
     macro avg      0.199     0.275     0.211      5337
  weighted avg      0.804     0.633     0.699      5337


Confusion matrix:


,Pred: Asthma,Pred: Bronchiectasis,Pred: Bronchiolitis,Pred: COPD,Pred: Healthy,Pred: LRTI,Pred: Pneumonia,Pred: URTI
Actual: Asthma,0,0,0,0,3,0,0,3
Actual: Bronchiectasis,0,51,0,32,1,0,0,0
Actual: Bronchiolitis,0,0,11,50,13,11,21,35
Actual: COPD,11,217,180,3128,149,52,488,184
Actual: Healthy,1,1,22,22,85,18,60,29
Actual: LRTI,0,0,14,0,8,0,6,4
Actual: Pneumonia,0,0,21,62,47,2,81,20
Actual: URTI,9,8,44,11,53,25,24,20



OUT-OF-FOLD CLASSIFICATION REPORT: LR_oversampled
                precision    recall  f1-score   support

        Asthma      0.000     0.000     0.000         6
Bronchiectasis      0.154     0.512     0.237        84
 Bronchiolitis      0.048     0.092     0.063       141
          COPD      0.943     0.708     0.809      4409
       Healthy      0.244     0.349     0.287       238
          LRTI      0.000     0.000     0.000        32
     Pneumonia      0.118     0.339     0.175       233
          URTI      0.049     0.082     0.061       194

      accuracy                          0.629      5337
     macro avg      0.195     0.260     0.204      5337
  weighted avg      0.800     0.629     0.696      5337


Confusion matrix:


,Pred: Asthma,Pred: Bronchiectasis,Pred: Bronchiolitis,Pred: COPD,Pred: Healthy,Pred: LRTI,Pred: Pneumonia,Pred: URTI
Actual: Asthma,0,0,0,0,2,0,1,3
Actual: Bronchiectasis,0,43,0,39,1,0,0,1
Actual: Bronchiolitis,0,0,13,45,11,9,28,35
Actual: COPD,10,225,156,3123,137,65,470,223
Actual: Healthy,1,3,20,24,83,20,60,27
Actual: LRTI,0,0,13,0,10,0,5,4
Actual: Pneumonia,0,0,24,61,44,7,79,18
Actual: URTI,8,8,44,21,52,20,25,16



Disease support in development data:


,cycle_count
disease,
COPD,4409
Healthy,238
Pneumonia,233
URTI,194
Bronchiolitis,141
Bronchiectasis,84
LRTI,32
Asthma,6


In [111]:

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

# Fixed development data and patient-level folds
X_all = cv_dataset[candidate_features]
y_all = cv_dataset["diagnosis"]
fold_all = cv_dataset["fold"]
patient_all = cv_dataset["patient_id"]

# Estimate class weights using each training fold only.
weight_strategies = ["balanced", "sqrt", "custom"]

weight_results = []
weight_predictions = []

for fold in sorted(fold_all.unique()):
    train_mask = fold_all != fold
    val_mask = fold_all == fold

    X_train = X_all.loc[train_mask]
    y_train = y_all.loc[train_mask]
    X_val = X_all.loc[val_mask]
    y_val = y_all.loc[val_mask]

    assert set(patient_all.loc[train_mask]).isdisjoint(
        set(patient_all.loc[val_mask])
    ), f"Patient overlap in fold {fold}"

    counts = y_train.value_counts()
    n_train = len(y_train)
    n_classes = len(counts)

    balanced_weights = {
        cls: n_train / (n_classes * count)
        for cls, count in counts.items()
    }

    sqrt_weights = {
        cls: np.sqrt(weight)
        for cls, weight in balanced_weights.items()
    }

    # Cap the largest weights to reduce extreme minority-class influence.
    custom_weights = {
        cls: min(weight, 5.0)
        for cls, weight in sqrt_weights.items()
    }

    strategies = {
        "balanced": "balanced",
        "sqrt": sqrt_weights,
        "custom": custom_weights
    }

    for strategy_name, class_weight in strategies.items():
        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("selector", SelectKBest(score_func=f_classif, k=20)),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=3000,
                class_weight=class_weight,
                random_state=42
            ))
        ])

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_val)

        weight_results.append({
            "strategy": strategy_name,
            "fold": int(fold),
            "accuracy": accuracy_score(y_val, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_val, y_pred),
            "macro_f1": f1_score(
                y_val, y_pred, average="macro", zero_division=0
            ),
            "weighted_f1": f1_score(
                y_val, y_pred, average="weighted", zero_division=0
            )
        })

        weight_predictions.append(pd.DataFrame({
            "strategy": strategy_name,
            "fold": int(fold),
            "patient_id": patient_all.loc[val_mask].to_numpy(),
            "y_true": y_val.to_numpy(),
            "y_pred": y_pred
        }))

weight_results_df = pd.DataFrame(weight_results)
weight_predictions_df = pd.concat(
    weight_predictions, ignore_index=True
)

print("========== SCORES BY FOLD ==========")
display(
    weight_results_df.sort_values(["strategy", "fold"]).round(4)
)

print("\n========== MEAN ± STANDARD DEVIATION ==========")
display(
    weight_results_df.groupby("strategy")[
        ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]
    ].agg(["mean", "std"]).round(4)
)

print("\nPredictions:", len(weight_predictions_df))
print("Expected:", len(cv_dataset) * len(weight_strategies))


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


========== SCORES BY FOLD ==========


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


,strategy,fold,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,balanced,1,0.6283,0.2873,0.2272,0.6804
3,balanced,2,0.4691,0.3004,0.2030,0.5736
6,balanced,3,0.7334,0.4204,0.1983,0.7882
2,custom,1,0.7048,0.2176,0.1920,0.7055
5,custom,2,0.5769,0.2792,0.2257,0.6580
8,custom,3,0.8364,0.2997,0.2050,0.8389
1,sqrt,1,0.7033,0.2164,0.1915,0.7045
4,sqrt,2,0.5777,0.2866,0.2297,0.6588
7,sqrt,3,0.8364,0.2997,0.2054,0.8389



========== MEAN ± STANDARD DEVIATION ==========


accuracy         balanced_accuracy         macro_f1          \
             mean     std              mean     std     mean     std   
strategy                                                               
balanced   0.6103  0.1331            0.3360  0.0734   0.2095  0.0155   
custom     0.7060  0.1298            0.2655  0.0428   0.2076  0.0170   
sqrt       0.7058  0.1294            0.2676  0.0448   0.2089  0.0193   

         weighted_f1          
                mean     std  
strategy                      
balanced      0.6807  0.1073  
custom        0.7342  0.0938  
sqrt          0.7341  0.0936


Predictions: 16011
Expected: 16011


In [112]:

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report
)

patient_level_rows = []

for strategy in ["balanced", "sqrt", "custom"]:
    subset = weight_predictions_df[
        weight_predictions_df["strategy"] == strategy
    ].copy()

    # Each patient must belong to exactly one held-out fold.
    assert subset.groupby("patient_id")["fold"].nunique().max() == 1

    # Aggregate cycle predictions into one prediction per patient.
    # Ground-truth diagnosis must also be consistent within each patient.
    patient_records = []

    for patient_id, group in subset.groupby("patient_id"):
        true_labels = group["y_true"].unique()

        assert len(true_labels) == 1, (
            f"Inconsistent true labels for patient {patient_id}"
        )

        # Majority vote; ties are resolved deterministically by sorting.
        vote_counts = group["y_pred"].value_counts()
        top_count = vote_counts.max()
        tied_labels = sorted(
            vote_counts[vote_counts == top_count].index.tolist()
        )
        predicted_label = tied_labels[0]

        patient_records.append({
            "patient_id": patient_id,
            "fold": int(group["fold"].iloc[0]),
            "y_true": true_labels[0],
            "y_pred": predicted_label,
            "cycle_count": len(group)
        })

    patient_df = pd.DataFrame(patient_records)

    patient_level_rows.append({
        "strategy": strategy,
        "patients": len(patient_df),
        "accuracy": accuracy_score(
            patient_df["y_true"], patient_df["y_pred"]
        ),
        "balanced_accuracy": balanced_accuracy_score(
            patient_df["y_true"], patient_df["y_pred"]
        ),
        "macro_f1": f1_score(
            patient_df["y_true"],
            patient_df["y_pred"],
            average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            patient_df["y_true"],
            patient_df["y_pred"],
            average="weighted",
            zero_division=0
        )
    })

    print(f"\n===== PATIENT-LEVEL REPORT: {strategy} =====")
    print(classification_report(
        patient_df["y_true"],
        patient_df["y_pred"],
        labels=sorted(cv_dataset["diagnosis"].unique()),
        zero_division=0,
        digits=3
    ))

    print("Patients evaluated:", len(patient_df))

print("\n===== PATIENT-LEVEL SUMMARY =====")
display(pd.DataFrame(patient_level_rows).set_index("strategy").round(4))



===== PATIENT-LEVEL REPORT: balanced =====
                precision    recall  f1-score   support

        Asthma      0.000     0.000     0.000         1
Bronchiectasis      0.667     0.800     0.727         5
 Bronchiolitis      0.000     0.000     0.000         5
          COPD      0.837     0.706     0.766        51
       Healthy      0.474     0.429     0.450        21
          LRTI      0.000     0.000     0.000         2
     Pneumonia      0.105     0.400     0.167         5
          URTI      0.000     0.000     0.000        11

      accuracy                          0.505       101
     macro avg      0.260     0.292     0.264       101
  weighted avg      0.559     0.505     0.525       101

Patients evaluated: 101

===== PATIENT-LEVEL REPORT: sqrt =====
                precision    recall  f1-score   support

        Asthma      0.000     0.000     0.000         1
Bronchiectasis      0.500     0.200     0.286         5
 Bronchiolitis      0.000     0.000     0.000   

,patients,accuracy,balanced_accuracy,macro_f1,weighted_f1
strategy,,,,,
balanced,101,0.5050,0.2918,0.2637,0.5246
sqrt,101,0.5347,0.2245,0.2184,0.5027
custom,101,0.5347,0.2245,0.2184,0.5027


In [113]:

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

feature_experiments = {
    "all_39_features": "all",
    "anova_top_20": 20,
    "anova_top_30": 30
}

feature_results = []
feature_predictions = []

X_all = cv_dataset[candidate_features]
y_all = cv_dataset["diagnosis"]
fold_all = cv_dataset["fold"]
patient_all = cv_dataset["patient_id"]

for experiment_name, k in feature_experiments.items():
    print(f"Running: {experiment_name}")

    for fold in sorted(fold_all.unique()):
        train_mask = fold_all != fold
        val_mask = fold_all == fold

        X_train = X_all.loc[train_mask]
        y_train = y_all.loc[train_mask]
        X_val = X_all.loc[val_mask]
        y_val = y_all.loc[val_mask]

        assert set(patient_all.loc[train_mask]).isdisjoint(
            set(patient_all.loc[val_mask])
        )

        steps = [
            ("imputer", SimpleImputer(strategy="median"))
        ]

        if k != "all":
            steps.append(
                ("selector", SelectKBest(score_func=f_classif, k=k))
            )

        steps.extend([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=42
            ))
        ])

        pipeline = Pipeline(steps)
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_val)

        feature_results.append({
            "experiment": experiment_name,
            "fold": int(fold),
            "accuracy": accuracy_score(y_val, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_val, y_pred),
            "macro_f1": f1_score(
                y_val, y_pred, average="macro", zero_division=0
            ),
            "weighted_f1": f1_score(
                y_val, y_pred, average="weighted", zero_division=0
            )
        })

        feature_predictions.append(pd.DataFrame({
            "experiment": experiment_name,
            "fold": int(fold),
            "patient_id": patient_all.loc[val_mask].to_numpy(),
            "y_true": y_val.to_numpy(),
            "y_pred": y_pred
        }))

feature_results_df = pd.DataFrame(feature_results)
feature_predictions_df = pd.concat(
    feature_predictions, ignore_index=True
)

print("\n========== MEAN ± STANDARD DEVIATION ==========")
display(
    feature_results_df.groupby("experiment")[
        ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]
    ].agg(["mean", "std"]).round(4)
)

print("\n========== SCORES BY FOLD ==========")
display(
    feature_results_df.sort_values(["experiment", "fold"]).round(4)
)


Running: all_39_features


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Running: anova_top_20


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Running: anova_top_30

========== MEAN ± STANDARD DEVIATION ==========


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


accuracy         balanced_accuracy         macro_f1          \
                    mean     std              mean     std     mean     std   
experiment                                                                    
all_39_features   0.6398  0.1353            0.3371  0.1420   0.2144  0.0532   
anova_top_20      0.6103  0.1331            0.3360  0.0734   0.2095  0.0155   
anova_top_30      0.6344  0.1313            0.3489  0.0929   0.2139  0.0359   

                weighted_f1          
                       mean     std  
experiment                           
all_39_features      0.7040  0.1102  
anova_top_20         0.6807  0.1073  
anova_top_30         0.6993  0.1023


========== SCORES BY FOLD ==========


,experiment,fold,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,all_39_features,1,0.6674,0.3450,0.2619,0.7061
1,all_39_features,2,0.4929,0.1913,0.1569,0.5929
2,all_39_features,3,0.7593,0.4750,0.2245,0.8131
3,anova_top_20,1,0.6283,0.2873,0.2272,0.6804
4,anova_top_20,2,0.4691,0.3004,0.2030,0.5736
5,anova_top_20,3,0.7334,0.4204,0.1983,0.7882
6,anova_top_30,1,0.6581,0.3503,0.2551,0.7005
7,anova_top_30,2,0.4929,0.2553,0.1892,0.5963
8,anova_top_30,3,0.7522,0.4411,0.1975,0.8009


In [114]:

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)
import pandas as pd

patient_feature_results = []
patient_feature_predictions = {}

for experiment_name in [
    "all_39_features",
    "anova_top_20",
    "anova_top_30"
]:
    subset = feature_predictions_df[
        feature_predictions_df["experiment"] == experiment_name
    ]

    patient_records = []

    for patient_id, group in subset.groupby("patient_id"):
        true_labels = group["y_true"].unique()
        assert len(true_labels) == 1, (
            f"Inconsistent diagnosis for patient {patient_id}"
        )

        vote_counts = group["y_pred"].value_counts()
        max_votes = vote_counts.max()
        tied_labels = sorted(
            vote_counts[vote_counts == max_votes].index.tolist()
        )

        patient_records.append({
            "patient_id": patient_id,
            "y_true": true_labels[0],
            "y_pred": tied_labels[0]
        })

    patient_df = pd.DataFrame(patient_records)
    patient_feature_predictions[experiment_name] = patient_df

    patient_feature_results.append({
        "experiment": experiment_name,
        "patients": len(patient_df),
        "accuracy": accuracy_score(
            patient_df["y_true"], patient_df["y_pred"]
        ),
        "balanced_accuracy": balanced_accuracy_score(
            patient_df["y_true"], patient_df["y_pred"]
        ),
        "macro_f1": f1_score(
            patient_df["y_true"],
            patient_df["y_pred"],
            average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            patient_df["y_true"],
            patient_df["y_pred"],
            average="weighted",
            zero_division=0
        )
    })

print("========== PATIENT-LEVEL FEATURE COMPARISON ==========")
display(
    pd.DataFrame(patient_feature_results)
    .set_index("experiment")
    .round(4)
)

print("\n========== PATIENT-LEVEL CLASSIFICATION REPORTS ==========")

for experiment_name, patient_df in patient_feature_predictions.items():
    print(f"\n--- {experiment_name} ---")
    print(
        __import__("sklearn.metrics", fromlist=["classification_report"])
        .classification_report(
            patient_df["y_true"],
            patient_df["y_pred"],
            labels=sorted(cv_dataset["diagnosis"].unique()),
            zero_division=0,
            digits=3
        )
    )


========== PATIENT-LEVEL FEATURE COMPARISON ==========


,patients,accuracy,balanced_accuracy,macro_f1,weighted_f1
experiment,,,,,
all_39_features,101,0.5644,0.2910,0.2881,0.5802
anova_top_20,101,0.5050,0.2918,0.2637,0.5246
anova_top_30,101,0.5347,0.3081,0.2809,0.5649



========== PATIENT-LEVEL CLASSIFICATION REPORTS ==========

--- all_39_features ---
                precision    recall  f1-score   support

        Asthma      0.000     0.000     0.000         1
Bronchiectasis      1.000     0.600     0.750         5
 Bronchiolitis      0.000     0.000     0.000         5
          COPD      0.872     0.804     0.837        51
       Healthy      0.550     0.524     0.537        21
          LRTI      0.000     0.000     0.000         2
     Pneumonia      0.118     0.400     0.182         5
          URTI      0.000     0.000     0.000        11

      accuracy                          0.564       101
     macro avg      0.317     0.291     0.288       101
  weighted avg      0.610     0.564     0.580       101


--- anova_top_20 ---
                precision    recall  f1-score   support

        Asthma      0.000     0.000     0.000         1
Bronchiectasis      0.667     0.800     0.727         5
 Bronchiolitis      0.000     0.000     0.000    

In [115]:

from sklearn.metrics import precision_recall_fscore_support
import pandas as pd

class_names = sorted(cv_dataset["diagnosis"].unique())
class_rows = []

for experiment_name, patient_df in patient_feature_predictions.items():
    precision, recall, f1, support = precision_recall_fscore_support(
        patient_df["y_true"],
        patient_df["y_pred"],
        labels=class_names,
        zero_division=0
    )

    for i, class_name in enumerate(class_names):
        class_rows.append({
            "configuration": experiment_name,
            "disease": class_name,
            "precision": precision[i],
            "recall": recall[i],
            "f1": f1[i],
            "patient_support": support[i]
        })

class_comparison_df = pd.DataFrame(class_rows)

print("========== PATIENT-LEVEL METRICS BY DISEASE ==========")
display(
    class_comparison_df.pivot(
        index="disease",
        columns="configuration",
        values="f1"
    ).round(3)
)

print("\n========== PATIENT SUPPORT ==========")
display(
    class_comparison_df[
        ["disease", "patient_support"]
    ].drop_duplicates().set_index("disease")
)

print("\n========== FULL METRICS ==========")
display(class_comparison_df.round(3))


========== PATIENT-LEVEL METRICS BY DISEASE ==========


configuration,all_39_features,anova_top_20,anova_top_30
disease,,,
Asthma,0.000,0.000,0.000
Bronchiectasis,0.750,0.727,0.667
Bronchiolitis,0.000,0.000,0.000
COPD,0.837,0.766,0.809
Healthy,0.537,0.450,0.500
LRTI,0.000,0.000,0.000
Pneumonia,0.182,0.167,0.167
URTI,0.000,0.000,0.105



========== PATIENT SUPPORT ==========


,patient_support
disease,
Asthma,1
Bronchiectasis,5
Bronchiolitis,5
COPD,51
Healthy,21
LRTI,2
Pneumonia,5
URTI,11



========== FULL METRICS ==========


,configuration,disease,precision,recall,f1,patient_support
0,all_39_features,Asthma,0.000,0.000,0.000,1
1,all_39_features,Bronchiectasis,1.000,0.600,0.750,5
2,all_39_features,Bronchiolitis,0.000,0.000,0.000,5
3,all_39_features,COPD,0.872,0.804,0.837,51
4,all_39_features,Healthy,0.550,0.524,0.537,21
5,all_39_features,LRTI,0.000,0.000,0.000,2
6,all_39_features,Pneumonia,0.118,0.400,0.182,5
7,all_39_features,URTI,0.000,0.000,0.000,11
8,anova_top_20,Asthma,0.000,0.000,0.000,1
9,anova_top_20,Bronchiectasis,0.667,0.800,0.727,5


In [116]:

import pandas as pd

# Collect the existing CV result tables without retraining anything.
result_tables = []

# Class-weighting experiments
if "weight_results_df" in globals():
    temp = weight_results_df.copy()
    temp["experiment_group"] = "class_weighting"
    temp["configuration"] = temp["strategy"]
    result_tables.append(temp)

# Feature-selection experiments
if "feature_results_df" in globals():
    temp = feature_results_df.copy()
    temp["experiment_group"] = "feature_selection"
    temp["configuration"] = temp["experiment"]
    result_tables.append(temp)

# Oversampling experiments
if "results_df" in globals():
    temp = results_df.copy()
    temp["experiment_group"] = "oversampling"
    temp["configuration"] = temp["experiment"]
    result_tables.append(temp)

if not result_tables:
    raise RuntimeError(
        "No previous CV result tables found. "
        "Check that the earlier experiment cells ran successfully."
    )

all_cv_results = pd.concat(result_tables, ignore_index=True, sort=False)

metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1"
]

summary = (
    all_cv_results
    .groupby(["experiment_group", "configuration"])[metric_columns]
    .agg(["mean", "std"])
    .round(4)
)

print("========== CONSOLIDATED 3-FOLD CV RESULTS ==========")
display(summary)

print("\n========== CONFIGURATIONS RANKED BY MEAN MACRO-F1 ==========")

ranking = (
    all_cv_results
    .groupby(["experiment_group", "configuration"])[metric_columns]
    .mean()
    .sort_values("macro_f1", ascending=False)
    .round(4)
)

display(ranking)

print("\nReserved test set was not used in this comparison.")


========== CONSOLIDATED 3-FOLD CV RESULTS ==========


accuracy         balanced_accuracy          \
                                      mean     std              mean     std   
experiment_group  configuration                                                
class_weighting   balanced          0.6103  0.1331            0.3360  0.0734   
                  custom            0.7060  0.1298            0.2655  0.0428   
                  sqrt              0.7058  0.1294            0.2676  0.0448   
feature_selection all_39_features   0.6398  0.1353            0.3371  0.1420   
                  anova_top_20      0.6103  0.1331            0.3360  0.0734   
                  anova_top_30      0.6344  0.1313            0.3489  0.0929   
oversampling      LR_balanced       0.6103  0.1331            0.3360  0.0734   
                  LR_oversampled    0.6078  0.1283            0.3233  0.0783   
                  RF_balanced       0.7433  0.0990            0.2541  0.0971   
                  RF_oversampled    0.7838  0.0806            0.1831  0.0330   

                                  macro_f1         weighted_f1          
                                      mean     std        mean     std  
experiment_group  configuration                                         
class_weighting   balanced          0.2095  0.0155      0.6807  0.1073  
                  custom            0.2076  0.0170      0.7342  0.0938  
                  sqrt              0.2089  0.0193      0.7341  0.0936  
feature_selection all_39_features   0.2144  0.0532      0.7040  0.1102  
                  anova_top_20      0.2095  0.0155      0.6807  0.1073  
                  anova_top_30      0.2139  0.0359      0.6993  0.1023  
oversampling      LR_balanced       0.2095  0.0155      0.6807  0.1073  
                  LR_oversampled    0.2054  0.0085      0.6790  0.1042  
                  RF_balanced       0.1932  0.0369      0.7514  0.0876  
                  RF_oversampled    0.1635  0.0126      0.7548  0.0809


========== CONFIGURATIONS RANKED BY MEAN MACRO-F1 ==========


accuracy  balanced_accuracy  macro_f1  \
experiment_group  configuration                                            
feature_selection all_39_features    0.6398             0.3371    0.2144   
                  anova_top_30       0.6344             0.3489    0.2139   
class_weighting   balanced           0.6103             0.3360    0.2095   
feature_selection anova_top_20       0.6103             0.3360    0.2095   
oversampling      LR_balanced        0.6103             0.3360    0.2095   
class_weighting   sqrt               0.7058             0.2676    0.2089   
                  custom             0.7060             0.2655    0.2076   
oversampling      LR_oversampled     0.6078             0.3233    0.2054   
                  RF_balanced        0.7433             0.2541    0.1932   
                  RF_oversampled     0.7838             0.1831    0.1635   

                                   weighted_f1  
experiment_group  configuration                 
feature_selection all_39_features       0.7040  
                  anova_top_30          0.6993  
class_weighting   balanced              0.6807  
feature_selection anova_top_20          0.6807  
oversampling      LR_balanced           0.6807  
class_weighting   sqrt                  0.7341  
                  custom                0.7342  
oversampling      LR_oversampled        0.6790  
                  RF_balanced           0.7514  
                  RF_oversampled        0.7548


Reserved test set was not used in this comparison.


In [117]:

from pathlib import Path

project_root = Path.cwd()

print("Current working directory:", project_root)

candidate_dirs = [
    project_root / "models",
    project_root / "results",
    project_root / "reports",
    project_root / "data" / "processed",
    project_root / "artifacts"
]

extensions = {".csv", ".json", ".txt", ".xlsx"}

print("\n========== EXISTING EVALUATION FILES ==========")

for directory in candidate_dirs:
    if directory.exists():
        print(f"\n[{directory}]")
        matches = [
            p for p in directory.rglob("*")
            if p.is_file()
            and p.suffix.lower() in extensions
            and any(
                term in p.name.lower()
                for term in [
                    "metric", "result", "evaluat",
                    "predict", "report", "summary",
                    "patient", "classification"
                ]
            )
        ]

        for path in matches[:60]:
            print(path.relative_to(project_root))

        if not matches:
            print("No matching evaluation files found.")
    else:
        print(f"\n[Not found] {directory}")


Current working directory: /Users/vs/Sem 5/Respiratory-Disease-Classification

========== EXISTING EVALUATION FILES ==========

[/Users/vs/Sem 5/Respiratory-Disease-Classification/models]
models/sound_phenotype_cnn/test_predictions.csv
models/sound_phenotype_cnn/final_test_summary.csv
models/deep_learning/cnn_lstm/cnn_lstm_test_predictions.csv
models/deep_learning/lightweight_mel_cnn/cnn_test_predictions.csv
models/deep_learning/regularized_multifeature_cnn/test_predictions.csv
models/deep_learning/multifeature_cnn/multifeature_cnn_test_predictions.csv
models/sound_phenotype_cnn/xgboost/validation_metrics.json
models/sound_phenotype_cnn/xgboost/validation_predictions.csv

[Not found] /Users/vs/Sem 5/Respiratory-Disease-Classification/results

[/Users/vs/Sem 5/Respiratory-Disease-Classification/reports]
reports/final_artifacts/cnn_lstm_original_metric_recheck.csv
reports/final_artifacts/patient_level_model_comparison.csv
reports/final_artifacts/prediction_metric_count_checks.csv
reports